# nb_hub_biblioteca

Gerado por `ferramentas/gerar_notebooks.py`. Não edite no Fabric: altere o repositório, gere novamente e reimporte.

Motor `hub` 0.1.0 (32 módulos) e registro declarativo (15 contratos). Assinatura `6b9c05c2f9eb78cb727aa1c25e96f8c9f5cb709e89b114d83a96d602d4d397ea`.

Use com `%run nb_hub_biblioteca`. Define `hub` e `REGISTRO`.

In [ ]:
import hashlib
import json
import os
import sys
import tempfile

HUB_ASSINATURA = (
    "6b9c05c2f9eb78cb727aa1c25e96f8c9f5cb709e89b114d83a96d602d4d397ea"
)
_HUB_MODULOS = {}

In [ ]:
_HUB_MODULOS["hub/__init__.py"] = r'''
"""Motor genérico de ingestão dirigido por contrato do Hub de Dados.

O pacote tem duas partes:

- módulos puros (sem Spark): contrato, plano, decisões, hashes, ciclo de
  vida, histórico, aquisição e inspeção. São cobertos pela suíte local;
- ``hub.spark``: executores finos que interpretam os planos no Spark e no
  Delta Lake. Só são exercitados por execução no Fabric.

Nenhum módulo deste pacote contém nome de fonte, base ou entidade.
"""

__version__ = "0.1.0"
'''

In [ ]:
_HUB_MODULOS["hub/ambiente.py"] = r'''
"""Configuração de ambiente: traduz identidade lógica em nome físico.

DEV, HML e PROD são workspaces distintos. O nome do objeto é igual nos
três; o ambiente resolve apenas o schema de cada camada e a raiz de
caminho. Nenhum identificador de tenant (GUID) é aceito aqui.
"""

from __future__ import annotations

import ntpath
import posixpath
import re
from collections.abc import Mapping
from dataclasses import dataclass
from typing import Any

from hub import vocabulario as v
from hub.leitor import Leitor

FORMATO_CONFIGURACAO = 1
_NOME_AMBIENTE = re.compile(r"^[a-z][a-z0-9]*$")
_NOME_SCHEMA = re.compile(r"^[a-z][a-z0-9_]*$")
_NOME_LAKEHOUSE = re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")
_CAMINHO_RELATIVO = re.compile(r"^[A-Za-z0-9_][A-Za-z0-9_\-/]*$")


@dataclass(frozen=True)
class Ambiente:
    """Resolução física de um ambiente (DEV, HML ou PROD)."""

    nome: str
    schemas: Mapping[str, str]
    raiz_raw: str
    montagem_local: str

    def tabela(self, camada: str, identificador: str) -> str:
        """Nome físico: <schema da camada>.<identificador lógico>."""
        return f"{self.schemas[camada]}.{identificador}"

    def tabela_controle(self, nome: str) -> str:
        """Nome físico de uma tabela operacional do motor."""
        return self.tabela(v.CONTROLE, nome)

    def caminho_raw(self, *partes: str) -> str:
        """Caminho relativo ao lakehouse padrão, usado pelo Spark."""
        return posixpath.join(self.raiz_raw, *partes)

    def local(self, caminho: str) -> str:
        """Converte caminho relativo em caminho local para E/S em Python."""
        return posixpath.join(self.montagem_local, caminho)


@dataclass(frozen=True)
class Configuracao:
    """Configuração global: lakehouse padrão e ambientes."""

    lakehouse: str
    ambientes: Mapping[str, Ambiente]

    def ambiente(self, nome: str) -> Ambiente:
        """Retorna o ambiente pelo nome.

        Raises:
            KeyError: se o ambiente não estiver configurado.
        """
        if nome not in self.ambientes:
            disponiveis = ", ".join(sorted(self.ambientes))
            raise KeyError(f"ambiente {nome!r} fora de: {disponiveis}")
        return self.ambientes[nome]


def _ambiente(
    leitor: Leitor, nome: str, valor: Any, caminho: str
) -> Ambiente | None:
    """Lê a resolução física de um ambiente."""
    dados = leitor.mapa(
        valor, caminho, ("schemas", "raiz_raw", "montagem_local")
    )
    schemas = leitor.mapa(
        dados.get("schemas"), f"{caminho}.schemas", v.CAMADAS_DE_SCHEMA
    )
    resolvidos = {}
    for camada in v.CAMADAS_DE_SCHEMA:
        schema = leitor.texto(
            schemas.get(camada), f"{caminho}.schemas.{camada}", _NOME_SCHEMA
        )
        if schema is not None:
            resolvidos[camada] = schema
    raiz = leitor.texto(
        dados.get("raiz_raw"), f"{caminho}.raiz_raw", _CAMINHO_RELATIVO
    )
    if raiz is not None and ".." in raiz.split("/"):
        leitor.erro(f"{caminho}.raiz_raw", "não use '..'")
    montagem = leitor.texto(
        dados.get("montagem_local"), f"{caminho}.montagem_local"
    )
    if montagem is not None and not (
        posixpath.isabs(montagem) or ntpath.isabs(montagem)
    ):
        leitor.erro(f"{caminho}.montagem_local", "use caminho absoluto")
        montagem = None
    completos = len(resolvidos) == len(v.CAMADAS_DE_SCHEMA)
    if not completos or raiz is None or montagem is None:
        return None
    return Ambiente(nome, resolvidos, raiz, montagem)


def carregar_configuracao(dados: Any) -> Configuracao:
    """Lê e valida a configuração de ambientes.

    Raises:
        ContratoInvalido: com todos os problemas encontrados.
    """
    leitor = Leitor()
    raiz = leitor.mapa(
        dados,
        "configuracao",
        ("formato_configuracao", "lakehouse", "ambientes"),
    )
    formato = leitor.inteiro(
        raiz.get("formato_configuracao"), "formato_configuracao"
    )
    if formato is not None and formato != FORMATO_CONFIGURACAO:
        leitor.erro("formato_configuracao", f"não suportado: {formato}")
    lakehouse = leitor.texto(
        raiz.get("lakehouse"), "lakehouse", _NOME_LAKEHOUSE
    )
    declarados = raiz.get("ambientes")
    ambientes = {}
    if not isinstance(declarados, Mapping) or not declarados:
        leitor.erro("ambientes", "declare ao menos um ambiente")
        declarados = {}
    for nome, valor in declarados.items():
        caminho = f"ambientes.{nome}"
        if not isinstance(nome, str) or not _NOME_AMBIENTE.match(nome):
            leitor.erro(caminho, "nome de ambiente inválido")
            continue
        ambiente = _ambiente(leitor, nome, valor, caminho)
        if ambiente is not None:
            ambientes[nome] = ambiente
    leitor.concluir("configuracao")
    return Configuracao(lakehouse, ambientes)
'''

In [ ]:
_HUB_MODULOS["hub/aquisicao.py"] = r'''
"""Aquisição: traz a publicação para a Raw e registra a evidência.

A Raw nunca é sobrescrita. Cada aquisição grava numa pasta própria:

    <raiz_raw>/<fonte>/<base>/<competencia>/<pasta da aquisição>/

ZIPs são extraídos em ``<pasta da aquisição>/<nome do zip sem extensão>/``
e cada arquivo tem SHA-256 calculado durante a escrita. Há dois modos:

- ``download``: baixa as URLs declaradas no descritor da base;
- ``deposito``: registra arquivos depositados por fora (upload, Data
  Pipeline, Power Automate) numa pasta informada pelo operador.
"""

from __future__ import annotations

import hashlib
import os
import posixpath
import re
import time
import urllib.error
import urllib.request
import zipfile
from collections.abc import Callable
from typing import BinaryIO

from hub.ambiente import Ambiente
from hub.bases import DescritorBase
from hub.hashes import TAMANHO_BLOCO, sha256_arquivo

DOWNLOAD = "download"
DEPOSITO = "deposito"
MODOS = (DOWNLOAD, DEPOSITO)

AGENTE = "hub-de-dados/0.1"
TEMPO_LIMITE = 300
_NOME_PASTA = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.\-]*$")


class ErroAquisicao(RuntimeError):
    """Falha ao obter ou registrar os arquivos da publicação."""


def gravar_fluxo(fluxo: BinaryIO, destino: str) -> tuple[int, str]:
    """Grava um fluxo em arquivo novo, calculando bytes e SHA-256.

    Raises:
        FileExistsError: se o destino já existir (a Raw não sobrescreve).
    """
    if os.path.exists(destino):
        raise FileExistsError(destino)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    resumo = hashlib.sha256()
    total = 0
    with open(destino, "wb") as arquivo:
        for bloco in iter(lambda: fluxo.read(TAMANHO_BLOCO), b""):
            arquivo.write(bloco)
            resumo.update(bloco)
            total += len(bloco)
    return total, resumo.hexdigest()


def _sha256_fluxo(fluxo: BinaryIO) -> str:
    """SHA-256 de um fluxo sem gravá-lo."""
    resumo = hashlib.sha256()
    for bloco in iter(lambda: fluxo.read(TAMANHO_BLOCO), b""):
        resumo.update(bloco)
    return resumo.hexdigest()


def baixar(
    url: str,
    destino: str,
    tentativas: int = 3,
    espera: float = 5.0,
    abrir: Callable = urllib.request.urlopen,
) -> tuple[int, str]:
    """Baixa a URL para um arquivo novo; repete em falha transitória.

    O conteúdo é gravado em ``<destino>.parcial<n>`` e renomeado ao fim,
    de modo que um download interrompido nunca aparece com o nome final.

    Raises:
        ErroAquisicao: após esgotar as tentativas ou em erro HTTP 4xx.
    """
    if os.path.exists(destino):
        raise FileExistsError(destino)
    for tentativa in range(1, tentativas + 1):
        parcial = f"{destino}.parcial{tentativa}"
        requisicao = urllib.request.Request(
            url, headers={"User-Agent": AGENTE}
        )
        try:
            with abrir(requisicao, timeout=TEMPO_LIMITE) as resposta:
                total, resumo = gravar_fluxo(resposta, parcial)
            os.replace(parcial, destino)
            return total, resumo
        except urllib.error.HTTPError as erro:
            if 400 <= erro.code < 500 or tentativa == tentativas:
                raise ErroAquisicao(f"{url}: HTTP {erro.code}") from erro
        except (urllib.error.URLError, TimeoutError, ConnectionError) as erro:
            if tentativa == tentativas:
                raise ErroAquisicao(f"{url}: {erro}") from erro
        time.sleep(espera * tentativa)
    raise AssertionError("inalcançável")


def extrair_zip(caminho_zip: str, pasta_destino: str) -> list[dict]:
    """Extrai os membros do ZIP em pasta plana, sem sobrescrever.

    O caminho interno do membro é descartado (evita path traversal).
    Membro já extraído com o mesmo SHA-256 é reaproveitado; com conteúdo
    diferente, a extração falha.

    Returns:
        Lista com nome, membro original, bytes e SHA-256 de cada arquivo.
    """
    extraidos = []
    nomes: set[str] = set()
    with zipfile.ZipFile(caminho_zip) as pacote:
        for membro in pacote.infolist():
            nome = posixpath.basename(membro.filename)
            if membro.is_dir() or not nome:
                continue
            if nome in nomes:
                raise ErroAquisicao(f"{caminho_zip}: membro repetido {nome}")
            nomes.add(nome)
            destino = os.path.join(pasta_destino, nome)
            with pacote.open(membro) as fluxo:
                if os.path.exists(destino):
                    resumo = _sha256_fluxo(fluxo)
                    if resumo != sha256_arquivo(destino):
                        raise ErroAquisicao(f"conteúdo divergente: {destino}")
                    total = os.path.getsize(destino)
                else:
                    total, resumo = gravar_fluxo(fluxo, destino)
            extraidos.append(
                {
                    "nome": nome,
                    "membro": membro.filename,
                    "bytes": total,
                    "sha256": resumo,
                }
            )
    return extraidos


def _registrar_arquivo(
    descritor: DescritorBase,
    ambiente: Ambiente,
    relativo: str,
    nome: str,
    competencia: str,
    modo: str,
    baixar_arquivo: Callable,
) -> dict:
    """Baixa ou confere um arquivo declarado e devolve sua evidência."""
    caminho = posixpath.join(relativo, nome)
    local = ambiente.local(caminho)
    url = None
    if modo == DOWNLOAD:
        url = descritor.url(nome, competencia)
        total, resumo = baixar_arquivo(url, local)
    elif not os.path.exists(local):
        raise ErroAquisicao(f"arquivo declarado não depositado: {caminho}")
    else:
        total, resumo = os.path.getsize(local), sha256_arquivo(local)
    return {
        "nome": nome,
        "caminho": caminho,
        "bytes": total,
        "sha256": resumo,
        "url": url,
    }


def _partes_do_arquivo(
    arquivo: dict, relativo: str, ambiente: Ambiente
) -> list[dict]:
    """Arquivos legíveis gerados por um arquivo adquirido."""
    nome = arquivo["nome"]
    if not nome.lower().endswith(".zip"):
        return [
            {
                "parte": nome,
                "caminho": arquivo["caminho"],
                "bytes": arquivo["bytes"],
                "sha256": arquivo["sha256"],
                "origem": None,
            }
        ]
    pasta = posixpath.splitext(nome)[0]
    destino = posixpath.join(relativo, pasta)
    partes = []
    for membro in extrair_zip(
        ambiente.local(arquivo["caminho"]), ambiente.local(destino)
    ):
        partes.append(
            {
                "parte": f"{pasta}/{membro['nome']}",
                "caminho": posixpath.join(destino, membro["nome"]),
                "bytes": membro["bytes"],
                "sha256": membro["sha256"],
                "origem": nome,
            }
        )
    return partes


def adquirir(
    descritor: DescritorBase,
    ambiente: Ambiente,
    competencia: str,
    id_aquisicao: str,
    modo: str = DOWNLOAD,
    pasta_deposito: str | None = None,
    baixar_arquivo: Callable = baixar,
) -> dict:
    """Traz a publicação para a Raw e devolve a evidência da aquisição.

    Args:
        descritor: descritor da base.
        ambiente: resolução física do ambiente.
        competencia: competência no formato AAAA-MM.
        id_aquisicao: identificador da execução (nome da pasta no download).
        modo: ``download`` ou ``deposito``.
        pasta_deposito: pasta já existente sob a competência (depósito).
        baixar_arquivo: função de download (injetável em testes).

    Returns:
        Detalhes com a pasta, os arquivos adquiridos e as partes legíveis.
    """
    if modo not in MODOS:
        raise ErroAquisicao(f"modo inválido: {modo}")
    pasta = id_aquisicao if modo == DOWNLOAD else pasta_deposito
    if not pasta or not _NOME_PASTA.match(pasta):
        raise ErroAquisicao(f"pasta de aquisição inválida: {pasta!r}")
    relativo = ambiente.caminho_raw(
        descritor.fonte, descritor.base, competencia, pasta
    )
    if modo == DOWNLOAD and os.path.exists(ambiente.local(relativo)):
        raise ErroAquisicao(f"pasta de aquisição já existe: {relativo}")
    arquivos = [
        _registrar_arquivo(
            descritor,
            ambiente,
            relativo,
            nome,
            competencia,
            modo,
            baixar_arquivo,
        )
        for nome in descritor.arquivos
    ]
    partes = [
        parte
        for arquivo in arquivos
        for parte in _partes_do_arquivo(arquivo, relativo, ambiente)
    ]
    return {
        "modo": modo,
        "pasta": relativo,
        "arquivos": arquivos,
        "partes": partes,
    }
'''

In [ ]:
_HUB_MODULOS["hub/arquivos.py"] = r'''
"""Leitura de arquivos no driver, em Python puro.

Usada para conferir cabeçalho, ler o total declarado pela fonte e
inspecionar amostras. Lê apenas o início de cada arquivo.
"""

from __future__ import annotations

import csv
import io

from hub import vocabulario as v
from hub.contrato import Contrato, Leitura
from hub.decisoes import verificar_cabecalho

LIMITE_AMOSTRA = 1 << 20


def argumentos_csv(leitura: Leitura) -> dict:
    """Dialeto do contrato no formato do módulo csv."""
    dobra_aspas = leitura.escape == leitura.aspas
    return {
        "delimiter": leitura.separador,
        "quotechar": leitura.aspas,
        "doublequote": dobra_aspas,
        "escapechar": None if dobra_aspas else leitura.escape,
        "strict": False,
    }


def ler_inicio(caminho: str, limite: int = LIMITE_AMOSTRA) -> bytes:
    """Lê até ``limite`` bytes, cortando na última quebra de linha."""
    with open(caminho, "rb") as arquivo:
        dados = arquivo.read(limite + 1)
    if len(dados) > limite:
        corte = dados.rfind(b"\n", 0, limite)
        dados = dados[: corte + 1] if corte >= 0 else dados[:limite]
    return dados


def decodificar(dados: bytes, codificacao: str) -> tuple[str, bool]:
    """Decodifica com o codec do contrato e indica se houve erro."""
    codec = v.CODIFICACOES[codificacao].python
    try:
        return dados.decode(codec), True
    except UnicodeDecodeError:
        return dados.decode(codec, errors="replace"), False


def registros_csv(texto: str, leitura: Leitura) -> list[list[str]]:
    """Registros do texto conforme o dialeto, sem linhas vazias."""
    leitor = csv.reader(
        io.StringIO(texto, newline=""), **argumentos_csv(leitura)
    )
    return [registro for registro in leitor if registro]


def ler_cabecalho(caminho: str, leitura: Leitura) -> list[str]:
    """Nomes da primeira linha do arquivo."""
    texto, _ = decodificar(ler_inicio(caminho, 1 << 16), leitura.codificacao)
    registros = registros_csv(texto, leitura)
    return registros[0] if registros else []


def ler_total_declarado(
    caminho: str, contrato_total: Contrato, coluna: str
) -> tuple[int | None, list[str]]:
    """Lê o total de registros publicado pela fonte.

    Args:
        caminho: caminho local do arquivo de totais.
        contrato_total: contrato da entidade de totais.
        coluna: coluna do contrato de totais com o valor desejado.

    Returns:
        Par (total, problemas); total é None quando há problemas.
    """
    leitura = contrato_total.leitura
    texto, decodificou = decodificar(ler_inicio(caminho), leitura.codificacao)
    if not decodificou:
        return None, [f"arquivo de totais não decodifica: {caminho}"]
    registros = registros_csv(texto, leitura)
    origem = contrato_total.colunas_origem
    if leitura.cabecalho:
        esperado = [(c.cabecalho, *c.aliases) for c in origem]
        primeiro = registros[0] if registros else []
        problemas = verificar_cabecalho(primeiro, esperado)
        if problemas:
            return None, [f"arquivo de totais: {p}" for p in problemas]
        registros = registros[1:]
    if len(registros) != 1:
        return None, [f"arquivo de totais com {len(registros)} registros"]
    indice = [c.nome for c in origem].index(coluna)
    registro = registros[0]
    valor = registro[indice].strip() if indice < len(registro) else ""
    if not valor.isdigit():
        return None, [f"total declarado não numérico: {valor!r}"]
    return int(valor), []
'''

In [ ]:
_HUB_MODULOS["hub/bases.py"] = r'''
"""Descritor de base: como a publicação de uma fonte é adquirida.

Uma base (ex.: fonte_base) agrupa as entidades que chegam na mesma
publicação. O descritor declara a URL e a lista de arquivos; as entidades
declaram, em seus contratos, qual arquivo da publicação cada uma lê.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Any

from hub.contrato import PADRAO_FONTE
from hub.leitor import Leitor

FORMATO_BASE = 1
MARCADOR_COMPETENCIA = "{competencia}"
_ESQUEMAS_URL = ("https://", "http://", "file://")
_NOME_ARQUIVO = re.compile(r"^[^/\\]+$")


@dataclass(frozen=True)
class DescritorBase:
    """Aquisição declarada de uma base."""

    fonte: str
    base: str
    descricao: str | None
    url_base: str
    arquivos: tuple[str, ...]
    fonte_documental: str | None
    pendencias: tuple[str, ...]

    @property
    def identificador(self) -> str:
        """Nome lógico da base: fonte_base."""
        return f"{self.fonte}_{self.base}"

    def url(self, arquivo: str, competencia: str) -> str:
        """Monta a URL de um arquivo para a competência informada."""
        raiz = self.url_base.replace(MARCADOR_COMPETENCIA, competencia)
        if not raiz.endswith("/"):
            raiz += "/"
        return raiz + arquivo


def carregar_base(dados: Any, origem: str = "base") -> DescritorBase:
    """Lê e valida o descritor de uma base.

    Raises:
        ContratoInvalido: com todos os problemas encontrados.
    """
    leitor = Leitor()
    raiz = leitor.mapa(
        dados,
        "base",
        ("formato_base", "identidade", "aquisicao"),
        ("descricao", "documentacao"),
    )
    formato = leitor.inteiro(raiz.get("formato_base"), "formato_base")
    if formato is not None and formato != FORMATO_BASE:
        leitor.erro("formato_base", f"formato não suportado: {formato}")
    identidade = leitor.mapa(
        raiz.get("identidade"), "identidade", ("fonte", "base")
    )
    fonte = leitor.texto(
        identidade.get("fonte"), "identidade.fonte", PADRAO_FONTE
    )
    base = leitor.texto(
        identidade.get("base"), "identidade.base", PADRAO_FONTE
    )
    aquisicao = leitor.mapa(
        raiz.get("aquisicao"), "aquisicao", ("url_base", "arquivos")
    )
    url_base = leitor.texto(aquisicao.get("url_base"), "aquisicao.url_base")
    if url_base is not None and not url_base.startswith(_ESQUEMAS_URL):
        leitor.erro("aquisicao.url_base", "use https://, http:// ou file://")
    arquivos = [
        leitor.texto(nome, f"aquisicao.arquivos[{indice}]", _NOME_ARQUIVO)
        for indice, nome in enumerate(
            leitor.lista(
                aquisicao.get("arquivos"), "aquisicao.arquivos", opcional=False
            )
        )
    ]
    if aquisicao.get("arquivos") == []:
        leitor.erro("aquisicao.arquivos", "lista vazia")
    leitor.unicos([a for a in arquivos if a], "aquisicao.arquivos")
    documentacao = leitor.mapa(
        raiz.get("documentacao", {}),
        "documentacao",
        opcionais=("fonte_documental", "pendencias"),
    )
    pendencias = [
        leitor.texto(item, f"documentacao.pendencias[{indice}]")
        for indice, item in enumerate(
            leitor.lista(documentacao.get("pendencias"), "pendencias")
        )
    ]
    descritor = DescritorBase(
        fonte=fonte,
        base=base,
        descricao=leitor.texto(
            raiz.get("descricao"), "descricao", opcional=True
        ),
        url_base=url_base,
        arquivos=tuple(a for a in arquivos if a is not None),
        fonte_documental=leitor.texto(
            documentacao.get("fonte_documental"),
            "documentacao.fonte_documental",
            opcional=True,
        ),
        pendencias=tuple(p for p in pendencias if p is not None),
    )
    nome = f"{fonte}_{base}" if fonte and base else origem
    leitor.concluir(nome)
    return descritor
'''

In [ ]:
_HUB_MODULOS["hub/catalogo.py"] = r'''
"""Catálogo: o que existe fisicamente e como usar.

Separa o conteúdo estrutural (derivado do mesmo plano que o motor executa
e atualizado a cada sincronização) do curatorial (eixo de informação,
data steward, descrições organizacionais), que é preservado entre
sincronizações e nasce como ``A_CONFIRMAR``. Nada é inferido pelo nome
da fonte, e o catálogo não guarda métrica de execução.
"""

from __future__ import annotations

from collections.abc import Iterable, Mapping

from hub import vocabulario as v
from hub.ambiente import Ambiente
from hub.contrato import Contrato
from hub.hashes import hash_contrato
from hub.plano import plano_bronze, plano_silver

TABELA_ATIVOS = "catalogo_ativos"
TABELA_COLUNAS = "catalogo_colunas"
TABELA_DEPENDENCIAS = "catalogo_dependencias"

ESQUEMAS = {
    TABELA_ATIVOS: (
        ("ativo", "STRING"),
        ("camada", "STRING"),
        ("identificador", "STRING"),
        ("fonte", "STRING"),
        ("base", "STRING"),
        ("entidade", "STRING"),
        ("descricao", "STRING"),
        ("chave", "STRING"),
        ("versao_contrato", "INT"),
        ("estado_contrato", "STRING"),
        ("hash_contrato", "STRING"),
        ("fonte_documental", "STRING"),
        ("existe", "BOOLEAN"),
        ("eixo_informacao", "STRING"),
        ("data_steward", "STRING"),
        ("descricao_organizacional", "STRING"),
    ),
    TABELA_COLUNAS: (
        ("ativo", "STRING"),
        ("coluna", "STRING"),
        ("ordem", "INT"),
        ("tipo", "STRING"),
        ("nulavel", "BOOLEAN"),
        ("chave", "BOOLEAN"),
        ("dominio", "STRING"),
        ("descricao", "STRING"),
        ("descricao_organizacional", "STRING"),
    ),
    TABELA_DEPENDENCIAS: (
        ("ativo", "STRING"),
        ("depende_de", "STRING"),
        ("tipo", "STRING"),
        ("coluna", "STRING"),
        ("coluna_referenciada", "STRING"),
    ),
}

CHAVES = {
    TABELA_ATIVOS: ("ativo",),
    TABELA_COLUNAS: ("ativo", "coluna"),
    TABELA_DEPENDENCIAS: ("ativo", "depende_de", "tipo", "coluna"),
}

CURATORIAIS = {
    TABELA_ATIVOS: (
        "eixo_informacao",
        "data_steward",
        "descricao_organizacional",
    ),
    TABELA_COLUNAS: ("descricao_organizacional",),
    TABELA_DEPENDENCIAS: (),
}

ORIGEM = "ORIGEM"
CAMADA = "CAMADA"
REFERENCIA = "REFERENCIA"


def colunas_estruturais(tabela: str) -> tuple[str, ...]:
    """Colunas que a sincronização atualiza (todas menos a curadoria)."""
    return tuple(
        nome for nome, _ in ESQUEMAS[tabela] if nome not in CURATORIAIS[tabela]
    )


def _ativo(
    contrato: Contrato, camada: str, tabela: str, existentes: set[str]
) -> dict:
    """Linha de um ativo; a curadoria nasce A_CONFIRMAR."""
    linha = {
        "ativo": tabela,
        "camada": camada,
        "identificador": contrato.identificador,
        "fonte": contrato.identidade.fonte,
        "base": contrato.identidade.base,
        "entidade": contrato.identidade.entidade,
        "descricao": contrato.descricao,
        "chave": ",".join(contrato.chave) or None,
        "versao_contrato": contrato.versao,
        "estado_contrato": contrato.documentacao.estado,
        "hash_contrato": hash_contrato(contrato),
        "fonte_documental": contrato.documentacao.fonte_documental,
        "existe": tabela in existentes,
    }
    linha.update({c: v.A_CONFIRMAR for c in CURATORIAIS[TABELA_ATIVOS]})
    return linha


def _colunas(
    contrato: Contrato,
    camada: str,
    tabela: str,
    schema: Iterable[tuple[str, str]],
    comentarios: Mapping[str, str],
) -> list[dict]:
    """Linhas de coluna a partir do schema físico planejado."""
    declaradas = {coluna.nome: coluna for coluna in contrato.colunas}
    linhas = []
    for ordem, (nome, tipo) in enumerate(schema, start=1):
        coluna = declaradas.get(nome)
        if nome in v.COLUNAS_LINHAGEM:
            nulavel = False
        elif camada == v.SILVER and coluna is not None:
            nulavel = coluna.nulavel and nome not in contrato.chave
        else:
            nulavel = True
        linhas.append(
            {
                "ativo": tabela,
                "coluna": nome,
                "ordem": ordem,
                "tipo": tipo,
                "nulavel": nulavel,
                "chave": nome in contrato.chave,
                "dominio": (
                    coluna.dominio
                    if camada == v.SILVER and coluna is not None
                    else None
                ),
                "descricao": comentarios.get(nome) or None,
                "descricao_organizacional": v.A_CONFIRMAR,
            }
        )
    return linhas


def _dependencias(
    contrato: Contrato, ambiente: Ambiente, bronze: str, silver: str
) -> list[dict]:
    """Dependências: origem na Raw, camada anterior e referências."""
    identidade = contrato.identidade
    linhas = [
        {
            "ativo": bronze,
            "depende_de": ambiente.caminho_raw(
                identidade.fonte, identidade.base
            ),
            "tipo": ORIGEM,
            "coluna": None,
            "coluna_referenciada": None,
        },
        {
            "ativo": silver,
            "depende_de": bronze,
            "tipo": CAMADA,
            "coluna": None,
            "coluna_referenciada": None,
        },
    ]
    for coluna in contrato.colunas:
        dominio = contrato.dominios.get(coluna.dominio or "")
        if dominio is None or dominio.tabela is None:
            continue
        linhas.append(
            {
                "ativo": silver,
                "depende_de": ambiente.tabela(
                    v.SILVER, dominio.tabela.identificador
                ),
                "tipo": REFERENCIA,
                "coluna": coluna.nome,
                "coluna_referenciada": dominio.coluna,
            }
        )
    return linhas


def linhas_catalogo(
    contratos: Iterable[Contrato], ambiente: Ambiente, existentes: set[str]
) -> dict[str, list[dict]]:
    """Linhas estruturais das três tabelas de catálogo.

    Args:
        contratos: contratos do registro.
        ambiente: resolução física do ambiente.
        existentes: nomes físicos das tabelas que existem no ambiente.
    """
    linhas: dict[str, list[dict]] = {nome: [] for nome in ESQUEMAS}
    for contrato in contratos:
        bronze = plano_bronze(contrato, ambiente)
        silver = plano_silver(contrato, ambiente)
        for camada, plano in ((v.BRONZE, bronze), (v.SILVER, silver)):
            linhas[TABELA_ATIVOS].append(
                _ativo(contrato, camada, plano.tabela, existentes)
            )
            linhas[TABELA_COLUNAS].extend(
                _colunas(
                    contrato,
                    camada,
                    plano.tabela,
                    plano.schema_fisico,
                    plano.comentarios_colunas,
                )
            )
        linhas[TABELA_DEPENDENCIAS].extend(
            _dependencias(contrato, ambiente, bronze.tabela, silver.tabela)
        )
    return linhas
'''

In [ ]:
_HUB_MODULOS["hub/ciclo_vida.py"] = r'''
"""Ciclo de vida do contrato: estado declarado × evidência registrada.

O estado é declarado no contrato (parte documental, fora do hash). A
evidência vive no registro operacional de cada ambiente:

- VALIDADO_FISICO exige inspeção COMPATIVEL com o mesmo hash_leitura;
- APROVADO exige também aprovação nominal com o mesmo hash_contrato.

Mudar a parte executável troca o hash e invalida o lastro anterior.
"""

from __future__ import annotations

from collections.abc import Iterable

from hub import vocabulario as v
from hub.contrato import Pendencia
from hub.historico import Evidencias


def verificar_lastro(
    estado: str,
    hash_leitura: str,
    hash_contrato: str,
    evidencias: Evidencias,
) -> list[str]:
    """Confere se o estado declarado tem lastro no ambiente.

    Returns:
        Lista de problemas; vazia quando o contrato pode publicar.
    """
    if estado == v.OBSOLETO:
        return ["contrato OBSOLETO não publica"]
    problemas = []
    ordem = v.ORDEM_ESTADOS[estado]
    validado = hash_leitura in evidencias.inspecoes_compativeis
    if ordem >= v.ORDEM_ESTADOS[v.VALIDADO_FISICO] and not validado:
        problemas.append(
            f"estado {estado} sem inspeção COMPATIVEL neste ambiente para "
            f"hash_leitura {hash_leitura[:12]}"
        )
    if estado == v.APROVADO and hash_contrato not in evidencias.aprovacoes:
        problemas.append(
            "estado APROVADO sem aprovação neste ambiente para "
            f"hash_contrato {hash_contrato[:12]}"
        )
    return problemas


def verificar_aprovavel(
    estado: str,
    pendencias: Iterable[Pendencia],
    hash_leitura: str,
    evidencias: Evidencias,
) -> list[str]:
    """Confere se o contrato pode receber aprovação neste ambiente.

    Aprovação não decorre de lista de pendências vazia: ela exige
    inspeção compatível e é registrada de forma nominal e datada.
    """
    problemas = []
    if estado == v.OBSOLETO:
        problemas.append("contrato OBSOLETO não pode ser aprovado")
    if hash_leitura not in evidencias.inspecoes_compativeis:
        problemas.append(
            "aprovação exige inspeção COMPATIVEL neste ambiente para "
            f"hash_leitura {hash_leitura[:12]}"
        )
    for pendencia in pendencias:
        problemas.append(f"pendência aberta: {pendencia.descricao}")
    return problemas
'''

In [ ]:
_HUB_MODULOS["hub/contrato.py"] = r'''
"""Modelo, leitura e validação do contrato de entidade.

O contrato declara o que deve existir: identidade lógica, seleção de
arquivos, dialeto de leitura, colunas, tipos, nulabilidade, chave,
transformações, domínios, regras e a ação de cada violação.

Não declara nome físico, caminho de ambiente, estado de execução,
contagem, timestamp nem detalhe de implementação Spark.
"""

from __future__ import annotations

import re
from collections.abc import Mapping
from dataclasses import dataclass
from typing import Any

from hub import vocabulario as v
from hub.expressao import validar_expressao
from hub.leitor import Leitor

FORMATO_CONTRATO = 1
LIMITE_DOMINIO_LITERAL = 15

PADRAO_FONTE = re.compile(r"^[a-z][a-z0-9]*$")
PADRAO_NOME = re.compile(r"^[a-z][a-z0-9_]*$")
_PADRAO_DATA = re.compile(r"^(AAAA|MM|DD|[-/.])+$")

LITERAL = "literal"
REFERENCIA = "referencia"


@dataclass(frozen=True)
class Identidade:
    """Identidade lógica: fonte, base, entidade e especificação opcional."""

    fonte: str
    base: str
    entidade: str
    especificacao: str | None = None

    @property
    def identificador(self) -> str:
        """Nome lógico do objeto: fonte_base_entidade[_especificacao]."""
        partes = [self.fonte, self.base, self.entidade]
        if self.especificacao:
            partes.append(self.especificacao)
        return "_".join(partes)

    @property
    def identificador_base(self) -> str:
        """Nome lógico da base: fonte_base."""
        return f"{self.fonte}_{self.base}"


@dataclass(frozen=True)
class Selecao:
    """Padrão do arquivo lido pela entidade dentro da publicação."""

    arquivo: str
    quantidade_partes: int | None


@dataclass(frozen=True)
class Leitura:
    """Dialeto físico do arquivo."""

    separador: str
    aspas: str
    escape: str
    codificacao: str
    cabecalho: bool
    tolerancia_malformadas: float


@dataclass(frozen=True)
class Transformacao:
    """Transformação do vocabulário fechado com seus parâmetros."""

    tipo: str
    parametros: Mapping[str, Any]


@dataclass(frozen=True)
class Coluna:
    """Coluna da entidade: layout de origem e tratamento na Silver."""

    nome: str
    tipo: str
    cabecalho: str | None
    aliases: tuple[str, ...]
    ausente_na_origem: bool
    nulavel: bool
    formato: str | None
    precisao: int | None
    escala: int | None
    separador_decimal: str | None
    transformacoes: tuple[Transformacao, ...]
    dominio: str | None
    descricao: str | None


@dataclass(frozen=True)
class Dominio:
    """Domínio literal (valores no contrato) ou por referência a tabela."""

    nome: str
    tipo: str
    valores: Mapping[str, str]
    coluna_descricao: str | None
    tabela: Identidade | None
    coluna: str | None
    descricao: str | None


@dataclass(frozen=True)
class Regra:
    """Regra de negócio expressa como predicado SQL validado."""

    id: str
    expressao: str
    colunas_referenciadas: tuple[str, ...]
    acao: str
    dimensao: str
    descricao: str | None


@dataclass(frozen=True)
class TotalDeclarado:
    """Total de registros publicado pela própria fonte em outra entidade."""

    entidade: str
    coluna: str


@dataclass(frozen=True)
class Pendencia:
    """Pendência explícita e o estado que ela impede."""

    descricao: str
    impede: str


@dataclass(frozen=True)
class Documentacao:
    """Parte documental do contrato; fica fora do hash."""

    estado: str
    fonte_documental: str | None
    pendencias: tuple[Pendencia, ...]
    observacoes: str | None


@dataclass(frozen=True)
class Contrato:
    """Contrato validado de uma entidade."""

    formato_contrato: int
    identidade: Identidade
    versao: int
    descricao: str | None
    selecao: Selecao
    leitura: Leitura
    transformacoes_padrao: tuple[Transformacao, ...]
    colunas: tuple[Coluna, ...]
    chave: tuple[str, ...]
    dominios: Mapping[str, Dominio]
    regras: tuple[Regra, ...]
    acoes_padrao: Mapping[str, str]
    total_declarado: TotalDeclarado | None
    documentacao: Documentacao

    @property
    def identificador(self) -> str:
        """Nome lógico da entidade."""
        return self.identidade.identificador

    @property
    def colunas_origem(self) -> tuple[Coluna, ...]:
        """Colunas presentes no arquivo, na ordem física."""
        return tuple(c for c in self.colunas if not c.ausente_na_origem)

    def coluna(self, nome: str) -> Coluna:
        """Retorna a coluna pelo nome.

        Raises:
            KeyError: se a coluna não existir.
        """
        for coluna in self.colunas:
            if coluna.nome == nome:
                return coluna
        raise KeyError(nome)

    def dominio_literal(self, coluna: Coluna) -> Dominio | None:
        """Domínio literal da coluna, se houver."""
        dominio = self.dominios.get(coluna.dominio or "")
        if dominio is not None and dominio.tipo == LITERAL:
            return dominio
        return None

    @property
    def colunas_saida(self) -> tuple[str, ...]:
        """Colunas de negócio da Silver, com as descrições de domínio."""
        nomes: list[str] = []
        for coluna in self.colunas:
            nomes.append(coluna.nome)
            dominio = self.dominio_literal(coluna)
            if dominio is not None and dominio.coluna_descricao:
                nomes.append(dominio.coluna_descricao)
        return tuple(nomes)


def ler_identidade(leitor: Leitor, valor: Any, caminho: str):
    """Lê uma identidade lógica (fonte, base, entidade, especificação)."""
    dados = leitor.mapa(
        valor, caminho, ("fonte", "base", "entidade"), ("especificacao",)
    )
    fonte = leitor.texto(dados.get("fonte"), f"{caminho}.fonte", PADRAO_FONTE)
    base = leitor.texto(dados.get("base"), f"{caminho}.base", PADRAO_FONTE)
    entidade = leitor.texto(
        dados.get("entidade"), f"{caminho}.entidade", PADRAO_NOME
    )
    especificacao = leitor.texto(
        dados.get("especificacao"),
        f"{caminho}.especificacao",
        PADRAO_NOME,
        opcional=True,
    )
    if fonte is None or base is None or entidade is None:
        return None
    return Identidade(fonte, base, entidade, especificacao)


def _caractere(leitor: Leitor, valor: Any, caminho: str) -> str | None:
    """Lê um único caractere (separador, aspas, escape)."""
    if not isinstance(valor, str) or len(valor) != 1:
        leitor.erro(caminho, "deve ser um único caractere")
        return None
    return valor


def _selecao(leitor: Leitor, valor: Any) -> Selecao | None:
    """Lê a seção de seleção de arquivos."""
    dados = leitor.mapa(valor, "selecao", ("arquivo",), ("quantidade_partes",))
    arquivo = leitor.texto(dados.get("arquivo"), "selecao.arquivo")
    if arquivo is not None and "/" in arquivo:
        leitor.erro("selecao.arquivo", "use apenas o nome do arquivo")
    partes = leitor.inteiro(
        dados.get("quantidade_partes"),
        "selecao.quantidade_partes",
        minimo=1,
        opcional=True,
    )
    if arquivo is None:
        return None
    return Selecao(arquivo, partes)


def _leitura(leitor: Leitor, valor: Any) -> Leitura | None:
    """Lê o dialeto físico do arquivo."""
    obrigatorias = (
        "separador",
        "aspas",
        "escape",
        "codificacao",
        "cabecalho",
        "tolerancia_malformadas",
    )
    dados = leitor.mapa(valor, "leitura", obrigatorias)
    separador = _caractere(leitor, dados.get("separador"), "leitura.separador")
    aspas = _caractere(leitor, dados.get("aspas"), "leitura.aspas")
    escape = _caractere(leitor, dados.get("escape"), "leitura.escape")
    codificacao = leitor.escolha(
        dados.get("codificacao"), "leitura.codificacao", v.CODIFICACOES
    )
    cabecalho = leitor.booleano_obrigatorio(
        dados.get("cabecalho"), "leitura.cabecalho"
    )
    tolerancia = leitor.fracao(
        dados.get("tolerancia_malformadas"), "leitura.tolerancia_malformadas"
    )
    valores = (separador, aspas, escape, codificacao, cabecalho, tolerancia)
    if any(valor is None for valor in valores):
        return None
    if separador == aspas:
        leitor.erro("leitura", "separador e aspas devem ser diferentes")
    return Leitura(*valores)


def _parametro_valido(valor: Any, especie: str) -> bool:
    """Confere o valor de um parâmetro de transformação."""
    if especie == v.LISTA_TEXTO:
        return (
            isinstance(valor, list)
            and bool(valor)
            and all(isinstance(item, str) for item in valor)
        )
    if especie == v.INTEIRO_POSITIVO:
        return (
            isinstance(valor, int)
            and not isinstance(valor, bool)
            and valor > 0
        )
    if especie == v.CARACTERE:
        return isinstance(valor, str) and len(valor) == 1
    return False


def _transformacao(
    leitor: Leitor, valor: Any, caminho: str
) -> Transformacao | None:
    """Lê uma transformação: nome simples ou {nome: parâmetros}."""
    if isinstance(valor, str):
        nome, parametros = valor, {}
    elif isinstance(valor, Mapping) and len(valor) == 1:
        nome, parametros = next(iter(valor.items()))
        parametros = {} if parametros is None else parametros
    else:
        leitor.erro(caminho, "use o nome ou {nome: parâmetros}")
        return None
    especificacao = v.TRANSFORMACOES.get(nome)
    if especificacao is None:
        leitor.erro(caminho, f"transformação desconhecida: {nome}")
        return None
    dados = leitor.mapa(parametros, f"{caminho}.{nome}", tuple(especificacao))
    validos = True
    for chave, especie in especificacao.items():
        if chave in dados and not _parametro_valido(dados[chave], especie):
            leitor.erro(f"{caminho}.{nome}.{chave}", f"esperado {especie}")
            validos = False
    if not validos or set(dados) != set(especificacao):
        return None
    return Transformacao(nome, dict(dados))


def _transformacoes(
    leitor: Leitor, valor: Any, caminho: str
) -> tuple[Transformacao, ...]:
    """Lê uma lista de transformações, preservando a ordem."""
    resultado = []
    for indice, item in enumerate(leitor.lista(valor, caminho)):
        transformacao = _transformacao(leitor, item, f"{caminho}[{indice}]")
        if transformacao is not None:
            resultado.append(transformacao)
    return tuple(resultado)


def _formato_data(leitor: Leitor, valor: Any, caminho: str) -> str | None:
    """Lê o formato de data na notação do layout (ex.: AAAA-MM-DD)."""
    texto = leitor.texto(valor, caminho)
    if texto is None:
        return None
    tokens_unicos = all(texto.count(token) == 1 for token in v.TOKENS_DATA)
    if not _PADRAO_DATA.match(texto) or not tokens_unicos:
        leitor.erro(caminho, "combine AAAA, MM e DD uma vez cada")
        return None
    return texto


def _parametros_tipo(
    leitor: Leitor, dados: dict, tipo: str | None, caminho: str
) -> tuple:
    """Lê os parâmetros exigidos pelo tipo e recusa os que não se aplicam."""
    if tipo is None:
        return None, None, None, None
    exigidos = v.TIPOS[tipo]
    for chave in v.PARAMETROS_DE_TIPO:
        if chave in dados and chave not in exigidos:
            leitor.erro(caminho, f"{chave} não se aplica ao tipo {tipo}")
        elif chave in exigidos and chave not in dados:
            leitor.erro(caminho, f"o tipo {tipo} exige {chave}")
    formato = precisao = escala = separador = None
    if tipo == v.DATA and "formato" in dados:
        formato = _formato_data(leitor, dados["formato"], f"{caminho}.formato")
    if tipo == v.DECIMAL:
        precisao = leitor.inteiro(
            dados.get("precisao"),
            f"{caminho}.precisao",
            minimo=1,
            maximo=v.PRECISAO_MAXIMA,
        )
        escala = leitor.inteiro(
            dados.get("escala"), f"{caminho}.escala", minimo=0
        )
        if precisao is not None and escala is not None and escala > precisao:
            leitor.erro(caminho, "escala maior que a precisão")
        separador = leitor.escolha(
            dados.get("separador_decimal"),
            f"{caminho}.separador_decimal",
            v.SEPARADORES_DECIMAIS,
        )
    return formato, precisao, escala, separador


_CHAVES_COLUNA = (
    "cabecalho",
    "aliases",
    "ausente_na_origem",
    "nulavel",
    "formato",
    "precisao",
    "escala",
    "separador_decimal",
    "transformacoes",
    "dominio",
    "descricao",
)


def _coluna(leitor: Leitor, valor: Any, caminho: str) -> Coluna | None:
    """Lê uma coluna da entidade."""
    dados = leitor.mapa(valor, caminho, ("nome", "tipo"), _CHAVES_COLUNA)
    nome = leitor.texto(dados.get("nome"), f"{caminho}.nome", PADRAO_NOME)
    if nome is not None:
        caminho = f"colunas.{nome}"
    tipo = leitor.escolha(dados.get("tipo"), f"{caminho}.tipo", v.TIPOS)
    aliases = [
        leitor.texto(alias, f"{caminho}.aliases[{indice}]")
        for indice, alias in enumerate(
            leitor.lista(dados.get("aliases"), f"{caminho}.aliases")
        )
    ]
    formato, precisao, escala, separador = _parametros_tipo(
        leitor, dados, tipo, caminho
    )
    coluna = Coluna(
        nome=nome,
        tipo=tipo,
        cabecalho=leitor.texto(
            dados.get("cabecalho"), f"{caminho}.cabecalho", opcional=True
        ),
        aliases=tuple(alias for alias in aliases if alias is not None),
        ausente_na_origem=leitor.booleano(
            dados.get("ausente_na_origem"),
            f"{caminho}.ausente_na_origem",
            False,
        ),
        nulavel=leitor.booleano(
            dados.get("nulavel"), f"{caminho}.nulavel", True
        ),
        formato=formato,
        precisao=precisao,
        escala=escala,
        separador_decimal=separador,
        transformacoes=_transformacoes(
            leitor, dados.get("transformacoes"), f"{caminho}.transformacoes"
        ),
        dominio=leitor.texto(
            dados.get("dominio"), f"{caminho}.dominio", PADRAO_NOME, True
        ),
        descricao=leitor.texto(
            dados.get("descricao"), f"{caminho}.descricao", opcional=True
        ),
    )
    if nome is None or tipo is None:
        return None
    return coluna


def _colunas(leitor: Leitor, valor: Any) -> tuple[Coluna, ...]:
    """Lê a lista de colunas na ordem física do arquivo."""
    itens = leitor.lista(valor, "colunas", opcional=False)
    if valor is not None and not itens:
        leitor.erro("colunas", "a entidade precisa de ao menos uma coluna")
    colunas = [
        _coluna(leitor, item, f"colunas[{indice}]")
        for indice, item in enumerate(itens)
    ]
    return tuple(coluna for coluna in colunas if coluna is not None)


def _chave(leitor: Leitor, valor: Any) -> tuple[str, ...]:
    """Lê a chave declarada (lista de colunas)."""
    if valor is None:
        return ()
    itens = leitor.lista(valor, "chave")
    if not itens:
        leitor.erro("chave", "lista vazia; omita a chave quando não houver")
    nomes = [
        leitor.texto(nome, f"chave[{indice}]", PADRAO_NOME)
        for indice, nome in enumerate(itens)
    ]
    return tuple(nome for nome in nomes if nome is not None)


def _valores_literais(
    leitor: Leitor, valor: Any, caminho: str
) -> dict[str, str]:
    """Lê o mapa código -> descrição de um domínio literal."""
    if not isinstance(valor, Mapping) or not valor:
        leitor.erro(caminho, "informe o mapa código: descrição")
        return {}
    if len(valor) > LIMITE_DOMINIO_LITERAL:
        leitor.erro(
            caminho,
            f"{len(valor)} valores excedem o limite de "
            f"{LIMITE_DOMINIO_LITERAL}; use domínio por referência",
        )
    valores = {}
    for codigo, descricao in valor.items():
        if not isinstance(codigo, str):
            leitor.erro(caminho, f"código {codigo!r} deve ser texto (aspas)")
        elif not isinstance(descricao, str) or not descricao:
            leitor.erro(caminho, f"descrição de {codigo!r} deve ser texto")
        else:
            valores[codigo] = descricao
    return valores


def _dominio(
    leitor: Leitor, nome: str, valor: Any, caminho: str
) -> Dominio | None:
    """Lê um domínio literal ou por referência."""
    dados = leitor.mapa(
        valor,
        caminho,
        ("tipo",),
        ("valores", "coluna_descricao", "tabela", "coluna", "descricao"),
    )
    tipo = leitor.escolha(
        dados.get("tipo"), f"{caminho}.tipo", (LITERAL, REFERENCIA)
    )
    descricao = leitor.texto(
        dados.get("descricao"), f"{caminho}.descricao", opcional=True
    )
    proprias = {
        LITERAL: ("valores", "coluna_descricao"),
        REFERENCIA: ("tabela", "coluna"),
    }
    for outro, chaves in proprias.items():
        for chave in chaves:
            if tipo is not None and outro != tipo and chave in dados:
                leitor.erro(caminho, f"{chave} não se aplica a {tipo}")
    if tipo == LITERAL:
        return Dominio(
            nome=nome,
            tipo=tipo,
            valores=_valores_literais(
                leitor, dados.get("valores"), f"{caminho}.valores"
            ),
            coluna_descricao=leitor.texto(
                dados.get("coluna_descricao"),
                f"{caminho}.coluna_descricao",
                PADRAO_NOME,
                opcional=True,
            ),
            tabela=None,
            coluna=None,
            descricao=descricao,
        )
    if tipo == REFERENCIA:
        tabela = ler_identidade(
            leitor, dados.get("tabela"), f"{caminho}.tabela"
        )
        coluna = leitor.texto(
            dados.get("coluna"), f"{caminho}.coluna", PADRAO_NOME
        )
        if tabela is not None and coluna is not None:
            return Dominio(nome, tipo, {}, None, tabela, coluna, descricao)
    return None


def _dominios(leitor: Leitor, valor: Any) -> dict[str, Dominio]:
    """Lê o mapa de domínios nomeados."""
    if valor is None:
        return {}
    if not isinstance(valor, Mapping):
        leitor.erro("dominios", "deve ser um mapa")
        return {}
    dominios = {}
    for nome, definicao in valor.items():
        if not isinstance(nome, str) or not PADRAO_NOME.match(nome):
            leitor.erro("dominios", f"nome de domínio inválido: {nome!r}")
            continue
        dominio = _dominio(leitor, nome, definicao, f"dominios.{nome}")
        if dominio is not None:
            dominios[nome] = dominio
    return dominios


def _regra(leitor: Leitor, valor: Any, caminho: str) -> Regra | None:
    """Lê uma regra de negócio."""
    dados = leitor.mapa(
        valor,
        caminho,
        ("id", "expressao", "colunas_referenciadas", "acao", "dimensao"),
        ("descricao",),
    )
    identificador = leitor.texto(dados.get("id"), f"{caminho}.id", PADRAO_NOME)
    if identificador is not None:
        caminho = f"regras.{identificador}"
    colunas = [
        leitor.texto(nome, f"{caminho}.colunas_referenciadas[{indice}]")
        for indice, nome in enumerate(
            leitor.lista(
                dados.get("colunas_referenciadas"),
                f"{caminho}.colunas_referenciadas",
                opcional=False,
            )
        )
    ]
    regra = Regra(
        id=identificador,
        expressao=leitor.texto(dados.get("expressao"), f"{caminho}.expressao"),
        colunas_referenciadas=tuple(c for c in colunas if c is not None),
        acao=leitor.escolha(
            dados.get("acao"),
            f"{caminho}.acao",
            sorted(v.ACOES_PERMITIDAS[v.REGRA]),
        ),
        dimensao=leitor.escolha(
            dados.get("dimensao"), f"{caminho}.dimensao", v.DIMENSOES
        ),
        descricao=leitor.texto(
            dados.get("descricao"), f"{caminho}.descricao", opcional=True
        ),
    )
    if None in (regra.id, regra.expressao, regra.acao, regra.dimensao):
        return None
    return regra


def _regras(leitor: Leitor, valor: Any) -> tuple[Regra, ...]:
    """Lê a lista de regras de negócio."""
    regras = [
        _regra(leitor, item, f"regras[{indice}]")
        for indice, item in enumerate(leitor.lista(valor, "regras"))
    ]
    return tuple(regra for regra in regras if regra is not None)


def _acoes_padrao(leitor: Leitor, valor: Any) -> dict[str, str]:
    """Lê as ações das violações que não pertencem a uma regra própria."""
    dados = leitor.mapa(valor, "acoes_padrao", v.CLASSES_PADRAO)
    acoes = {}
    for classe in v.CLASSES_PADRAO:
        if classe not in dados:
            continue
        acao = leitor.escolha(
            dados[classe],
            f"acoes_padrao.{classe}",
            sorted(v.ACOES_PERMITIDAS[classe]),
        )
        if acao is not None:
            acoes[classe] = acao
    return acoes


def _reconciliacao(leitor: Leitor, valor: Any) -> TotalDeclarado | None:
    """Lê a reconciliação com total declarado pela fonte, se houver."""
    if valor is None:
        return None
    dados = leitor.mapa(valor, "reconciliacao", ("total_declarado",))
    total = leitor.mapa(
        dados.get("total_declarado"),
        "reconciliacao.total_declarado",
        ("entidade", "coluna"),
    )
    entidade = leitor.texto(
        total.get("entidade"),
        "reconciliacao.total_declarado.entidade",
        PADRAO_NOME,
    )
    coluna = leitor.texto(
        total.get("coluna"),
        "reconciliacao.total_declarado.coluna",
        PADRAO_NOME,
    )
    if entidade is None or coluna is None:
        return None
    return TotalDeclarado(entidade, coluna)


def ler_pendencias(leitor: Leitor, valor: Any, caminho: str) -> tuple:
    """Lê pendências com o estado que cada uma impede."""
    pendencias = []
    for indice, item in enumerate(leitor.lista(valor, caminho)):
        local = f"{caminho}[{indice}]"
        dados = leitor.mapa(item, local, ("descricao", "impede"))
        descricao = leitor.texto(dados.get("descricao"), f"{local}.descricao")
        impede = leitor.escolha(
            dados.get("impede"), f"{local}.impede", v.ESTADOS_IMPEDIVEIS
        )
        if descricao is not None and impede is not None:
            pendencias.append(Pendencia(descricao, impede))
    return tuple(pendencias)


def _documentacao(leitor: Leitor, valor: Any) -> Documentacao | None:
    """Lê a parte documental: estado, fonte documental e pendências."""
    dados = leitor.mapa(
        valor,
        "documentacao",
        ("estado",),
        ("fonte_documental", "pendencias", "observacoes"),
    )
    estado = leitor.escolha(
        dados.get("estado"), "documentacao.estado", v.ESTADOS
    )
    documentacao = Documentacao(
        estado=estado,
        fonte_documental=leitor.texto(
            dados.get("fonte_documental"),
            "documentacao.fonte_documental",
            opcional=True,
        ),
        pendencias=ler_pendencias(
            leitor, dados.get("pendencias"), "documentacao.pendencias"
        ),
        observacoes=leitor.texto(
            dados.get("observacoes"), "documentacao.observacoes", opcional=True
        ),
    )
    return documentacao if estado is not None else None


def _validar_cabecalhos(leitor: Leitor, contrato: Contrato) -> None:
    """Confere cabeçalho e aliases contra o dialeto declarado."""
    com_cabecalho = contrato.leitura.cabecalho
    for coluna in contrato.colunas:
        caminho = f"colunas.{coluna.nome}"
        declarou = bool(coluna.cabecalho or coluna.aliases)
        if coluna.ausente_na_origem and declarou:
            leitor.erro(caminho, "coluna ausente na origem não tem cabeçalho")
        elif not coluna.ausente_na_origem:
            if com_cabecalho and not coluna.cabecalho:
                leitor.erro(caminho, "arquivo com cabeçalho exige 'cabecalho'")
            if not com_cabecalho and declarou:
                leitor.erro(
                    caminho, "arquivo sem cabeçalho: remova 'cabecalho'"
                )
    nomes = [
        nome
        for coluna in contrato.colunas_origem
        for nome in (coluna.cabecalho, *coluna.aliases)
        if nome
    ]
    leitor.unicos(nomes, "colunas.cabecalho")


def _validar_chave(leitor: Leitor, contrato: Contrato) -> None:
    """Confere se a chave usa colunas existentes e presentes na origem."""
    leitor.unicos(contrato.chave, "chave")
    nomes = {coluna.nome: coluna for coluna in contrato.colunas}
    for nome in contrato.chave:
        coluna = nomes.get(nome)
        if coluna is None:
            leitor.erro("chave", f"coluna inexistente: {nome}")
        elif coluna.ausente_na_origem:
            leitor.erro("chave", f"coluna ausente na origem: {nome}")


def _validar_dominios(leitor: Leitor, contrato: Contrato) -> None:
    """Confere o uso dos domínios e as colunas de descrição geradas."""
    usados = set()
    for coluna in contrato.colunas:
        if coluna.dominio is None:
            continue
        caminho = f"colunas.{coluna.nome}.dominio"
        if coluna.dominio not in contrato.dominios:
            leitor.erro(caminho, f"domínio inexistente: {coluna.dominio}")
            continue
        usados.add(coluna.dominio)
        if coluna.tipo != v.TEXTO:
            leitor.erro(caminho, "domínio exige coluna do tipo texto")
    for nome in contrato.dominios:
        if nome not in usados:
            leitor.erro(f"dominios.{nome}", "domínio declarado e não usado")
    saida = contrato.colunas_saida
    leitor.unicos(saida, "colunas de saída")
    for nome in saida:
        if nome in v.COLUNAS_LINHAGEM or nome.startswith("_"):
            leitor.erro("colunas", f"nome reservado ao motor: {nome}")


def _validar_regras(leitor: Leitor, contrato: Contrato) -> None:
    """Valida identificadores e expressões das regras."""
    leitor.unicos([regra.id for regra in contrato.regras], "regras")
    saida = contrato.colunas_saida
    for regra in contrato.regras:
        problemas = validar_expressao(
            regra.expressao, regra.colunas_referenciadas, saida
        )
        for problema in problemas:
            leitor.erro(f"regras.{regra.id}.expressao", problema)


def _validar_documentacao(leitor: Leitor, contrato: Contrato) -> None:
    """Recusa estado declarado que uma pendência impede."""
    documentacao = contrato.documentacao
    if documentacao.estado == v.OBSOLETO:
        return
    ordem = v.ORDEM_ESTADOS[documentacao.estado]
    for pendencia in documentacao.pendencias:
        if ordem >= v.ORDEM_ESTADOS[pendencia.impede]:
            leitor.erro(
                "documentacao",
                f"estado {documentacao.estado} impedido pela pendência: "
                f"{pendencia.descricao}",
            )


def _validar_consistencia(leitor: Leitor, contrato: Contrato) -> None:
    """Validações que cruzam seções do contrato."""
    if not contrato.colunas_origem:
        leitor.erro("colunas", "ao menos uma coluna deve existir na origem")
    _validar_cabecalhos(leitor, contrato)
    _validar_chave(leitor, contrato)
    _validar_dominios(leitor, contrato)
    _validar_regras(leitor, contrato)
    total = contrato.total_declarado
    if total is not None and total.entidade == contrato.identidade.entidade:
        leitor.erro("reconciliacao", "total declarado aponta para si mesmo")
    _validar_documentacao(leitor, contrato)


_OBRIGATORIAS = (
    "formato_contrato",
    "identidade",
    "versao",
    "selecao",
    "leitura",
    "colunas",
    "acoes_padrao",
    "documentacao",
)
_OPCIONAIS = (
    "descricao",
    "transformacoes_padrao",
    "chave",
    "dominios",
    "regras",
    "reconciliacao",
)


def carregar_contrato(dados: Any, origem: str = "contrato") -> Contrato:
    """Lê e valida um contrato a partir do mapa declarativo.

    Args:
        dados: conteúdo do contrato (YAML ou JSON já desserializado).
        origem: nome usado nas mensagens quando a identidade é inválida.

    Returns:
        Contrato validado.

    Raises:
        ContratoInvalido: com todos os problemas encontrados.
    """
    leitor = Leitor()
    raiz = leitor.mapa(dados, "contrato", _OBRIGATORIAS, _OPCIONAIS)
    formato = leitor.inteiro(raiz.get("formato_contrato"), "formato_contrato")
    if formato is not None and formato != FORMATO_CONTRATO:
        leitor.erro("formato_contrato", f"formato não suportado: {formato}")
    identidade = ler_identidade(leitor, raiz.get("identidade"), "identidade")
    partes = {
        "formato_contrato": formato,
        "identidade": identidade,
        "versao": leitor.inteiro(raiz.get("versao"), "versao", minimo=1),
        "descricao": leitor.texto(
            raiz.get("descricao"), "descricao", opcional=True
        ),
        "selecao": _selecao(leitor, raiz.get("selecao")),
        "leitura": _leitura(leitor, raiz.get("leitura")),
        "transformacoes_padrao": _transformacoes(
            leitor, raiz.get("transformacoes_padrao"), "transformacoes_padrao"
        ),
        "colunas": _colunas(leitor, raiz.get("colunas")),
        "chave": _chave(leitor, raiz.get("chave")),
        "dominios": _dominios(leitor, raiz.get("dominios")),
        "regras": _regras(leitor, raiz.get("regras")),
        "acoes_padrao": _acoes_padrao(leitor, raiz.get("acoes_padrao")),
        "total_declarado": _reconciliacao(leitor, raiz.get("reconciliacao")),
        "documentacao": _documentacao(leitor, raiz.get("documentacao")),
    }
    nome = identidade.identificador if identidade is not None else origem
    leitor.concluir(nome)
    contrato = Contrato(**partes)
    _validar_consistencia(leitor, contrato)
    leitor.concluir(nome)
    return contrato
'''

In [ ]:
_HUB_MODULOS["hub/decisoes.py"] = r'''
"""Decisões puras do motor.

Nenhuma função aqui depende de Spark. Cada verificação devolve a lista
de problemas encontrados (vazia quando aprovada); o executor converte os
problemas em ``BloqueioPublicacao`` e registra a tentativa.
"""

from __future__ import annotations

import fnmatch
import posixpath
import re
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import dataclass

from hub import vocabulario as v

_COMPETENCIA = re.compile(r"^\d{4}-(0[1-9]|1[0-2])$")

PUBLICAR = "PUBLICAR"
RETIRAR = "QUARENTENA"
BLOQUEAR = "BLOQUEIO"


def validar_competencia(competencia: str) -> list[str]:
    """Confere o formato AAAA-MM da competência."""
    if not isinstance(competencia, str) or not _COMPETENCIA.match(competencia):
        return [f"competência inválida {competencia!r}; use AAAA-MM"]
    return []


def destino_da_linha(acoes: Iterable[str]) -> str:
    """Destino de uma linha a partir das ações de suas violações.

    Qualquer BLOQUEIA_PUBLICACAO bloqueia a publicação inteira; qualquer
    ação de quarentena retira a linha; ALERTA (ou nenhuma violação)
    mantém a linha publicada.
    """
    acoes = set(acoes)
    if v.BLOQUEIA_PUBLICACAO in acoes:
        return BLOQUEAR
    if acoes & v.ACOES_QUE_RETIRAM:
        return RETIRAR
    return PUBLICAR


def gera_alerta(acoes: Iterable[str]) -> bool:
    """Indica se alguma ação registra alerta na execução."""
    return bool(set(acoes) & v.ACOES_QUE_ALERTAM)


@dataclass(frozen=True)
class Parte:
    """Arquivo físico lido por uma entidade.

    ``parte`` é o caminho relativo à pasta da aquisição (ex.: nome do
    arquivo ou <zip sem extensão>/<membro>), estável entre aquisições.
    """

    parte: str
    caminho: str
    sha256: str
    bytes: int


def selecionar_partes(arquivos: Iterable[Mapping], padrao: str) -> list[Parte]:
    """Filtra os arquivos legíveis da aquisição pelo padrão da entidade.

    A comparação usa só o nome do arquivo e ignora maiúsculas.
    """
    selecionadas = []
    for arquivo in arquivos:
        nome = posixpath.basename(arquivo["caminho"])
        if fnmatch.fnmatchcase(nome.lower(), padrao.lower()):
            selecionadas.append(
                Parte(
                    parte=arquivo["parte"],
                    caminho=arquivo["caminho"],
                    sha256=arquivo["sha256"],
                    bytes=arquivo["bytes"],
                )
            )
    return sorted(selecionadas, key=lambda parte: parte.parte)


def verificar_quantidade_partes(
    partes: Sequence[Parte], esperada: int | None
) -> list[str]:
    """Confere se a seleção encontrou as partes contratadas."""
    if not partes:
        return ["nenhum arquivo da aquisição corresponde ao contrato"]
    if esperada is not None and len(partes) != esperada:
        return [
            f"{len(partes)} partes encontradas; contrato espera {esperada}"
        ]
    return []


def verificar_cabecalho(
    encontrado: Sequence[str], esperado: Sequence[Sequence[str]]
) -> list[str]:
    """Compara o cabeçalho físico com o layout declarado (nome e posição).

    Args:
        encontrado: nomes lidos na primeira linha do arquivo.
        esperado: por posição, o nome declarado seguido dos aliases.
    """
    nomes = [nome.strip().lstrip("﻿").strip() for nome in encontrado]
    problemas = []
    if len(nomes) != len(esperado):
        problemas.append(
            f"cabeçalho com {len(nomes)} colunas; contrato declara "
            f"{len(esperado)}"
        )
    posicao_declarada = {
        nome: posicao
        for posicao, aceitos in enumerate(esperado)
        for nome in aceitos
    }
    for posicao, nome in enumerate(nomes):
        declarada = posicao_declarada.get(nome)
        if declarada is None:
            problemas.append(f"coluna não declarada: {nome!r}")
        elif declarada != posicao:
            problemas.append(
                f"coluna {nome!r} na posição {posicao + 1}; contrato "
                f"declara {declarada + 1}"
            )
    presentes = set(nomes)
    for aceitos in esperado:
        if not presentes.intersection(aceitos):
            problemas.append(f"coluna ausente no arquivo: {aceitos[0]!r}")
    return problemas


def linhas_origem(
    fisicas: Mapping[str, int], partes: Iterable[str], cabecalho: bool
) -> int:
    """Registros esperados pela contagem física (independente do parser)."""
    desconto = 1 if cabecalho else 0
    return sum(max(fisicas.get(parte, 0) - desconto, 0) for parte in partes)


def verificar_partes_lidas(
    partes: Sequence[str],
    fisicas: Mapping[str, int],
    lidas: Mapping[str, int],
    validas: Mapping[str, int],
    cabecalho: bool,
) -> list[str]:
    """Reconcilia cada parte: linhas físicas × registros lidos.

    Também exige que toda parte contratada contribua com ao menos uma
    linha válida: arquivo sem registros é anomalia, não sucesso.
    """
    problemas = []
    desconto = 1 if cabecalho else 0
    for parte in partes:
        esperado = max(fisicas.get(parte, 0) - desconto, 0)
        encontrado = lidas.get(parte, 0)
        if encontrado != esperado:
            problemas.append(
                f"parte {parte}: {esperado} linhas físicas e {encontrado} "
                "registros lidos"
            )
        if validas.get(parte, 0) == 0:
            problemas.append(f"parte {parte} sem nenhuma linha válida")
    for parte in sorted((set(lidas) | set(fisicas)) - set(partes)):
        problemas.append(f"parte não contratada foi lida: {parte}")
    return problemas


def verificar_tolerancia(
    malformadas: int, lidas: int, tolerancia: float
) -> list[str]:
    """Acima da tolerância, linhas malformadas são divergência estrutural."""
    if lidas == 0 or malformadas / lidas <= tolerancia:
        return []
    return [
        f"{malformadas} de {lidas} linhas malformadas excedem a tolerância "
        f"de {tolerancia:.4%}: divergência estrutural"
    ]


def verificar_total_declarado(total: int, lidas: int) -> list[str]:
    """Compara os registros lidos com o total publicado pela fonte."""
    if total == lidas:
        return []
    return [f"total declarado pela fonte ({total}) difere do lido ({lidas})"]


def verificar_reconciliacao(
    origem: int, publicadas: int, quarentena: int
) -> list[str]:
    """Identidade obrigatória: origem = publicadas + quarentena."""
    if origem == publicadas + quarentena:
        return []
    return [
        f"reconciliação falhou: origem {origem} != publicadas {publicadas}"
        f" + quarentena {quarentena}"
    ]


def verificar_competencia(
    solicitada: str, vigente: str | None, permitir_anterior: bool
) -> list[str]:
    """Snapshot não regride de competência sem pedido explícito."""
    if vigente is None or solicitada >= vigente or permitir_anterior:
        return []
    return [
        f"competência {solicitada} é anterior à vigente {vigente}; use "
        "permitir_competencia_anterior para reverter"
    ]


def verificar_versao(
    versao: int, hash_contrato: str, hashes_por_versao: Mapping[int, set]
) -> list[str]:
    """Mesma versão com parte executável diferente é mudança silenciosa."""
    publicados = set(hashes_por_versao.get(versao, set()))
    if publicados - {hash_contrato}:
        return [
            f"versão {versao} já foi publicada com outro hash_contrato; "
            "incremente a versão do contrato"
        ]
    return []


def verificar_procedencia(
    publicacao_bronze: Mapping | None, competencia: str, hash_leitura: str
) -> list[str]:
    """A Silver só lê Bronze publicada pela mesma seção de leitura."""
    if publicacao_bronze is None:
        return ["não há publicação Bronze efetiva neste ambiente"]
    problemas = []
    if publicacao_bronze["competencia"] != competencia:
        problemas.append(
            f"Bronze vigente é da competência "
            f"{publicacao_bronze['competencia']}; solicitada {competencia}"
        )
    if publicacao_bronze["hash_leitura"] != hash_leitura:
        problemas.append(
            "Bronze vigente foi publicada com outro hash_leitura; "
            "reprocesse a Bronze"
        )
    return problemas


def verificar_snapshot_bronze(
    contagens: Mapping[tuple[str, str], int], publicacao_bronze: Mapping
) -> list[str]:
    """Confere se a tabela Bronze contém exatamente a publicação registrada.

    Args:
        contagens: linhas por (id_execucao, competencia) lidas da tabela.
        publicacao_bronze: registro da publicação Bronze efetiva.
    """
    esperado = (
        publicacao_bronze["id_execucao"],
        publicacao_bronze["competencia"],
    )
    problemas = []
    if set(contagens) != {esperado}:
        problemas.append(
            f"tabela Bronze contém {sorted(contagens)}; esperado {esperado}"
        )
    total = sum(contagens.values())
    if total != publicacao_bronze["linhas_publicadas"]:
        problemas.append(
            f"tabela Bronze tem {total} linhas; publicação registrou "
            f"{publicacao_bronze['linhas_publicadas']}"
        )
    return problemas


@dataclass(frozen=True)
class DiferencaSchema:
    """Diferença entre o schema físico atual e o planejado."""

    adicionadas: tuple[str, ...]
    removidas: tuple[str, ...]
    alteradas: tuple[str, ...]
    reordenada: bool

    @property
    def vazia(self) -> bool:
        """Indica schemas equivalentes."""
        return not (
            self.adicionadas
            or self.removidas
            or self.alteradas
            or self.reordenada
        )

    def descrever(self) -> str:
        """Descrição legível da diferença."""
        partes = []
        if self.adicionadas:
            partes.append(f"adicionadas {list(self.adicionadas)}")
        if self.removidas:
            partes.append(f"removidas {list(self.removidas)}")
        if self.alteradas:
            partes.append(f"tipo alterado {list(self.alteradas)}")
        if self.reordenada:
            partes.append("ordem alterada")
        return "; ".join(partes)


def diferenca_schema(
    atual: Sequence[tuple[str, str]], planejado: Sequence[tuple[str, str]]
) -> DiferencaSchema:
    """Compara nomes, tipos e ordem das colunas."""
    tipos_atuais = dict(atual)
    tipos_planejados = dict(planejado)
    comuns_atual = [nome for nome, _ in atual if nome in tipos_planejados]
    comuns_planejado = [nome for nome, _ in planejado if nome in tipos_atuais]
    return DiferencaSchema(
        adicionadas=tuple(
            nome for nome, _ in planejado if nome not in tipos_atuais
        ),
        removidas=tuple(
            nome for nome, _ in atual if nome not in tipos_planejados
        ),
        alteradas=tuple(
            nome
            for nome, tipo in planejado
            if nome in tipos_atuais and tipos_atuais[nome] != tipo
        ),
        reordenada=comuns_atual != comuns_planejado,
    )


def verificar_mudanca_schema(
    diferenca: DiferencaSchema,
    versao_contrato: int,
    versao_vigente: int | None,
) -> list[str]:
    """Mudança de schema só entra com nova versão de contrato."""
    if diferenca.vazia:
        return []
    if versao_vigente is None:
        return [
            "tabela existente diverge do contrato e não tem publicação "
            f"registrada: {diferenca.descrever()}"
        ]
    if versao_contrato <= versao_vigente:
        return [
            f"schema diverge do publicado pela versão {versao_vigente} sem "
            f"nova versão de contrato: {diferenca.descrever()}"
        ]
    return []


_GRAVIDADE_DIAGNOSTICO = (v.CONFERIDA, v.COM_ORFAOS, v.REFERENCIA_AUSENTE)


def resultado_diagnostico(itens: Iterable[Mapping]) -> str:
    """Resultado geral do diagnóstico: o relacionamento mais grave."""
    resultados = [item["resultado"] for item in itens]
    if not resultados:
        return v.CONFERIDA
    return max(resultados, key=_GRAVIDADE_DIAGNOSTICO.index)
'''

In [ ]:
_HUB_MODULOS["hub/erros.py"] = r'''
"""Exceções do motor."""


class ContratoInvalido(ValueError):
    """Contrato, descritor de base ou configuração fora do formato."""

    def __init__(self, origem: str, problemas: list[str]) -> None:
        self.origem = origem
        self.problemas = list(problemas)
        detalhe = "\n".join(f"  - {p}" for p in self.problemas)
        super().__init__(
            f"{origem}: {len(self.problemas)} problema(s)\n{detalhe}"
        )


class BloqueioPublicacao(Exception):
    """Interrupção decidida pelo motor; a execução fica BLOQUEADA."""

    def __init__(self, problemas: list[str]) -> None:
        self.problemas = list(problemas)
        super().__init__("; ".join(self.problemas))


def exigir(problemas: list[str]) -> None:
    """Interrompe a publicação quando a lista de problemas não é vazia.

    Args:
        problemas: resultado de uma verificação pura (vazio = aprovado).

    Raises:
        BloqueioPublicacao: se houver ao menos um problema.
    """
    if problemas:
        raise BloqueioPublicacao(problemas)
'''

In [ ]:
_HUB_MODULOS["hub/execucao.py"] = r'''
"""Registro operacional de execução e esquemas das tabelas do motor.

Cada tentativa (aquisição, inspeção, publicação, aprovação, diagnóstico
ou catálogo) gera uma linha em ``controle.execucoes``, inclusive as que
falham ou são bloqueadas. O ``id_execucao`` distingue tentativas e nunca
é usado para deduplicar publicações.
"""

from __future__ import annotations

import json
import secrets
from dataclasses import dataclass, field
from datetime import UTC, datetime

from hub import vocabulario as v

TABELA_EXECUCOES = "execucoes"
TABELA_QUARENTENA = "quarentena"
LIMITE_MOTIVO = 4000

COLUNAS_EXECUCOES = (
    ("id_execucao", "STRING"),
    ("identificador", "STRING"),
    ("tipo", "STRING"),
    ("status", "STRING"),
    ("ambiente", "STRING"),
    ("fonte", "STRING"),
    ("base", "STRING"),
    ("entidade", "STRING"),
    ("competencia", "STRING"),
    ("tabela_destino", "STRING"),
    ("versao_contrato", "INT"),
    ("estado_contrato", "STRING"),
    ("hash_leitura", "STRING"),
    ("hash_contrato", "STRING"),
    ("publication_fingerprint", "STRING"),
    ("id_execucao_origem", "STRING"),
    ("versao_delta", "BIGINT"),
    ("linhas_origem", "BIGINT"),
    ("linhas_publicadas", "BIGINT"),
    ("linhas_quarentena", "BIGINT"),
    ("resultado", "STRING"),
    ("responsavel", "STRING"),
    ("motivo", "STRING"),
    ("detalhes", "STRING"),
    ("versao_motor", "STRING"),
    ("iniciado_em", "TIMESTAMP"),
    ("finalizado_em", "TIMESTAMP"),
)

COLUNAS_QUARENTENA = (
    ("id_quarentena", "STRING"),
    ("identificador", "STRING"),
    ("camada", "STRING"),
    ("fonte", "STRING"),
    ("base", "STRING"),
    ("entidade", "STRING"),
    ("competencia", "STRING"),
    ("publication_fingerprint", "STRING"),
    ("versao_contrato", "INT"),
    ("hash_leitura", "STRING"),
    ("hash_contrato", "STRING"),
    ("id_execucao", "STRING"),
    ("chave", "STRING"),
    ("grupo_problema", "STRING"),
    ("motivos", "STRING"),
    ("registro", "STRING"),
    ("arquivo_origem", "STRING"),
    ("ocorrencia", "INT"),
    ("registrado_em", "TIMESTAMP"),
)

# As duas tabelas são particionadas pelo identificador lógico: execuções
# paralelas de entidades diferentes não disputam os mesmos arquivos Delta.
COLUNA_PARTICAO = "identificador"


def agora_utc() -> datetime:
    """Instante atual em UTC."""
    return datetime.now(UTC)


def novo_id_execucao(
    momento: datetime | None = None, sufixo: str | None = None
) -> str:
    """Identificador da tentativa no formato AAAAMMDDTHHMMSSZ-<8 hex>."""
    momento = momento or agora_utc()
    sufixo = sufixo or secrets.token_hex(4)
    return f"{momento:%Y%m%dT%H%M%SZ}-{sufixo}"


@dataclass
class RegistroExecucao:
    """Estado de uma tentativa, gravado no início e no fim da execução."""

    id_execucao: str
    identificador: str
    tipo: str
    ambiente: str
    versao_motor: str
    fonte: str | None = None
    base: str | None = None
    entidade: str | None = None
    competencia: str | None = None
    status: str = v.EM_EXECUCAO
    tabela_destino: str | None = None
    versao_contrato: int | None = None
    estado_contrato: str | None = None
    hash_leitura: str | None = None
    hash_contrato: str | None = None
    publication_fingerprint: str | None = None
    id_execucao_origem: str | None = None
    versao_delta: int | None = None
    linhas_origem: int | None = None
    linhas_publicadas: int | None = None
    linhas_quarentena: int | None = None
    resultado: str | None = None
    responsavel: str | None = None
    motivo: str | None = None
    detalhes: dict = field(default_factory=dict)
    alertas: list[str] = field(default_factory=list)
    iniciado_em: datetime = field(default_factory=agora_utc)
    finalizado_em: datetime | None = None

    def alertar(self, mensagem: str) -> None:
        """Registra um alerta; a execução termina SUCESSO_COM_ALERTA."""
        self.alertas.append(mensagem)

    def status_sucesso(self) -> str:
        """Status de término sem bloqueio nem falha."""
        return v.SUCESSO_COM_ALERTA if self.alertas else v.SUCESSO

    def finalizar(
        self,
        status: str,
        motivo: str | None = None,
        momento: datetime | None = None,
    ) -> None:
        """Encerra a tentativa com o status e o motivo informados."""
        self.status = status
        if motivo:
            self.motivo = motivo[:LIMITE_MOTIVO]
        self.finalizado_em = momento or agora_utc()

    def detalhes_json(self) -> str | None:
        """Detalhes e alertas serializados como JSON (coluna escalar)."""
        detalhes = dict(self.detalhes)
        if self.alertas:
            detalhes["alertas"] = list(self.alertas)
        if not detalhes:
            return None
        return json.dumps(detalhes, ensure_ascii=False, sort_keys=True)

    def para_linha(self) -> tuple:
        """Valores na ordem de COLUNAS_EXECUCOES."""
        valores = []
        for nome, _ in COLUNAS_EXECUCOES:
            if nome == "detalhes":
                valores.append(self.detalhes_json())
            else:
                valores.append(getattr(self, nome))
        return tuple(valores)

    def resumo(self) -> dict:
        """Resumo exibido no notebook (não é evidência)."""
        return {
            "id_execucao": self.id_execucao,
            "tipo": self.tipo,
            "identificador": self.identificador,
            "competencia": self.competencia,
            "status": self.status,
            "resultado": self.resultado,
            "linhas_origem": self.linhas_origem,
            "linhas_publicadas": self.linhas_publicadas,
            "linhas_quarentena": self.linhas_quarentena,
            "motivo": self.motivo,
        }
'''

In [ ]:
_HUB_MODULOS["hub/expressao.py"] = r'''
"""Validação das expressões SQL declaradas nas regras do contrato.

A expressão só é executada depois de aprovada aqui. Ela pode usar apenas:

- colunas listadas em ``colunas_referenciadas``;
- palavras e funções do vocabulário fechado (``hub.vocabulario``);
- literais numéricos e de texto entre aspas simples;
- operadores de comparação, aritméticos, parênteses e vírgula.

Comentário, ponto e vírgula, ponto (encadeamento), aspas duplas, crase e
qualquer identificador fora da lista reprovam a regra.
"""

import re
from collections.abc import Iterable

from hub.vocabulario import FUNCOES_EXPRESSAO, PALAVRAS_EXPRESSAO

_TOKEN = re.compile(
    r"""
      (?P<espaco>\s+)
    | (?P<texto>'(?:[^']|'')*')
    | (?P<numero>\d+(?:\.\d+)?)
    | (?P<palavra>[A-Za-z_][A-Za-z0-9_]*)
    | (?P<operador><=|>=|<>|!=|=|<|>|\+|-|\*|/|%|\(|\)|,)
    """,
    re.VERBOSE,
)
_TEXTO_LITERAL = re.compile(r"'(?:[^']|'')*'")
_MARCADORES_PROIBIDOS = ("--", "/*", "*/", ";")


def tokenizar(expressao: str) -> list[tuple[str, str]]:
    """Divide a expressão em tokens ``(tipo, valor)``, sem espaços.

    Raises:
        ValueError: se houver caractere fora do vocabulário léxico.
    """
    tokens = []
    posicao = 0
    while posicao < len(expressao):
        encontrado = _TOKEN.match(expressao, posicao)
        if encontrado is None:
            caractere = expressao[posicao]
            raise ValueError(
                f"caractere não permitido na posição {posicao}: "
                f"{caractere!r}"
            )
        if encontrado.lastgroup != "espaco":
            tokens.append((encontrado.lastgroup, encontrado.group()))
        posicao = encontrado.end()
    return tokens


def _marcadores_proibidos(expressao: str) -> list[str]:
    """Procura comentário e separador de comando fora de literais."""
    sem_literais = _TEXTO_LITERAL.sub("''", expressao)
    return [
        f"marcador proibido: {marcador!r}"
        for marcador in _MARCADORES_PROIBIDOS
        if marcador in sem_literais
    ]


def _parenteses_balanceados(tokens: list[tuple[str, str]]) -> bool:
    """Confere se os parênteses abrem e fecham na ordem correta."""
    profundidade = 0
    for tipo, valor in tokens:
        if tipo != "operador":
            continue
        if valor == "(":
            profundidade += 1
        elif valor == ")":
            profundidade -= 1
            if profundidade < 0:
                return False
    return profundidade == 0


def identificadores_usados(tokens: list[tuple[str, str]]) -> list[str]:
    """Lista, em ordem, os identificadores que não são palavra nem função."""
    usados = []
    for indice, (tipo, valor) in enumerate(tokens):
        if tipo != "palavra" or valor.upper() in PALAVRAS_EXPRESSAO:
            continue
        seguinte = tokens[indice + 1][1] if indice + 1 < len(tokens) else ""
        if seguinte == "(":
            continue
        usados.append(valor)
    return usados


def _funcoes_nao_permitidas(tokens: list[tuple[str, str]]) -> list[str]:
    """Lista problemas de funções fora do vocabulário."""
    problemas = []
    for indice, (tipo, valor) in enumerate(tokens):
        if tipo != "palavra" or valor.upper() in PALAVRAS_EXPRESSAO:
            continue
        seguinte = tokens[indice + 1][1] if indice + 1 < len(tokens) else ""
        if seguinte == "(" and valor.lower() not in FUNCOES_EXPRESSAO:
            problemas.append(f"função não permitida: {valor}")
    return problemas


def validar_expressao(
    expressao: str,
    colunas_referenciadas: Iterable[str],
    colunas_existentes: Iterable[str],
) -> list[str]:
    """Valida uma expressão de regra antes de qualquer execução.

    Args:
        expressao: predicado SQL declarado no contrato.
        colunas_referenciadas: colunas que a regra declara usar.
        colunas_existentes: colunas de saída disponíveis na entidade.

    Returns:
        Lista de problemas; vazia quando a expressão é aceita.
    """
    if not isinstance(expressao, str) or not expressao.strip():
        return ["expressão vazia"]
    referenciadas = list(colunas_referenciadas)
    existentes = set(colunas_existentes)
    problemas = _marcadores_proibidos(expressao)
    try:
        tokens = tokenizar(expressao)
    except ValueError as erro:
        return problemas + [str(erro)]
    if not _parenteses_balanceados(tokens):
        problemas.append("parênteses desbalanceados")
    problemas.extend(_funcoes_nao_permitidas(tokens))
    usados = identificadores_usados(tokens)
    for nome in dict.fromkeys(usados):
        if nome not in referenciadas:
            problemas.append(
                f"identificador fora de colunas_referenciadas: {nome}"
            )
    for nome in referenciadas:
        if nome not in existentes:
            problemas.append(f"coluna referenciada inexistente: {nome}")
        if nome not in usados:
            problemas.append(f"coluna referenciada e não usada: {nome}")
    return problemas
'''

In [ ]:
_HUB_MODULOS["hub/hashes.py"] = r'''
"""Hashes determinísticos do Hub.

Modelo deliberadamente simples:

- ``arquivo_hash``: SHA-256 de cada arquivo preservado na Raw;
- ``publication_fingerprint``: SHA-256 do conjunto ordenado de partes;
- ``hash_leitura``: parte executável que a Bronze usa (seleção, dialeto,
  layout de origem, ação para linha malformada e total declarado);
- ``hash_contrato``: toda a parte executável do contrato.

Documentação (descrições, estado, pendências) e ``versao`` ficam fora dos
hashes. Não existe hash semântico, físico, de plano ou de DataFrame.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Iterable
from typing import Any

from hub import vocabulario as v
from hub.contrato import Contrato, Identidade, Transformacao

TAMANHO_BLOCO = 1 << 20


def canonico(valor: Any) -> str:
    """Serializa em JSON canônico: chaves ordenadas e sem espaços."""
    return json.dumps(
        valor, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    )


def sha256_texto(texto: str) -> str:
    """SHA-256 hexadecimal de um texto em UTF-8."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()


def sha256_arquivo(caminho: str, tamanho_bloco: int = TAMANHO_BLOCO) -> str:
    """SHA-256 hexadecimal de um arquivo, lido em blocos."""
    resumo = hashlib.sha256()
    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(tamanho_bloco), b""):
            resumo.update(bloco)
    return resumo.hexdigest()


def _identidade(identidade: Identidade) -> dict:
    """Representação canônica da identidade lógica."""
    return {
        "fonte": identidade.fonte,
        "base": identidade.base,
        "entidade": identidade.entidade,
        "especificacao": identidade.especificacao,
    }


def _transformacoes(transformacoes: Iterable[Transformacao]) -> list:
    """Representação canônica de uma lista de transformações."""
    return [
        {"tipo": t.tipo, "parametros": dict(t.parametros)}
        for t in transformacoes
    ]


def parte_leitura(contrato: Contrato) -> dict:
    """Parte executável usada pela Bronze."""
    total = contrato.total_declarado
    return {
        "formato_contrato": contrato.formato_contrato,
        "identidade": _identidade(contrato.identidade),
        "selecao": {
            "arquivo": contrato.selecao.arquivo,
            "quantidade_partes": contrato.selecao.quantidade_partes,
        },
        "leitura": {
            "separador": contrato.leitura.separador,
            "aspas": contrato.leitura.aspas,
            "escape": contrato.leitura.escape,
            "codificacao": contrato.leitura.codificacao,
            "cabecalho": contrato.leitura.cabecalho,
            "tolerancia_malformadas": contrato.leitura.tolerancia_malformadas,
        },
        "layout": [
            {
                "nome": coluna.nome,
                "cabecalho": coluna.cabecalho,
                "aliases": list(coluna.aliases),
                "ausente_na_origem": coluna.ausente_na_origem,
            }
            for coluna in contrato.colunas
        ],
        "linha_malformada": contrato.acoes_padrao[v.LINHA_MALFORMADA],
        "total_declarado": (
            None
            if total is None
            else {"entidade": total.entidade, "coluna": total.coluna}
        ),
    }


def parte_executavel(contrato: Contrato) -> dict:
    """Parte executável completa (Bronze e Silver)."""
    parte = parte_leitura(contrato)
    parte["transformacoes_padrao"] = _transformacoes(
        contrato.transformacoes_padrao
    )
    parte["colunas"] = [
        {
            "nome": coluna.nome,
            "tipo": coluna.tipo,
            "nulavel": coluna.nulavel,
            "formato": coluna.formato,
            "precisao": coluna.precisao,
            "escala": coluna.escala,
            "separador_decimal": coluna.separador_decimal,
            "transformacoes": _transformacoes(coluna.transformacoes),
            "dominio": coluna.dominio,
        }
        for coluna in contrato.colunas
    ]
    parte["chave"] = list(contrato.chave)
    parte["dominios"] = {
        nome: {
            "tipo": dominio.tipo,
            "valores": dict(dominio.valores),
            "coluna_descricao": dominio.coluna_descricao,
            "tabela": (
                None if dominio.tabela is None else _identidade(dominio.tabela)
            ),
            "coluna": dominio.coluna,
        }
        for nome, dominio in contrato.dominios.items()
    }
    parte["regras"] = [
        {
            "id": regra.id,
            "expressao": regra.expressao,
            "colunas_referenciadas": list(regra.colunas_referenciadas),
            "acao": regra.acao,
            "dimensao": regra.dimensao,
        }
        for regra in contrato.regras
    ]
    parte["acoes_padrao"] = dict(contrato.acoes_padrao)
    return parte


def hash_leitura(contrato: Contrato) -> str:
    """Hash da parte executável usada pela Bronze."""
    return sha256_texto(canonico(parte_leitura(contrato)))


def hash_contrato(contrato: Contrato) -> str:
    """Hash da parte executável completa do contrato."""
    return sha256_texto(canonico(parte_executavel(contrato)))


def fingerprint(partes: Iterable[tuple[str, str]]) -> str:
    """Fingerprint da publicação: partes ordenadas por identificador.

    Args:
        partes: pares (identificador da parte, SHA-256 do arquivo).
    """
    linhas = [f"{parte}|{resumo}" for parte, resumo in sorted(partes)]
    return sha256_texto("\n".join(linhas))
'''

In [ ]:
_HUB_MODULOS["hub/historico.py"] = r'''
"""Consultas puras sobre o histórico de execuções de um identificador.

O executor Spark lê as linhas de ``controle.execucoes`` de um
identificador e as entrega aqui como dicionários. Assim, as decisões de
lastro, procedência, versão e competência são testáveis sem Spark.
"""

from __future__ import annotations

from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from datetime import datetime

from hub import vocabulario as v


@dataclass(frozen=True)
class Evidencias:
    """Evidências registradas no ambiente para o ciclo de vida."""

    inspecoes_compativeis: frozenset[str]
    aprovacoes: frozenset[str]


def _momento(registro: Mapping) -> datetime:
    """Instante de ordenação: fim da execução ou, na falta, o início."""
    momento = registro.get("finalizado_em") or registro.get("iniciado_em")
    return momento or datetime.min


class Historico:
    """Execuções de um identificador, da mais recente para a mais antiga."""

    def __init__(self, registros: Iterable[Mapping]) -> None:
        self._registros = sorted(
            (dict(registro) for registro in registros),
            key=_momento,
            reverse=True,
        )

    def efetivos(self, tipo: str) -> list[dict]:
        """Execuções do tipo com status de publicação efetiva."""
        return [
            registro
            for registro in self._registros
            if registro["tipo"] == tipo
            and registro["status"] in v.STATUS_EFETIVOS
        ]

    def ultima_efetiva(self, tipo: str) -> dict | None:
        """Execução efetiva mais recente do tipo, se houver."""
        efetivos = self.efetivos(tipo)
        return efetivos[0] if efetivos else None

    def aquisicao_efetiva(self, competencia: str) -> dict | None:
        """Aquisição mais recente que trouxe arquivos para a competência."""
        for registro in self.efetivos(v.AQUISICAO):
            if (
                registro["competencia"] == competencia
                and registro["resultado"] == v.ADQUIRIDA
            ):
                return registro
        return None

    def evidencias(self) -> Evidencias:
        """Inspeções compatíveis e aprovações registradas."""
        inspecoes = {
            registro["hash_leitura"]
            for registro in self.efetivos(v.INSPECAO)
            if registro["resultado"] == v.COMPATIVEL
        }
        aprovacoes = {
            registro["hash_contrato"]
            for registro in self.efetivos(v.APROVACAO)
            if registro["resultado"] == v.APROVADA
        }
        return Evidencias(frozenset(inspecoes), frozenset(aprovacoes))

    def hashes_por_versao(self) -> dict[int, set[str]]:
        """hash_contrato de cada versão já publicada (Bronze ou Silver)."""
        resultado: dict[int, set[str]] = {}
        for tipo in (v.PUBLICACAO_BRONZE, v.PUBLICACAO_SILVER):
            for registro in self.efetivos(tipo):
                versao = registro["versao_contrato"]
                resultado.setdefault(versao, set()).add(
                    registro["hash_contrato"]
                )
        return resultado
'''

In [ ]:
_HUB_MODULOS["hub/inspecao.py"] = r'''
"""Inspeção física sob demanda: na descoberta ou diante de mudança.

Não é etapa mensal. Lê apenas o início de cada parte, no driver, e
compara com o contrato: codificação, terminador de linha, separador
provável, cabeçalho, largura dos registros e amostras de valores. O
resultado é registrado como evidência (COMPATIVEL ou INCOMPATIVEL).
"""

from __future__ import annotations

from collections import Counter
from collections.abc import Iterable

from hub import vocabulario as v
from hub.arquivos import decodificar, ler_inicio, registros_csv
from hub.contrato import Contrato
from hub.decisoes import verificar_cabecalho

CANDIDATOS_SEPARADOR = (";", ",", "|", "\t")
LIMITE_REGISTROS = 1000
AMOSTRAS_POR_COLUNA = 3
TAMANHO_AMOSTRA = 60
_BOM_UTF8 = b"\xef\xbb\xbf"


def terminador_linha(dados: bytes) -> str:
    """Classifica o terminador de linha predominante na amostra."""
    crlf = dados.count(b"\r\n")
    lf = dados.count(b"\n") - crlf
    if crlf and lf:
        return "misto"
    if crlf:
        return "CRLF"
    return "LF" if lf else "indefinido"


def separador_provavel(linhas: Iterable[str]) -> str | None:
    """Candidato presente em todas as linhas com a contagem mais estável.

    Heurística apenas informativa: conta ocorrências sem considerar aspas.
    """
    linhas = [linha for linha in linhas if linha][:50]
    melhor, melhor_nota = None, (0, 0)
    for candidato in CANDIDATOS_SEPARADOR:
        contagens = [linha.count(candidato) for linha in linhas]
        if not contagens or min(contagens) == 0:
            continue
        valor, frequencia = Counter(contagens).most_common(1)[0]
        nota = (frequencia, valor)
        if nota > melhor_nota:
            melhor, melhor_nota = candidato, nota
    return melhor


def _amostras(
    registros: list[list[str]], nomes: list[str]
) -> dict[str, list[str]]:
    """Até três valores distintos e não vazios por coluna."""
    amostras: dict[str, list[str]] = {nome: [] for nome in nomes}
    for registro in registros:
        if len(registro) != len(nomes):
            continue
        for nome, valor in zip(nomes, registro, strict=True):
            valores = amostras[nome]
            valor = valor[:TAMANHO_AMOSTRA]
            if valor and valor not in valores:
                if len(valores) < AMOSTRAS_POR_COLUNA:
                    valores.append(valor)
    return amostras


def inspecionar_parte(caminho: str, parte: str, contrato: Contrato) -> dict:
    """Inspeciona o início de uma parte contra o contrato.

    Args:
        caminho: caminho local do arquivo.
        parte: identificador da parte na aquisição.
        contrato: contrato da entidade.

    Returns:
        Relatório da parte, com a lista de problemas e o veredito.
    """
    leitura = contrato.leitura
    dados = ler_inicio(caminho)
    texto, decodificou = decodificar(dados, leitura.codificacao)
    _, utf8 = decodificar(dados, "utf-8")
    registros = registros_csv(texto, leitura)[: LIMITE_REGISTROS + 1]
    problemas = []
    if not decodificou:
        problemas.append(f"amostra não decodifica como {leitura.codificacao}")
    nao_ascii = any(byte > 127 for byte in dados)
    if leitura.codificacao != "utf-8" and utf8 and nao_ascii:
        problemas.append(
            "amostra com acentuação válida em UTF-8; a codificação "
            f"declarada ({leitura.codificacao}) corromperia os acentos"
        )
    provavel = separador_provavel(texto.splitlines())
    if provavel is not None and provavel != leitura.separador:
        problemas.append(
            f"separador provável {provavel!r} difere do declarado "
            f"{leitura.separador!r}"
        )
    origem = contrato.colunas_origem
    cabecalho = None
    if leitura.cabecalho and registros:
        cabecalho = registros.pop(0)
        esperado = [(c.cabecalho, *c.aliases) for c in origem]
        problemas.extend(verificar_cabecalho(cabecalho, esperado))
    if not registros:
        problemas.append("amostra sem registros de dados")
    larguras = Counter(len(registro) for registro in registros)
    divergentes = sum(
        quantidade
        for largura, quantidade in larguras.items()
        if largura != len(origem)
    )
    if divergentes:
        problemas.append(
            f"{divergentes} de {len(registros)} registros da amostra com "
            f"largura diferente de {len(origem)}"
        )
    return {
        "parte": parte,
        "bytes_lidos": len(dados),
        "codificacao_declarada": leitura.codificacao,
        "decodifica_declarada": decodificou,
        "decodifica_utf8": utf8,
        "bom_utf8": dados.startswith(_BOM_UTF8),
        "terminador": terminador_linha(dados),
        "separador_declarado": leitura.separador,
        "separador_provavel": provavel,
        "cabecalho_encontrado": cabecalho,
        "registros_amostra": len(registros),
        "largura_esperada": len(origem),
        "larguras": {str(k): n for k, n in sorted(larguras.items())},
        "amostras": _amostras(registros, [c.nome for c in origem]),
        "problemas": problemas,
        "veredito": v.INCOMPATIVEL if problemas else v.COMPATIVEL,
    }


def inspecionar(partes: Iterable[tuple[str, str]], contrato: Contrato) -> dict:
    """Inspeciona todas as partes selecionadas para a entidade.

    Args:
        partes: pares (identificador da parte, caminho local).
        contrato: contrato da entidade.

    Returns:
        Relatório com o veredito geral e o relatório de cada parte.
    """
    relatorios = [
        inspecionar_parte(caminho, parte, contrato)
        for parte, caminho in partes
    ]
    compativel = bool(relatorios) and all(
        relatorio["veredito"] == v.COMPATIVEL for relatorio in relatorios
    )
    return {
        "veredito": v.COMPATIVEL if compativel else v.INCOMPATIVEL,
        "partes": relatorios,
    }
'''

In [ ]:
_HUB_MODULOS["hub/leitor.py"] = r'''
"""Leitura validada de estruturas declarativas.

Usado para contratos, descritores de base e configuração de ambiente. O
leitor acumula todos os problemas com o caminho do campo, para que a
validação reporte tudo de uma vez em vez de parar no primeiro erro.
"""

import re
from collections.abc import Iterable, Mapping
from typing import Any

from hub.erros import ContratoInvalido


class Leitor:
    """Acumula problemas de validação com o caminho do campo."""

    def __init__(self) -> None:
        self.problemas: list[str] = []

    def erro(self, caminho: str, mensagem: str) -> None:
        """Registra um problema no campo indicado."""
        self.problemas.append(f"{caminho}: {mensagem}")

    def concluir(self, origem: str) -> None:
        """Encerra a leitura.

        Raises:
            ContratoInvalido: se algum problema foi registrado.
        """
        if self.problemas:
            raise ContratoInvalido(origem, self.problemas)

    def mapa(
        self,
        valor: Any,
        caminho: str,
        obrigatorias: Iterable[str] = (),
        opcionais: Iterable[str] = (),
    ) -> dict:
        """Lê um mapa e confere chaves obrigatórias e desconhecidas."""
        if not isinstance(valor, Mapping):
            self.erro(caminho, "deve ser um mapa")
            return {}
        obrigatorias = tuple(obrigatorias)
        permitidas = set(obrigatorias) | set(opcionais)
        for chave in valor:
            if chave not in permitidas:
                self.erro(caminho, f"chave desconhecida: {chave}")
        for chave in obrigatorias:
            if chave not in valor:
                self.erro(caminho, f"chave obrigatória ausente: {chave}")
        return dict(valor)

    def texto(
        self,
        valor: Any,
        caminho: str,
        padrao: re.Pattern | None = None,
        opcional: bool = False,
    ) -> str | None:
        """Lê um texto não vazio, opcionalmente conferindo um padrão."""
        if valor is None:
            if not opcional:
                self.erro(caminho, "obrigatório")
            return None
        if not isinstance(valor, str) or valor == "":
            self.erro(caminho, "deve ser texto não vazio")
            return None
        if padrao is not None and not padrao.match(valor):
            self.erro(caminho, f"fora do padrão {padrao.pattern}: {valor!r}")
            return None
        return valor

    def inteiro(
        self,
        valor: Any,
        caminho: str,
        minimo: int | None = None,
        maximo: int | None = None,
        opcional: bool = False,
    ) -> int | None:
        """Lê um inteiro (booleano não é aceito) dentro dos limites."""
        if valor is None:
            if not opcional:
                self.erro(caminho, "obrigatório")
            return None
        if isinstance(valor, bool) or not isinstance(valor, int):
            self.erro(caminho, "deve ser inteiro")
            return None
        if minimo is not None and valor < minimo:
            self.erro(caminho, f"deve ser maior ou igual a {minimo}")
            return None
        if maximo is not None and valor > maximo:
            self.erro(caminho, f"deve ser menor ou igual a {maximo}")
            return None
        return valor

    def fracao(self, valor: Any, caminho: str) -> float | None:
        """Lê um número no intervalo [0, 1)."""
        if valor is None:
            self.erro(caminho, "obrigatório")
            return None
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            self.erro(caminho, "deve ser número")
            return None
        if not 0 <= valor < 1:
            self.erro(caminho, "deve estar no intervalo [0, 1)")
            return None
        return float(valor)

    def booleano(self, valor: Any, caminho: str, padrao: bool) -> bool:
        """Lê um booleano; ausência assume o padrão informado."""
        if valor is None:
            return padrao
        if not isinstance(valor, bool):
            self.erro(caminho, "deve ser true ou false")
            return padrao
        return valor

    def booleano_obrigatorio(self, valor: Any, caminho: str) -> bool | None:
        """Lê um booleano que precisa estar declarado."""
        if not isinstance(valor, bool):
            self.erro(caminho, "obrigatório: true ou false")
            return None
        return valor

    def lista(self, valor: Any, caminho: str, opcional: bool = True) -> list:
        """Lê uma lista; ausência retorna lista vazia quando opcional."""
        if valor is None:
            if not opcional:
                self.erro(caminho, "obrigatório")
            return []
        if not isinstance(valor, list):
            self.erro(caminho, "deve ser uma lista")
            return []
        return valor

    def escolha(
        self,
        valor: Any,
        caminho: str,
        opcoes: Iterable[str],
        opcional: bool = False,
    ) -> str | None:
        """Lê um valor que precisa pertencer ao vocabulário informado."""
        opcoes = tuple(opcoes)
        if valor is None:
            if not opcional:
                self.erro(caminho, "obrigatório")
            return None
        if valor not in opcoes:
            self.erro(caminho, f"{valor!r} fora de {', '.join(opcoes)}")
            return None
        return valor

    def unicos(self, valores: Iterable[str], caminho: str) -> None:
        """Registra valores repetidos."""
        vistos: set[str] = set()
        for valor in valores:
            if valor in vistos:
                self.erro(caminho, f"valor repetido: {valor}")
            vistos.add(valor)
'''

In [ ]:
_HUB_MODULOS["hub/orquestracao.py"] = r'''
"""Pontos de entrada chamados pelos notebooks finos de orquestração.

Cada função recebe ambiente, entidade (ou base) e competência, registra a
tentativa em ``controle.execucoes`` e chama o motor. As verificações de
governança (lastro, versão, competência, procedência) acontecem dentro
do registro, para que a tentativa bloqueada também fique documentada.
"""

from __future__ import annotations

import json

from pyspark.sql import SparkSession

from hub import aquisicao, arquivos, ciclo_vida, decisoes, hashes, inspecao
from hub import vocabulario as v
from hub.ambiente import Ambiente
from hub.contrato import Contrato
from hub.erros import exigir
from hub.execucao import RegistroExecucao, novo_id_execucao
from hub.historico import Historico
from hub.plano import plano_bronze, plano_silver
from hub.registro import Registro
from hub.spark import (
    bronze,
    controle,
    diagnostico,
    publicacao,
    quarentena,
    sessao,
    silver,
)
from hub.spark import catalogo as catalogo_spark


def _preparar(
    spark: SparkSession, registro: Registro, nome_ambiente: str
) -> Ambiente:
    """Resolve o ambiente e garante sessão, schemas e tabela de controle."""
    ambiente = registro.ambiente(nome_ambiente)
    sessao.preparar(spark, ambiente)
    controle.garantir_tabela(spark, ambiente)
    return ambiente


def _nova_execucao(
    registro: Registro,
    ambiente: Ambiente,
    tipo: str,
    identificador: str,
    contrato: Contrato | None = None,
    competencia: str | None = None,
) -> RegistroExecucao:
    """Cria o registro da tentativa com a identificação do contrato."""
    execucao = RegistroExecucao(
        id_execucao=novo_id_execucao(),
        identificador=identificador,
        tipo=tipo,
        ambiente=ambiente.nome,
        versao_motor=registro.versao_motor,
        competencia=competencia,
    )
    if contrato is not None:
        execucao.fonte = contrato.identidade.fonte
        execucao.base = contrato.identidade.base
        execucao.entidade = contrato.identidade.entidade
        execucao.versao_contrato = contrato.versao
        execucao.estado_contrato = contrato.documentacao.estado
        execucao.hash_leitura = hashes.hash_leitura(contrato)
        execucao.hash_contrato = hashes.hash_contrato(contrato)
    return execucao


def _exigir_governanca(
    contrato: Contrato,
    execucao: RegistroExecucao,
    historico: Historico,
    permitir_anterior: bool,
) -> None:
    """Lastro do estado, coerência de versão e regressão de competência."""
    problemas = ciclo_vida.verificar_lastro(
        contrato.documentacao.estado,
        execucao.hash_leitura,
        execucao.hash_contrato,
        historico.evidencias(),
    )
    problemas += decisoes.verificar_versao(
        contrato.versao, execucao.hash_contrato, historico.hashes_por_versao()
    )
    vigente = historico.ultima_efetiva(execucao.tipo)
    problemas += decisoes.verificar_competencia(
        execucao.competencia,
        vigente["competencia"] if vigente else None,
        permitir_anterior,
    )
    if permitir_anterior:
        execucao.detalhes["permitir_competencia_anterior"] = True
    exigir(problemas)


def _aquisicao_efetiva(
    spark: SparkSession, ambiente: Ambiente, contrato: Contrato, competencia
) -> tuple[dict, dict]:
    """Registro e detalhes da aquisição efetiva da base na competência."""
    historico = controle.ler_historico(
        spark, ambiente, contrato.identidade.identificador_base
    )
    registro = historico.aquisicao_efetiva(competencia)
    if registro is None:
        exigir([f"não há aquisição efetiva da base para {competencia}"])
    return registro, json.loads(registro["detalhes"])


def _partes_da_entidade(
    contrato: Contrato, detalhes_aquisicao: dict
) -> list[decisoes.Parte]:
    """Partes da aquisição selecionadas pelo contrato."""
    partes = decisoes.selecionar_partes(
        detalhes_aquisicao["partes"], contrato.selecao.arquivo
    )
    exigir(
        decisoes.verificar_quantidade_partes(
            partes, contrato.selecao.quantidade_partes
        )
    )
    return partes


def _destino(
    plano, ambiente: Ambiente, versao_vigente: int | None
) -> publicacao.Destino:
    """Destino físico do snapshot a partir do plano."""
    return publicacao.Destino(
        tabela=plano.tabela,
        schema_fisico=plano.schema_fisico,
        comentario_tabela=plano.comentario_tabela,
        comentarios_colunas=dict(plano.comentarios_colunas),
        tabela_quarentena=plano.tabela_quarentena,
        identificador=plano.identificador,
        versao_contrato=plano.versao,
        versao_vigente=versao_vigente,
    )


def _contexto(
    execucao: RegistroExecucao, camada: str, hash_decisao: str
) -> quarentena.ContextoQuarentena:
    """Identificação da publicação para os registros de quarentena."""
    return quarentena.ContextoQuarentena(
        identificador=execucao.identificador,
        camada=camada,
        fonte=execucao.fonte,
        base=execucao.base,
        entidade=execucao.entidade,
        competencia=execucao.competencia,
        publication_fingerprint=execucao.publication_fingerprint,
        versao_contrato=execucao.versao_contrato,
        hash_leitura=execucao.hash_leitura,
        hash_contrato=execucao.hash_contrato,
        hash_decisao=hash_decisao,
        id_execucao=execucao.id_execucao,
    )


def adquirir(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    fonte: str,
    base: str,
    competencia: str,
    modo: str = aquisicao.DOWNLOAD,
    pasta_deposito: str | None = None,
    forcar: bool = False,
) -> dict:
    """Adquire a publicação de uma base para a Raw.

    Sem ``forcar``, uma competência já adquirida não é baixada de novo.
    """
    amb = _preparar(spark, registro, ambiente)
    descritor = registro.base(f"{fonte}_{base}")
    execucao = _nova_execucao(
        registro, amb, v.AQUISICAO, descritor.identificador
    )
    execucao.fonte, execucao.base = fonte, base
    execucao.competencia = competencia
    with controle.execucao(spark, amb, execucao):
        exigir(decisoes.validar_competencia(competencia))
        historico = controle.ler_historico(spark, amb, descritor.identificador)
        existente = historico.aquisicao_efetiva(competencia)
        if existente is not None and not forcar:
            execucao.resultado = v.EXISTENTE
            execucao.id_execucao_origem = existente["id_execucao"]
            execucao.publication_fingerprint = existente[
                "publication_fingerprint"
            ]
        else:
            detalhes = aquisicao.adquirir(
                descritor,
                amb,
                competencia,
                execucao.id_execucao,
                modo,
                pasta_deposito,
            )
            execucao.detalhes.update(detalhes)
            execucao.tabela_destino = detalhes["pasta"]
            execucao.publication_fingerprint = hashes.fingerprint(
                (parte["parte"], parte["sha256"])
                for parte in detalhes["partes"]
            )
            execucao.resultado = v.ADQUIRIDA
    return execucao.resumo()


def inspecionar(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    entidade: str,
    competencia: str,
) -> dict:
    """Inspeção física sob demanda; o veredito vira evidência."""
    amb = _preparar(spark, registro, ambiente)
    contrato = registro.contrato(entidade)
    execucao = _nova_execucao(
        registro,
        amb,
        v.INSPECAO,
        contrato.identificador,
        contrato,
        competencia,
    )
    with controle.execucao(spark, amb, execucao):
        exigir(decisoes.validar_competencia(competencia))
        aquisicao_efetiva, detalhes = _aquisicao_efetiva(
            spark, amb, contrato, competencia
        )
        partes = _partes_da_entidade(contrato, detalhes)
        execucao.id_execucao_origem = aquisicao_efetiva["id_execucao"]
        execucao.publication_fingerprint = hashes.fingerprint(
            (parte.parte, parte.sha256) for parte in partes
        )
        relatorio = inspecao.inspecionar(
            [(parte.parte, amb.local(parte.caminho)) for parte in partes],
            contrato,
        )
        execucao.resultado = relatorio["veredito"]
        execucao.detalhes["inspecao"] = relatorio
    return {**execucao.resumo(), "relatorio": relatorio}


def _verificar_cabecalhos(
    partes: list[decisoes.Parte], contrato: Contrato, ambiente: Ambiente
) -> list[str]:
    """Cabeçalho físico de cada parte contra o layout do contrato."""
    esperado = [(c.cabecalho, *c.aliases) for c in contrato.colunas_origem]
    problemas = []
    for parte in partes:
        encontrado = arquivos.ler_cabecalho(
            ambiente.local(parte.caminho), contrato.leitura
        )
        problemas += [
            f"{parte.parte}: {problema}"
            for problema in decisoes.verificar_cabecalho(encontrado, esperado)
        ]
    return problemas


def _total_declarado(
    registro: Registro,
    contrato: Contrato,
    detalhes_aquisicao: dict,
    ambiente: Ambiente,
) -> int | None:
    """Total publicado pela fonte, lido direto da Raw (sem ordem de carga)."""
    contrato_total = registro.contrato_do_total(contrato)
    if contrato_total is None:
        return None
    partes = _partes_da_entidade(contrato_total, detalhes_aquisicao)
    exigir(decisoes.verificar_quantidade_partes(partes, 1))
    total, problemas = arquivos.ler_total_declarado(
        ambiente.local(partes[0].caminho),
        contrato_total,
        contrato.total_declarado.coluna,
    )
    exigir(problemas)
    return total


def publicar_bronze(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    entidade: str,
    competencia: str,
    permitir_competencia_anterior: bool = False,
) -> dict:
    """Publica a Bronze da entidade para a competência."""
    amb = _preparar(spark, registro, ambiente)
    contrato = registro.contrato(entidade)
    plano = plano_bronze(contrato, amb)
    execucao = _nova_execucao(
        registro,
        amb,
        v.PUBLICACAO_BRONZE,
        contrato.identificador,
        contrato,
        competencia,
    )
    execucao.tabela_destino = plano.tabela
    with controle.execucao(spark, amb, execucao):
        exigir(decisoes.validar_competencia(competencia))
        historico = controle.ler_historico(spark, amb, contrato.identificador)
        _exigir_governanca(
            contrato, execucao, historico, permitir_competencia_anterior
        )
        aquisicao_efetiva, detalhes = _aquisicao_efetiva(
            spark, amb, contrato, competencia
        )
        partes = _partes_da_entidade(contrato, detalhes)
        execucao.id_execucao_origem = aquisicao_efetiva["id_execucao"]
        execucao.publication_fingerprint = hashes.fingerprint(
            (parte.parte, parte.sha256) for parte in partes
        )
        if contrato.leitura.cabecalho:
            exigir(_verificar_cabecalhos(partes, contrato, amb))
        total = _total_declarado(registro, contrato, detalhes, amb)
        vigente = historico.ultima_efetiva(v.PUBLICACAO_BRONZE)
        bronze.publicar(
            spark,
            plano,
            _destino(
                plano, amb, vigente["versao_contrato"] if vigente else None
            ),
            execucao,
            _contexto(execucao, v.BRONZE, execucao.hash_leitura),
            partes,
            detalhes["pasta"],
            total,
        )
    return execucao.resumo()


def publicar_silver(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    entidade: str,
    competencia: str,
    permitir_competencia_anterior: bool = False,
) -> dict:
    """Publica a Silver da entidade a partir da Bronze vigente."""
    amb = _preparar(spark, registro, ambiente)
    contrato = registro.contrato(entidade)
    plano = plano_silver(contrato, amb)
    execucao = _nova_execucao(
        registro,
        amb,
        v.PUBLICACAO_SILVER,
        contrato.identificador,
        contrato,
        competencia,
    )
    execucao.tabela_destino = plano.tabela
    with controle.execucao(spark, amb, execucao):
        exigir(decisoes.validar_competencia(competencia))
        historico = controle.ler_historico(spark, amb, contrato.identificador)
        _exigir_governanca(
            contrato, execucao, historico, permitir_competencia_anterior
        )
        publicacao_bronze = historico.ultima_efetiva(v.PUBLICACAO_BRONZE)
        exigir(
            decisoes.verificar_procedencia(
                publicacao_bronze, competencia, execucao.hash_leitura
            )
        )
        execucao.id_execucao_origem = publicacao_bronze["id_execucao"]
        execucao.publication_fingerprint = publicacao_bronze[
            "publication_fingerprint"
        ]
        vigente = historico.ultima_efetiva(v.PUBLICACAO_SILVER)
        silver.publicar(
            spark,
            plano,
            _destino(
                plano, amb, vigente["versao_contrato"] if vigente else None
            ),
            execucao,
            _contexto(execucao, v.SILVER, execucao.hash_contrato),
            publicacao_bronze,
        )
    return execucao.resumo()


def aprovar(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    entidade: str,
    responsavel: str,
    hash_contrato: str,
) -> dict:
    """Registra aprovação nominal e datada do contrato neste ambiente.

    O hash informado precisa ser o da parte executável carregada: a
    aprovação vale só para aquele conteúdo.
    """
    amb = _preparar(spark, registro, ambiente)
    contrato = registro.contrato(entidade)
    execucao = _nova_execucao(
        registro, amb, v.APROVACAO, contrato.identificador, contrato
    )
    with controle.execucao(spark, amb, execucao):
        problemas = []
        if not responsavel or not responsavel.strip():
            problemas.append("informe o responsável pela aprovação")
        if hash_contrato != execucao.hash_contrato:
            problemas.append(
                f"hash informado {str(hash_contrato)[:12]} difere do "
                f"contrato carregado {execucao.hash_contrato[:12]}"
            )
        historico = controle.ler_historico(spark, amb, contrato.identificador)
        problemas += ciclo_vida.verificar_aprovavel(
            contrato.documentacao.estado,
            contrato.documentacao.pendencias,
            execucao.hash_leitura,
            historico.evidencias(),
        )
        exigir(problemas)
        execucao.responsavel = responsavel.strip()
        execucao.resultado = v.APROVADA
    return execucao.resumo()


def diagnosticar_referencias(
    spark: SparkSession, registro: Registro, ambiente: str, entidade: str
) -> dict:
    """Diagnóstico de integridade referencial; não bloqueia a entidade."""
    amb = _preparar(spark, registro, ambiente)
    contrato = registro.contrato(entidade)
    execucao = _nova_execucao(
        registro, amb, v.DIAGNOSTICO, contrato.identificador, contrato
    )
    with controle.execucao(spark, amb, execucao):
        itens = diagnostico.referencias(spark, contrato, amb)
        execucao.detalhes["referencias"] = itens
        execucao.resultado = decisoes.resultado_diagnostico(itens)
    return {**execucao.resumo(), "referencias": itens}


def sincronizar_catalogo(
    spark: SparkSession, registro: Registro, ambiente: str
) -> dict:
    """Sincroniza o catálogo estrutural e os comentários físicos."""
    amb = _preparar(spark, registro, ambiente)
    execucao = _nova_execucao(
        registro, amb, v.CATALOGO, v.IDENTIFICADOR_GLOBAL
    )
    with controle.execucao(spark, amb, execucao):
        execucao.detalhes["linhas"] = catalogo_spark.sincronizar(
            spark, amb, list(registro.contratos.values())
        )
        execucao.resultado = v.SINCRONIZADO
    return execucao.resumo()


def executar_base(
    spark: SparkSession,
    registro: Registro,
    ambiente: str,
    fonte: str,
    base: str,
    competencia: str,
    forcar_aquisicao: bool = False,
) -> list[dict]:
    """Aquisição, Bronze e Silver de todas as entidades da base.

    Entidades são independentes: a falha de uma não impede as demais.
    Ao final, qualquer falha ou bloqueio é reportado como erro.
    """
    resultados = [
        adquirir(
            spark,
            registro,
            ambiente,
            fonte,
            base,
            competencia,
            forcar=forcar_aquisicao,
        )
    ]
    falhas = []
    for contrato in registro.contratos_da_base(f"{fonte}_{base}"):
        for etapa in (publicar_bronze, publicar_silver):
            try:
                resultados.append(
                    etapa(
                        spark,
                        registro,
                        ambiente,
                        contrato.identificador,
                        competencia,
                    )
                )
            except Exception as erro:
                falhas.append(
                    f"{contrato.identificador} ({etapa.__name__}): {erro}"
                )
                break
    if falhas:
        raise RuntimeError("falhas na execução da base:\n" + "\n".join(falhas))
    return resultados
'''

In [ ]:
_HUB_MODULOS["hub/plano.py"] = r'''
"""Plano de execução: tradução pura do contrato para cada camada.

O plano resolve nome físico (via ambiente), ordem de colunas,
transformações efetivas, tipos físicos, padrões de data, opções do leitor
CSV e comentários. Os executores Spark apenas interpretam o plano, o que
mantém toda a decisão testável sem Spark.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass

from hub import hashes
from hub import vocabulario as v
from hub.ambiente import Ambiente
from hub.contrato import Coluna, Contrato, Regra, TotalDeclarado, Transformacao
from hub.execucao import TABELA_QUARENTENA

COLUNA_MALFORMADA = "_hub_linha_malformada"

_TIPOS_FISICOS = {
    v.TEXTO: "string",
    v.INTEIRO: "int",
    v.INTEIRO_LONGO: "bigint",
    v.DATA: "date",
}

LINHAGEM_FISICA = tuple((nome, "string") for nome in v.COLUNAS_LINHAGEM)

COMENTARIOS_LINHAGEM = {
    v.COLUNA_COMPETENCIA: "Competência da publicação de origem (AAAA-MM).",
    v.COLUNA_ARQUIVO: "Parte física de origem na Raw (<pasta>/<arquivo>).",
    v.COLUNA_EXECUCAO: "Execução que publicou a linha (controle.execucoes).",
}


def tipo_fisico(coluna: Coluna) -> str:
    """Tipo físico da coluna na Silver."""
    if coluna.tipo == v.DECIMAL:
        return f"decimal({coluna.precisao},{coluna.escala})"
    return _TIPOS_FISICOS[coluna.tipo]


def padrao_data(formato: str) -> str:
    """Converte a notação do layout (AAAA-MM-DD) no padrão yyyy-MM-dd."""
    padrao = formato
    for token, equivalente in v.TOKENS_DATA.items():
        padrao = padrao.replace(token, equivalente)
    return padrao


def opcoes_leitor_csv(contrato: Contrato) -> dict[str, str]:
    """Traduz o dialeto do contrato em opções do leitor CSV do Spark.

    Linha com número de campos diferente do layout é marcada como
    malformada (modo PERMISSIVE com coluna de registro corrompido).
    """
    leitura = contrato.leitura
    return {
        "sep": leitura.separador,
        "quote": leitura.aspas,
        "escape": leitura.escape,
        "encoding": v.CODIFICACOES[leitura.codificacao].spark,
        "header": "true" if leitura.cabecalho else "false",
        "mode": "PERMISSIVE",
        "columnNameOfCorruptRecord": COLUNA_MALFORMADA,
        "multiLine": "false",
        "ignoreLeadingWhiteSpace": "false",
        "ignoreTrailingWhiteSpace": "false",
    }


def _comentario_tabela(contrato: Contrato, camada: str) -> str:
    """Comentário físico da tabela derivado do contrato."""
    papel = {
        v.BRONZE: "Bronze: valores textuais conforme o arquivo de origem.",
        v.SILVER: "Silver: tipos, domínios e regras do contrato aplicados.",
    }[camada]
    partes = [contrato.descricao or "", papel]
    partes.append(
        f"Contrato {contrato.identificador} versão {contrato.versao}."
    )
    return " ".join(parte for parte in partes if parte)


def _comentarios_colunas(contrato: Contrato, camada: str) -> dict[str, str]:
    """Comentários físicos de coluna derivados do contrato."""
    comentarios = {}
    for coluna in contrato.colunas:
        comentarios[coluna.nome] = coluna.descricao or ""
        dominio = contrato.dominio_literal(coluna)
        if camada == v.SILVER and dominio and dominio.coluna_descricao:
            comentarios[dominio.coluna_descricao] = (
                f"Descrição de {coluna.nome} conforme o domínio "
                f"{dominio.nome}."
            )
    comentarios.update(COMENTARIOS_LINHAGEM)
    return comentarios


@dataclass(frozen=True)
class PlanoBronze:
    """O que a Bronze executa para uma entidade."""

    identificador: str
    tabela: str
    tabela_quarentena: str
    padrao_arquivo: str
    quantidade_partes: int | None
    colunas_origem: tuple[str, ...]
    colunas_saida: tuple[str, ...]
    cabecalho_esperado: tuple[tuple[str, ...], ...] | None
    opcoes_csv: Mapping[str, str]
    cabecalho: bool
    tolerancia: float
    acao_malformada: str
    total_declarado: TotalDeclarado | None
    schema_fisico: tuple[tuple[str, str], ...]
    comentario_tabela: str
    comentarios_colunas: Mapping[str, str]
    versao: int
    hash_leitura: str
    hash_contrato: str


@dataclass(frozen=True)
class ColunaSilver:
    """Tratamento de uma coluna na Silver."""

    nome: str
    tipo: str
    tipo_fisico: str
    padrao_data: str | None
    separador_decimal: str | None
    transformacoes: tuple[Transformacao, ...]
    verificar_nulidade: bool
    na_chave: bool
    valores_dominio: Mapping[str, str] | None
    coluna_descricao: str | None


@dataclass(frozen=True)
class PlanoSilver:
    """O que a Silver executa para uma entidade."""

    identificador: str
    tabela: str
    tabela_bronze: str
    tabela_quarentena: str
    colunas: tuple[ColunaSilver, ...]
    chave: tuple[str, ...]
    regras: tuple[Regra, ...]
    acoes: Mapping[str, str]
    colunas_negocio: tuple[str, ...]
    schema_fisico: tuple[tuple[str, str], ...]
    comentario_tabela: str
    comentarios_colunas: Mapping[str, str]
    versao: int
    hash_leitura: str
    hash_contrato: str


def plano_bronze(contrato: Contrato, ambiente: Ambiente) -> PlanoBronze:
    """Compila o plano da Bronze."""
    origem = contrato.colunas_origem
    cabecalho = None
    if contrato.leitura.cabecalho:
        cabecalho = tuple(
            (coluna.cabecalho, *coluna.aliases) for coluna in origem
        )
    return PlanoBronze(
        identificador=contrato.identificador,
        tabela=ambiente.tabela(v.BRONZE, contrato.identificador),
        tabela_quarentena=ambiente.tabela_controle(TABELA_QUARENTENA),
        padrao_arquivo=contrato.selecao.arquivo,
        quantidade_partes=contrato.selecao.quantidade_partes,
        colunas_origem=tuple(coluna.nome for coluna in origem),
        colunas_saida=tuple(coluna.nome for coluna in contrato.colunas),
        cabecalho_esperado=cabecalho,
        opcoes_csv=opcoes_leitor_csv(contrato),
        cabecalho=contrato.leitura.cabecalho,
        tolerancia=contrato.leitura.tolerancia_malformadas,
        acao_malformada=contrato.acoes_padrao[v.LINHA_MALFORMADA],
        total_declarado=contrato.total_declarado,
        schema_fisico=tuple(
            (coluna.nome, "string") for coluna in contrato.colunas
        )
        + LINHAGEM_FISICA,
        comentario_tabela=_comentario_tabela(contrato, v.BRONZE),
        comentarios_colunas=_comentarios_colunas(contrato, v.BRONZE),
        versao=contrato.versao,
        hash_leitura=hashes.hash_leitura(contrato),
        hash_contrato=hashes.hash_contrato(contrato),
    )


def _coluna_silver(contrato: Contrato, coluna: Coluna) -> ColunaSilver:
    """Compila o tratamento Silver de uma coluna."""
    dominio = contrato.dominio_literal(coluna)
    return ColunaSilver(
        nome=coluna.nome,
        tipo=coluna.tipo,
        tipo_fisico=tipo_fisico(coluna),
        padrao_data=padrao_data(coluna.formato) if coluna.formato else None,
        separador_decimal=coluna.separador_decimal,
        transformacoes=contrato.transformacoes_padrao + coluna.transformacoes,
        verificar_nulidade=(
            not coluna.nulavel and coluna.nome not in contrato.chave
        ),
        na_chave=coluna.nome in contrato.chave,
        valores_dominio=dict(dominio.valores) if dominio else None,
        coluna_descricao=dominio.coluna_descricao if dominio else None,
    )


def plano_silver(contrato: Contrato, ambiente: Ambiente) -> PlanoSilver:
    """Compila o plano da Silver."""
    colunas = tuple(_coluna_silver(contrato, c) for c in contrato.colunas)
    schema: list[tuple[str, str]] = []
    for coluna in colunas:
        schema.append((coluna.nome, coluna.tipo_fisico))
        if coluna.coluna_descricao:
            schema.append((coluna.coluna_descricao, "string"))
    return PlanoSilver(
        identificador=contrato.identificador,
        tabela=ambiente.tabela(v.SILVER, contrato.identificador),
        tabela_bronze=ambiente.tabela(v.BRONZE, contrato.identificador),
        tabela_quarentena=ambiente.tabela_controle(TABELA_QUARENTENA),
        colunas=colunas,
        chave=contrato.chave,
        regras=contrato.regras,
        acoes=dict(contrato.acoes_padrao),
        colunas_negocio=contrato.colunas_saida,
        schema_fisico=tuple(schema) + LINHAGEM_FISICA,
        comentario_tabela=_comentario_tabela(contrato, v.SILVER),
        comentarios_colunas=_comentarios_colunas(contrato, v.SILVER),
        versao=contrato.versao,
        hash_leitura=hashes.hash_leitura(contrato),
        hash_contrato=hashes.hash_contrato(contrato),
    )
'''

In [ ]:
_HUB_MODULOS["hub/registro.py"] = r'''
"""Registro dos artefatos declarativos carregados: ambientes, bases e
contratos, com as validações que cruzam arquivos diferentes.

O mesmo dicionário é produzido pelo gerador a partir do YAML versionado
e embutido no notebook-biblioteca; por isso o Fabric não precisa de YAML.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
from typing import Any

from hub import vocabulario as v
from hub.ambiente import Ambiente, Configuracao, carregar_configuracao
from hub.bases import DescritorBase, carregar_base
from hub.contrato import REFERENCIA, Contrato, carregar_contrato
from hub.erros import ContratoInvalido

_TIPOS_TOTAL = (v.INTEIRO, v.INTEIRO_LONGO)


@dataclass(frozen=True)
class Registro:
    """Contratos, bases e ambientes validados."""

    configuracao: Configuracao
    bases: Mapping[str, DescritorBase]
    contratos: Mapping[str, Contrato]
    versao_motor: str

    @classmethod
    def de_dicionario(
        cls, dados: Mapping[str, Any], versao_motor: str = "local"
    ) -> Registro:
        """Carrega e valida todo o conteúdo declarativo.

        Args:
            dados: mapa com as chaves configuracao, bases e contratos.
            versao_motor: identificação do código do motor em execução.

        Raises:
            ContratoInvalido: com os problemas de todos os artefatos.
        """
        problemas: list[str] = []
        configuracao = _carregar(
            problemas, carregar_configuracao, dados.get("configuracao")
        )
        bases = {}
        for chave, conteudo in sorted(dados.get("bases", {}).items()):
            base = _carregar(problemas, carregar_base, conteudo, chave)
            if base is not None:
                _conferir_chave(problemas, chave, base.identificador)
                bases[base.identificador] = base
        contratos = {}
        for chave, conteudo in sorted(dados.get("contratos", {}).items()):
            contrato = _carregar(problemas, carregar_contrato, conteudo, chave)
            if contrato is not None:
                _conferir_chave(problemas, chave, contrato.identificador)
                contratos[contrato.identificador] = contrato
        if configuracao is not None:
            _validar_cruzamentos(problemas, bases, contratos)
        if problemas:
            raise ContratoInvalido("registro", problemas)
        return cls(configuracao, bases, contratos, versao_motor)

    def ambiente(self, nome: str) -> Ambiente:
        """Resolução física do ambiente informado."""
        return self.configuracao.ambiente(nome)

    def contrato(self, identificador: str) -> Contrato:
        """Contrato pelo identificador lógico (fonte_base_entidade)."""
        if identificador not in self.contratos:
            raise KeyError(f"contrato inexistente: {identificador}")
        return self.contratos[identificador]

    def base(self, identificador: str) -> DescritorBase:
        """Descritor da base pelo identificador (fonte_base)."""
        if identificador not in self.bases:
            raise KeyError(f"base inexistente: {identificador}")
        return self.bases[identificador]

    def contratos_da_base(self, identificador_base: str) -> list[Contrato]:
        """Contratos de uma base, em ordem de identificador."""
        return [
            contrato
            for nome, contrato in sorted(self.contratos.items())
            if contrato.identidade.identificador_base == identificador_base
        ]

    def contrato_do_total(self, contrato: Contrato) -> Contrato | None:
        """Contrato da entidade que publica o total declarado."""
        total = contrato.total_declarado
        if total is None:
            return None
        base = contrato.identidade.identificador_base
        return self.contrato(f"{base}_{total.entidade}")


def _carregar(problemas: list[str], carregar, conteudo, *argumentos):
    """Executa um carregador acumulando os problemas encontrados."""
    try:
        return carregar(conteudo, *argumentos)
    except ContratoInvalido as erro:
        problemas.extend(f"{erro.origem}: {p}" for p in erro.problemas)
        return None


def _conferir_chave(problemas: list[str], chave: str, esperado: str) -> None:
    """A chave do dicionário precisa ser o identificador do artefato."""
    if chave != esperado:
        problemas.append(f"{chave}: identificador declarado é {esperado}")


def _validar_cruzamentos(
    problemas: list[str],
    bases: Mapping[str, DescritorBase],
    contratos: Mapping[str, Contrato],
) -> None:
    """Validações que dependem de mais de um artefato."""
    for identificador, contrato in contratos.items():
        base = contrato.identidade.identificador_base
        if base not in bases:
            problemas.append(f"{identificador}: base {base} não declarada")
        for dominio in contrato.dominios.values():
            if dominio.tipo == REFERENCIA:
                _validar_referencia(problemas, contrato, dominio, contratos)
        if contrato.total_declarado is not None:
            _validar_total(problemas, contrato, contratos)


def _validar_referencia(problemas, contrato, dominio, contratos) -> None:
    """A referência precisa apontar para coluna texto de contrato existente."""
    alvo = dominio.tabela.identificador
    origem = f"{contrato.identificador}: domínio {dominio.nome}"
    if alvo not in contratos:
        problemas.append(f"{origem} referencia contrato inexistente {alvo}")
        return
    try:
        coluna = contratos[alvo].coluna(dominio.coluna)
    except KeyError:
        problemas.append(f"{origem}: coluna {dominio.coluna} não existe")
        return
    if coluna.tipo != v.TEXTO:
        problemas.append(f"{origem}: coluna referenciada deve ser texto")


def _validar_total(problemas, contrato, contratos) -> None:
    """O total declarado aponta para coluna inteira da mesma base."""
    total = contrato.total_declarado
    alvo = f"{contrato.identidade.identificador_base}_{total.entidade}"
    origem = f"{contrato.identificador}: reconciliacao"
    if alvo not in contratos:
        problemas.append(f"{origem} aponta para contrato inexistente {alvo}")
        return
    try:
        coluna = contratos[alvo].coluna(total.coluna)
    except KeyError:
        problemas.append(f"{origem}: coluna {total.coluna} não existe")
        return
    if coluna.tipo not in _TIPOS_TOTAL:
        problemas.append(f"{origem}: coluna do total deve ser inteira")
    if coluna.ausente_na_origem:
        problemas.append(f"{origem}: coluna do total ausente na origem")
'''

In [ ]:
_HUB_MODULOS["hub/retentativa.py"] = r'''
"""Retentativa de escritas Delta que perdem disputa de concorrência."""

from __future__ import annotations

import time
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T")


def conflito_concorrente(erro: BaseException) -> bool:
    """Indica se o erro é conflito de concorrência do Delta Lake."""
    texto = f"{type(erro).__name__} {erro}"[:2000]
    return "Concurrent" in texto


def com_retentativa(
    operacao: Callable[[], T],
    tentativas: int = 4,
    espera_inicial: float = 2.0,
    dormir: Callable[[float], None] = time.sleep,
) -> T:
    """Executa a operação, repetindo apenas em conflito de concorrência.

    A espera dobra a cada tentativa. Qualquer outro erro é propagado de
    imediato.
    """
    for tentativa in range(1, tentativas + 1):
        try:
            return operacao()
        except Exception as erro:
            if tentativa == tentativas or not conflito_concorrente(erro):
                raise
            dormir(espera_inicial * 2 ** (tentativa - 1))
    raise AssertionError("inalcançável")
'''

In [ ]:
_HUB_MODULOS["hub/spark/__init__.py"] = r'''
"""Executores Spark e Delta Lake.

Módulos finos que interpretam os planos puros. Dependem de pyspark e
delta-spark, presentes no runtime do Fabric (Spark 3.4 ou superior); não
são importados pela suíte local de testes.
"""
'''

In [ ]:
_HUB_MODULOS["hub/spark/bronze.py"] = r'''
"""Publicação da Bronze.

Lê conforme o contrato, mantém tudo como texto e acrescenta a linhagem
mínima. Só separa o que o parser não representa no layout (linha com
número de campos diferente), sem bloquear as demais, até a tolerância
declarada. Não aplica limpeza semântica, domínio nem regra.
"""

from __future__ import annotations

import re
from collections.abc import Sequence
from urllib.parse import unquote

from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

from hub import decisoes
from hub import vocabulario as v
from hub.decisoes import Parte
from hub.erros import exigir
from hub.execucao import RegistroExecucao
from hub.plano import COLUNA_MALFORMADA, PlanoBronze
from hub.spark import publicacao, quarentena

COLUNA_PARTE = "_hub_parte"


def coluna_parte(pasta_aquisicao: str) -> Column:
    """Identificador da parte: caminho físico relativo à pasta da aquisição."""
    padrao = "/" + re.escape(pasta_aquisicao.strip("/")) + "/(.+)$"
    return F.regexp_extract(F.col("_metadata.file_path"), padrao, 1)


def ler(
    spark: SparkSession,
    plano: PlanoBronze,
    partes: Sequence[Parte],
    pasta: str,
) -> DataFrame:
    """Lê as partes como texto, marcando linhas malformadas."""
    campos = [
        StructField(nome, StringType(), True) for nome in plano.colunas_origem
    ]
    campos.append(StructField(COLUNA_MALFORMADA, StringType(), True))
    return (
        spark.read.format("csv")
        .schema(StructType(campos))
        .options(**plano.opcoes_csv)
        .load([parte.caminho for parte in partes])
        .withColumn(COLUNA_PARTE, coluna_parte(pasta))
    )


def contar_linhas_fisicas(
    spark: SparkSession, partes: Sequence[Parte], pasta: str
) -> dict[str, int]:
    """Linhas não vazias por parte, contadas sem o parser CSV."""
    linhas = (
        spark.read.text([parte.caminho for parte in partes])
        .select(coluna_parte(pasta).alias("parte"), "value")
        .where(F.length("value") > 0)
        .groupBy("parte")
        .count()
        .collect()
    )
    return {unquote(linha["parte"]): linha["count"] for linha in linhas}


def _contagens(df: DataFrame) -> tuple[dict[str, int], dict[str, int]]:
    """Registros lidos e malformados por parte."""
    linhas = (
        df.groupBy(COLUNA_PARTE)
        .agg(
            F.count(F.lit(1)).alias("lidas"),
            F.sum(F.col(COLUNA_MALFORMADA).isNotNull().cast("int")).alias(
                "malformadas"
            ),
        )
        .collect()
    )
    lidas = {unquote(linha[COLUNA_PARTE]): linha["lidas"] for linha in linhas}
    malformadas = {
        unquote(linha[COLUNA_PARTE]): linha["malformadas"] or 0
        for linha in linhas
    }
    return lidas, malformadas


def _saida(
    df: DataFrame, plano: PlanoBronze, competencia: str, id_execucao: str
) -> DataFrame:
    """Linhas válidas no layout do contrato, com a linhagem mínima."""
    colunas = [
        (
            F.col(nome)
            if nome in plano.colunas_origem
            else F.lit(None).cast("string").alias(nome)
        )
        for nome in plano.colunas_saida
    ]
    colunas += [
        F.lit(competencia).alias(v.COLUNA_COMPETENCIA),
        F.col(COLUNA_PARTE).alias(v.COLUNA_ARQUIVO),
        F.lit(id_execucao).alias(v.COLUNA_EXECUCAO),
    ]
    return df.where(F.col(COLUNA_MALFORMADA).isNull()).select(*colunas)


def _retirados(
    df: DataFrame,
    plano: PlanoBronze,
    contexto: quarentena.ContextoQuarentena,
) -> DataFrame:
    """Linhas malformadas no formato da quarentena, com a linha original."""
    original = F.col(COLUNA_MALFORMADA)
    motivo = F.struct(
        F.lit(v.LINHA_MALFORMADA).alias("classe"),
        F.lit(None).cast("string").alias("regra"),
        F.lit(None).cast("string").alias("coluna"),
        F.substring(original, 1, 200).alias("valor"),
        F.lit(plano.acao_malformada).alias("acao"),
        F.lit(v.DIMENSAO_POR_CLASSE[v.LINHA_MALFORMADA]).alias("dimensao"),
    )
    return quarentena.montar(
        df.where(original.isNotNull()),
        contexto=contexto,
        registro=original,
        hash_registro=F.sha2(original, 256),
        motivos=F.to_json(F.array(motivo)),
        chave=F.lit(None).cast("string"),
        grupo=F.lit(v.LINHA_MALFORMADA),
        arquivo=F.col(COLUNA_PARTE),
    )


def _verificar_leitura(
    plano: PlanoBronze,
    registro: RegistroExecucao,
    partes: Sequence[str],
    fisicas: dict[str, int],
    lidas: dict[str, int],
    malformadas: dict[str, int],
    total_declarado: int | None,
) -> list[str]:
    """Reconciliação antes da escrita e limites estruturais."""
    validas = {p: n - malformadas.get(p, 0) for p, n in lidas.items()}
    total_lidas = sum(lidas.values())
    total_malformadas = sum(malformadas.values())
    problemas = decisoes.verificar_partes_lidas(
        partes, fisicas, lidas, validas, plano.cabecalho
    )
    problemas += decisoes.verificar_tolerancia(
        total_malformadas, total_lidas, plano.tolerancia
    )
    if total_declarado is not None:
        registro.detalhes["total_declarado"] = total_declarado
        problemas += decisoes.verificar_total_declarado(
            total_declarado, total_lidas
        )
    if total_malformadas and plano.acao_malformada == v.BLOQUEIA_PUBLICACAO:
        problemas.append(
            f"{total_malformadas} linhas malformadas com ação "
            f"{v.BLOQUEIA_PUBLICACAO}"
        )
    return problemas


def publicar(
    spark: SparkSession,
    plano: PlanoBronze,
    destino: publicacao.Destino,
    registro: RegistroExecucao,
    contexto: quarentena.ContextoQuarentena,
    partes: Sequence[Parte],
    pasta: str,
    total_declarado: int | None,
) -> None:
    """Executa a Bronze de ponta a ponta, preenchendo o registro."""
    df = ler(spark, plano, partes, pasta).persist(StorageLevel.MEMORY_AND_DISK)
    try:
        nomes = [parte.parte for parte in partes]
        lidas, malformadas = _contagens(df)
        fisicas = contar_linhas_fisicas(spark, partes, pasta)
        registro.linhas_origem = decisoes.linhas_origem(
            fisicas, nomes, plano.cabecalho
        )
        registro.detalhes["partes"] = {
            parte: {
                "fisicas": fisicas.get(parte, 0),
                "lidas": lidas.get(parte, 0),
                "malformadas": malformadas.get(parte, 0),
            }
            for parte in sorted(set(nomes) | set(lidas) | set(fisicas))
        }
        exigir(
            _verificar_leitura(
                plano,
                registro,
                nomes,
                fisicas,
                lidas,
                malformadas,
                total_declarado,
            )
        )
        total_malformadas = sum(malformadas.values())
        if total_malformadas and plano.acao_malformada in v.ACOES_QUE_ALERTAM:
            registro.alertar(
                f"{total_malformadas} linhas malformadas na quarentena"
            )
        publicacao.publicar(
            spark,
            destino,
            registro,
            saida=_saida(
                df, plano, contexto.competencia, registro.id_execucao
            ),
            publicadas_esperadas=sum(lidas.values()) - total_malformadas,
            retirados=_retirados(df, plano, contexto),
            quarentena_esperada=total_malformadas,
        )
    finally:
        df.unpersist()
'''

In [ ]:
_HUB_MODULOS["hub/spark/catalogo.py"] = r'''
"""Sincronização do catálogo e dos comentários físicos.

A parte estrutural é atualizada a cada sincronização; a curatorial só é
escrita na inserção (A_CONFIRMAR) e depois preservada.
"""

from __future__ import annotations

from collections.abc import Sequence

from pyspark.sql import SparkSession

from hub import catalogo
from hub.ambiente import Ambiente
from hub.contrato import Contrato
from hub.plano import plano_bronze, plano_silver
from hub.spark import delta


def _tabelas_existentes(
    spark: SparkSession, contratos: Sequence[Contrato], ambiente: Ambiente
) -> set[str]:
    """Tabelas Bronze e Silver dos contratos que existem no ambiente."""
    existentes = set()
    for contrato in contratos:
        for plano in (
            plano_bronze(contrato, ambiente),
            plano_silver(contrato, ambiente),
        ):
            if delta.tabela_existe(spark, plano.tabela):
                existentes.add(plano.tabela)
    return existentes


def _aplicar_comentarios(
    spark: SparkSession,
    contratos: Sequence[Contrato],
    ambiente: Ambiente,
    existentes: set[str],
) -> None:
    """Comentários físicos de tabela e coluna derivados do contrato."""
    for contrato in contratos:
        for plano in (
            plano_bronze(contrato, ambiente),
            plano_silver(contrato, ambiente),
        ):
            if plano.tabela in existentes:
                delta.aplicar_comentarios(
                    spark,
                    plano.tabela,
                    plano.comentario_tabela,
                    dict(plano.comentarios_colunas),
                )


def sincronizar(
    spark: SparkSession, ambiente: Ambiente, contratos: Sequence[Contrato]
) -> dict[str, int]:
    """Atualiza as tabelas de catálogo e os comentários físicos.

    Returns:
        Quantidade de linhas estruturais por tabela de catálogo.
    """
    existentes = _tabelas_existentes(spark, contratos, ambiente)
    linhas = catalogo.linhas_catalogo(contratos, ambiente, existentes)
    for nome, registros in linhas.items():
        esquema = catalogo.ESQUEMAS[nome]
        tabela = ambiente.tabela_controle(nome)
        delta.criar_tabela(spark, tabela, esquema)
        origem = spark.createDataFrame(
            [tuple(r[coluna] for coluna, _ in esquema) for r in registros],
            delta.estrutura(esquema),
        )
        condicao = " AND ".join(
            f"t.{coluna} <=> s.{coluna}" for coluna in catalogo.CHAVES[nome]
        )
        delta.mesclar(
            spark,
            tabela,
            origem,
            condicao,
            atualizar=catalogo.colunas_estruturais(nome),
        )
    _aplicar_comentarios(spark, contratos, ambiente, existentes)
    return {nome: len(registros) for nome, registros in linhas.items()}
'''

In [ ]:
_HUB_MODULOS["hub/spark/controle.py"] = r'''
"""Registro operacional: tabela única ``controle.execucoes``.

Cada tentativa é gravada ao iniciar (EM_EXECUCAO) e ao terminar, com
qualquer status. A escrita é um MERGE por ``id_execucao``, idempotente.
"""

from __future__ import annotations

from collections.abc import Iterator
from contextlib import contextmanager

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from hub import vocabulario as v
from hub.ambiente import Ambiente
from hub.erros import BloqueioPublicacao
from hub.execucao import (
    COLUNA_PARTICAO,
    COLUNAS_EXECUCOES,
    TABELA_EXECUCOES,
    RegistroExecucao,
)
from hub.historico import Historico
from hub.spark import delta


def garantir_tabela(spark: SparkSession, ambiente: Ambiente) -> None:
    """Cria controle.execucoes, se ainda não existir."""
    delta.criar_tabela(
        spark,
        ambiente.tabela_controle(TABELA_EXECUCOES),
        COLUNAS_EXECUCOES,
        COLUNA_PARTICAO,
    )


def gravar(
    spark: SparkSession, ambiente: Ambiente, registro: RegistroExecucao
) -> None:
    """Insere ou atualiza a linha da execução."""
    origem = spark.createDataFrame(
        [registro.para_linha()], delta.estrutura(COLUNAS_EXECUCOES)
    )
    identificador = delta.literal_sql(registro.identificador)
    condicao = (
        f"t.identificador = '{identificador}' "
        "AND t.id_execucao = s.id_execucao"
    )
    delta.mesclar(
        spark,
        ambiente.tabela_controle(TABELA_EXECUCOES),
        origem,
        condicao,
        atualizar=[nome for nome, _ in COLUNAS_EXECUCOES],
    )


def ler_historico(
    spark: SparkSession, ambiente: Ambiente, identificador: str
) -> Historico:
    """Execuções registradas para o identificador neste ambiente."""
    linhas = (
        spark.table(ambiente.tabela_controle(TABELA_EXECUCOES))
        .where(F.col("identificador") == identificador)
        .collect()
    )
    return Historico(linha.asDict() for linha in linhas)


@contextmanager
def execucao(
    spark: SparkSession, ambiente: Ambiente, registro: RegistroExecucao
) -> Iterator[RegistroExecucao]:
    """Registra a tentativa no início e no fim, inclusive em falha.

    BloqueioPublicacao termina como BLOQUEADA; qualquer outra exceção,
    como FALHA. Em ambos os casos a exceção é propagada depois do registro.
    """
    gravar(spark, ambiente, registro)
    try:
        yield registro
    except BloqueioPublicacao as erro:
        registro.finalizar(v.BLOQUEADA, motivo=str(erro))
        raise
    except Exception as erro:
        registro.finalizar(v.FALHA, motivo=f"{type(erro).__name__}: {erro}")
        raise
    else:
        registro.finalizar(registro.status_sucesso())
    finally:
        gravar(spark, ambiente, registro)
'''

In [ ]:
_HUB_MODULOS["hub/spark/delta.py"] = r'''
"""Operações Delta Lake usadas pelo motor."""

from __future__ import annotations

from collections.abc import Iterable
from dataclasses import dataclass

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

from hub.retentativa import com_retentativa

_TIPOS = {
    "STRING": StringType(),
    "INT": IntegerType(),
    "BIGINT": LongType(),
    "BOOLEAN": BooleanType(),
    "TIMESTAMP": TimestampType(),
}


@dataclass(frozen=True)
class Escrita:
    """Resultado da escrita de um snapshot."""

    versao_anterior: int | None
    versao: int
    linhas: int


def estrutura(colunas: Iterable[tuple[str, str]]) -> StructType:
    """StructType a partir de pares (nome, tipo SQL)."""
    return StructType(
        [StructField(nome, _TIPOS[tipo], True) for nome, tipo in colunas]
    )


def literal_sql(texto: str) -> str:
    """Escapa texto para uso entre aspas simples no Spark SQL."""
    return texto.replace("\\", "\\\\").replace("'", "\\'")


def tabela_existe(spark: SparkSession, tabela: str) -> bool:
    """Indica se a tabela existe no catálogo."""
    return spark.catalog.tableExists(tabela)


def versao_atual(spark: SparkSession, tabela: str) -> int:
    """Versão Delta mais recente da tabela."""
    ultima = DeltaTable.forName(spark, tabela).history(1)
    return ultima.select("version").first()[0]


def schema_fisico(spark: SparkSession, tabela: str) -> list[tuple[str, str]]:
    """Pares (coluna, tipo) do schema atual da tabela."""
    return [
        (campo.name, campo.dataType.simpleString())
        for campo in spark.table(tabela).schema.fields
    ]


def criar_tabela(
    spark: SparkSession,
    tabela: str,
    colunas: Iterable[tuple[str, str]],
    particao: str | None = None,
) -> None:
    """Cria a tabela Delta com schema explícito, se ainda não existir."""
    definicao = ", ".join(f"`{nome}` {tipo}" for nome, tipo in colunas)
    clausula = f" PARTITIONED BY (`{particao}`)" if particao else ""
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {tabela} ({definicao}) "
        f"USING DELTA{clausula}"
    )


def publicar_snapshot(
    spark: SparkSession, df: DataFrame, tabela: str, sobrescrever_schema: bool
) -> Escrita:
    """Substitui o conteúdo da tabela pelo DataFrame (snapshot vigente).

    As linhas escritas vêm da métrica do commit de escrita. Se outro commit
    vier logo depois (ex.: compactação automática) ou a métrica faltar, a
    contagem é feita na tabela após a escrita.
    """
    existia = tabela_existe(spark, tabela)
    anterior = versao_atual(spark, tabela) if existia else None
    escritor = df.write.format("delta").mode("overwrite")
    if sobrescrever_schema:
        escritor = escritor.option("overwriteSchema", "true")
    escritor.saveAsTable(tabela)
    commits = (
        DeltaTable.forName(spark, tabela)
        .history(5)
        .select("version", "operationMetrics")
        .collect()
    )
    novos = [c for c in commits if anterior is None or c["version"] > anterior]
    escritas = [
        c for c in novos if "numOutputRows" in (c["operationMetrics"] or {})
    ]
    if escritas:
        commit = max(escritas, key=lambda c: c["version"])
        return Escrita(
            anterior,
            commit["version"],
            int(commit["operationMetrics"]["numOutputRows"]),
        )
    versao = max(c["version"] for c in novos)
    return Escrita(anterior, versao, spark.table(tabela).count())


def restaurar(
    spark: SparkSession, tabela: str, versao_anterior: int | None
) -> None:
    """Desfaz a escrita: volta à versão anterior ou remove a tabela nova."""
    if versao_anterior is None:
        spark.sql(f"DROP TABLE IF EXISTS {tabela}")
    else:
        DeltaTable.forName(spark, tabela).restoreToVersion(versao_anterior)


def mesclar(
    spark: SparkSession,
    tabela: str,
    origem: DataFrame,
    condicao: str,
    atualizar: Iterable[str] = (),
) -> None:
    """MERGE com retentativa em conflito de concorrência.

    Linhas novas são inseridas; linhas existentes têm apenas as colunas
    em ``atualizar`` sobrescritas (nenhuma, se vazio).
    """
    colunas = tuple(atualizar)

    def operacao() -> None:
        fusao = (
            DeltaTable.forName(spark, tabela)
            .alias("t")
            .merge(origem.alias("s"), condicao)
        )
        if colunas:
            fusao = fusao.whenMatchedUpdate(
                set={coluna: f"s.{coluna}" for coluna in colunas}
            )
        fusao.whenNotMatchedInsertAll().execute()

    com_retentativa(operacao)


def aplicar_comentarios(
    spark: SparkSession,
    tabela: str,
    comentario_tabela: str,
    comentarios_colunas: dict[str, str],
) -> None:
    """Aplica comentários de tabela e coluna que divergem do contrato."""
    detalhe = DeltaTable.forName(spark, tabela).detail()
    atual = detalhe.select("description").first()[0] or ""
    if atual != comentario_tabela:
        spark.sql(
            f"COMMENT ON TABLE {tabela} IS "
            f"'{literal_sql(comentario_tabela)}'"
        )
    campos = {
        campo.name: campo.metadata.get("comment", "")
        for campo in spark.table(tabela).schema.fields
    }
    for coluna, comentario in comentarios_colunas.items():
        if coluna in campos and campos[coluna] != comentario:
            spark.sql(
                f"ALTER TABLE {tabela} ALTER COLUMN `{coluna}` "
                f"COMMENT '{literal_sql(comentario)}'"
            )
'''

In [ ]:
_HUB_MODULOS["hub/spark/diagnostico.py"] = r'''
"""Diagnóstico dirigido de integridade referencial.

Fora do fluxo obrigatório: roda sob demanda e não bloqueia a entidade de
negócio. Para cada domínio por referência, conta valores sem
correspondência na tabela oficial. Se a referência não existe, o
relacionamento é marcado como REFERENCIA_AUSENTE.
"""

from __future__ import annotations

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from hub import vocabulario as v
from hub.ambiente import Ambiente
from hub.contrato import REFERENCIA, Contrato
from hub.erros import exigir
from hub.spark import delta

LIMITE_EXEMPLOS = 10


def referencias(
    spark: SparkSession, contrato: Contrato, ambiente: Ambiente
) -> list[dict]:
    """Confere cada coluna com domínio por referência.

    Raises:
        BloqueioPublicacao: se a própria tabela Silver não existir.
    """
    tabela = ambiente.tabela(v.SILVER, contrato.identificador)
    if not delta.tabela_existe(spark, tabela):
        exigir([f"tabela Silver inexistente: {tabela}"])
    itens = []
    for coluna in contrato.colunas:
        dominio = contrato.dominios.get(coluna.dominio or "")
        if dominio is None or dominio.tipo != REFERENCIA:
            continue
        alvo = ambiente.tabela(v.SILVER, dominio.tabela.identificador)
        item = {
            "coluna": coluna.nome,
            "dominio": dominio.nome,
            "referencia": alvo,
            "coluna_referenciada": dominio.coluna,
        }
        if not delta.tabela_existe(spark, alvo):
            item["resultado"] = v.REFERENCIA_AUSENTE
            itens.append(item)
            continue
        valores = (
            spark.table(tabela)
            .select(F.col(coluna.nome).alias("valor"))
            .where(F.col("valor").isNotNull())
            .distinct()
        )
        oficiais = (
            spark.table(alvo).select(F.col(dominio.coluna).alias("valor"))
        ).distinct()
        orfaos = valores.join(oficiais, "valor", "left_anti")
        item["orfaos_distintos"] = orfaos.count()
        item["exemplos"] = [
            linha["valor"]
            for linha in orfaos.orderBy("valor")
            .limit(LIMITE_EXEMPLOS)
            .collect()
        ]
        item["resultado"] = (
            v.CONFERIDA if item["orfaos_distintos"] == 0 else v.COM_ORFAOS
        )
        itens.append(item)
    return itens
'''

In [ ]:
_HUB_MODULOS["hub/spark/publicacao.py"] = r'''
"""Escrita reconciliada de um snapshot, comum à Bronze e à Silver.

Ordem: confere o schema, grava a quarentena (se houver), sobrescreve o
snapshot, reconcilia com medidas tomadas depois da escrita e, em
divergência, restaura a versão anterior antes de abortar.
"""

from __future__ import annotations

from dataclasses import dataclass

from pyspark.sql import DataFrame, SparkSession

from hub import decisoes
from hub.erros import exigir
from hub.execucao import RegistroExecucao
from hub.spark import delta, quarentena


@dataclass(frozen=True)
class Destino:
    """Tabela de destino e o que o contrato espera dela."""

    tabela: str
    schema_fisico: tuple[tuple[str, str], ...]
    comentario_tabela: str
    comentarios_colunas: dict[str, str]
    tabela_quarentena: str
    identificador: str
    versao_contrato: int
    versao_vigente: int | None


def sobrescrever_schema(spark: SparkSession, destino: Destino) -> bool:
    """Confere a mudança de schema e indica se ela deve ser aplicada.

    Raises:
        BloqueioPublicacao: mudança sem nova versão de contrato.
    """
    if not delta.tabela_existe(spark, destino.tabela):
        return False
    diferenca = decisoes.diferenca_schema(
        delta.schema_fisico(spark, destino.tabela), destino.schema_fisico
    )
    exigir(
        decisoes.verificar_mudanca_schema(
            diferenca, destino.versao_contrato, destino.versao_vigente
        )
    )
    return not diferenca.vazia


def publicar(
    spark: SparkSession,
    destino: Destino,
    registro: RegistroExecucao,
    saida: DataFrame,
    publicadas_esperadas: int,
    retirados: DataFrame,
    quarentena_esperada: int,
) -> None:
    """Grava quarentena e snapshot e reconcilia depois da escrita.

    Pré-condição: ``registro.linhas_origem`` já medido de forma
    independente do parser.

    Raises:
        BloqueioPublicacao: schema sem nova versão ou reconciliação
            divergente (neste caso, após restaurar o snapshot anterior).
    """
    aplicar_schema = sobrescrever_schema(spark, destino)
    gravadas = 0
    if quarentena_esperada:
        gravadas = quarentena.gravar(
            spark, destino.tabela_quarentena, retirados, destino.identificador
        )
    escrita = delta.publicar_snapshot(
        spark, saida, destino.tabela, aplicar_schema
    )
    registro.versao_delta = escrita.versao
    registro.linhas_publicadas = escrita.linhas
    registro.linhas_quarentena = gravadas
    problemas = decisoes.verificar_reconciliacao(
        registro.linhas_origem, escrita.linhas, gravadas
    )
    if escrita.linhas != publicadas_esperadas:
        problemas.append(
            f"publicadas {escrita.linhas}; esperadas {publicadas_esperadas}"
        )
    if gravadas != quarentena_esperada:
        problemas.append(
            f"quarentena {gravadas}; esperada {quarentena_esperada}"
        )
    if problemas:
        delta.restaurar(spark, destino.tabela, escrita.versao_anterior)
        registro.detalhes["snapshot_restaurado"] = escrita.versao_anterior
        exigir(problemas)
    delta.aplicar_comentarios(
        spark,
        destino.tabela,
        destino.comentario_tabela,
        destino.comentarios_colunas,
    )
'''

In [ ]:
_HUB_MODULOS["hub/spark/quarentena.py"] = r'''
"""Quarentena: tabela única para todas as entidades.

A tabela é criada apenas quando há registros. Cada linha preserva o
registro (linha original na Bronze, JSON transformado na Silver), a
chave, o grupo do problema, todos os motivos e a identificação da
publicação. A quarentena nunca altera nem exclui registro da Bronze.
"""

from __future__ import annotations

from dataclasses import dataclass

from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F

from hub.execucao import COLUNA_PARTICAO, COLUNAS_QUARENTENA
from hub.spark import delta


@dataclass(frozen=True)
class ContextoQuarentena:
    """Identificação da publicação que retirou os registros.

    ``hash_decisao`` é o hash da seção executada pela camada
    (hash_leitura na Bronze, hash_contrato na Silver).
    """

    identificador: str
    camada: str
    fonte: str
    base: str
    entidade: str
    competencia: str
    publication_fingerprint: str
    versao_contrato: int
    hash_leitura: str
    hash_contrato: str
    hash_decisao: str
    id_execucao: str


def montar(
    df: DataFrame,
    contexto: ContextoQuarentena,
    registro: Column,
    hash_registro: Column,
    motivos: Column,
    chave: Column,
    grupo: Column,
    arquivo: Column,
) -> DataFrame:
    """Padroniza os registros retirados no formato da quarentena.

    O identificador é determinístico (publicação, seção executada,
    conteúdo e índice de ocorrência): reexecutar a mesma publicação sob o
    mesmo contrato não duplica evidência. Duplicatas idênticas recebem
    índices distintos, sem eleger vencedora.
    """
    base = df.select(
        registro.alias("registro"),
        hash_registro.alias("_hash_registro"),
        motivos.alias("motivos"),
        chave.alias("chave"),
        grupo.alias("grupo_problema"),
        arquivo.alias("arquivo_origem"),
    )
    janela = Window.partitionBy("_hash_registro").orderBy("_hash_registro")
    base = base.withColumn("ocorrencia", F.row_number().over(janela))
    identificacao = F.sha2(
        F.concat_ws(
            "|",
            F.lit(contexto.identificador),
            F.lit(contexto.camada),
            F.lit(contexto.competencia),
            F.lit(contexto.publication_fingerprint),
            F.lit(contexto.hash_decisao),
            F.col("_hash_registro"),
            F.col("ocorrencia").cast("string"),
        ),
        256,
    )
    valores = {
        "id_quarentena": identificacao,
        "identificador": F.lit(contexto.identificador),
        "camada": F.lit(contexto.camada),
        "fonte": F.lit(contexto.fonte),
        "base": F.lit(contexto.base),
        "entidade": F.lit(contexto.entidade),
        "competencia": F.lit(contexto.competencia),
        "publication_fingerprint": F.lit(contexto.publication_fingerprint),
        "versao_contrato": F.lit(contexto.versao_contrato).cast("int"),
        "hash_leitura": F.lit(contexto.hash_leitura),
        "hash_contrato": F.lit(contexto.hash_contrato),
        "id_execucao": F.lit(contexto.id_execucao),
        "chave": F.col("chave"),
        "grupo_problema": F.col("grupo_problema"),
        "motivos": F.col("motivos"),
        "registro": F.col("registro"),
        "arquivo_origem": F.col("arquivo_origem"),
        "ocorrencia": F.col("ocorrencia").cast("int"),
        "registrado_em": F.current_timestamp(),
    }
    return base.select(
        *[valores[nome].alias(nome) for nome, _ in COLUNAS_QUARENTENA]
    )


def gravar(
    spark: SparkSession, tabela: str, registros: DataFrame, identificador: str
) -> int:
    """Insere os registros novos e confere a presença de todos.

    Returns:
        Quantidade de registros desta publicação presentes na tabela,
        medida por leitura após a escrita.
    """
    if not delta.tabela_existe(spark, tabela):
        delta.criar_tabela(spark, tabela, COLUNAS_QUARENTENA, COLUNA_PARTICAO)
    literal = delta.literal_sql(identificador)
    condicao = (
        f"t.identificador = '{literal}' "
        "AND t.id_quarentena = s.id_quarentena"
    )
    delta.mesclar(spark, tabela, registros, condicao)
    return (
        spark.table(tabela)
        .where(F.col("identificador") == identificador)
        .join(registros.select("id_quarentena"), "id_quarentena", "left_semi")
        .count()
    )
'''

In [ ]:
_HUB_MODULOS["hub/spark/sessao.py"] = r'''
"""Configuração de sessão exigida pelo motor."""

from pyspark.sql import SparkSession

from hub.ambiente import Ambiente

CONFIGURACOES = {
    # Linha com número de campos diferente do layout é marcada como
    # malformada mesmo quando a consulta lê só parte das colunas.
    "spark.sql.csv.parser.columnPruning.enabled": "false",
    # Data inválida vira nulo (violação de conversão), não exceção.
    "spark.sql.legacy.timeParserPolicy": "CORRECTED",
}


def preparar(spark: SparkSession, ambiente: Ambiente) -> None:
    """Aplica as configurações de sessão e garante os schemas do ambiente."""
    for chave, valor in CONFIGURACOES.items():
        spark.conf.set(chave, valor)
    for schema in sorted(set(ambiente.schemas.values())):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
'''

In [ ]:
_HUB_MODULOS["hub/spark/silver.py"] = r'''
"""Publicação da Silver.

Aplica transformações, tipos, domínios literais, chave e regras
declarados. Cada violação carrega a ação declarada no contrato; o motor
não tem ação implícita. Linhas retiradas vão para a quarentena com
todos os motivos; BLOQUEIA_PUBLICACAO impede publicar qualquer linha.
"""

from __future__ import annotations

from functools import reduce
from operator import or_

from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, SparkSession, Window
from pyspark.sql import functions as F

from hub import decisoes
from hub import vocabulario as v
from hub.erros import exigir
from hub.execucao import RegistroExecucao
from hub.plano import ColunaSilver, PlanoSilver
from hub.spark import publicacao, quarentena, transformacoes

MOTIVOS = "_hub_motivos"
RETIRAR = "_hub_retirar"
ALERTAR = "_hub_alertar"
BLOQUEAR = "_hub_bloquear"
CONTAGEM_CHAVE = "_hub_contagem_chave"
TIPO_MOTIVOS = (
    "array<struct<classe:string,regra:string,coluna:string,"
    "valor:string,acao:string,dimensao:string>>"
)
LIMITE_EXEMPLOS = 10


def _origem(nome: str) -> str:
    """Nome interno do valor original (texto da Bronze)."""
    return f"_hub_origem_{nome}"


def _texto(nome: str) -> str:
    """Nome interno do valor transformado, ainda como texto."""
    return f"_hub_texto_{nome}"


def _motivo(
    condicao: Column,
    classe: str,
    acao: str,
    dimensao: str,
    coluna: str | None = None,
    regra: str | None = None,
    valor: Column | None = None,
) -> Column:
    """Motivo de violação quando a condição é verdadeira; nulo caso não."""
    return F.when(
        condicao,
        F.struct(
            F.lit(classe).alias("classe"),
            F.lit(regra).cast("string").alias("regra"),
            F.lit(coluna).cast("string").alias("coluna"),
            (F.lit(None) if valor is None else valor)
            .cast("string")
            .alias("valor"),
            F.lit(acao).alias("acao"),
            F.lit(dimensao).alias("dimensao"),
        ),
    )


def _motivos_coluna(coluna: ColunaSilver, acoes: dict) -> list[Column]:
    """Conversão, nulidade e domínio literal de uma coluna."""
    tipada = F.col(coluna.nome)
    original = F.col(_origem(coluna.nome))
    motivos = []
    if coluna.tipo != v.TEXTO:
        motivos.append(
            _motivo(
                F.col(_texto(coluna.nome)).isNotNull() & tipada.isNull(),
                v.CONVERSAO,
                acoes[v.CONVERSAO],
                v.DIMENSAO_POR_CLASSE[v.CONVERSAO],
                coluna=coluna.nome,
                valor=original,
            )
        )
    if coluna.verificar_nulidade:
        motivos.append(
            _motivo(
                tipada.isNull(),
                v.NULIDADE,
                acoes[v.NULIDADE],
                v.DIMENSAO_POR_CLASSE[v.NULIDADE],
                coluna=coluna.nome,
                valor=original,
            )
        )
    if coluna.valores_dominio is not None:
        fora = tipada.isNotNull() & ~tipada.isin(list(coluna.valores_dominio))
        motivos.append(
            _motivo(
                fora,
                v.DOMINIO,
                acoes[v.DOMINIO],
                v.DIMENSAO_POR_CLASSE[v.DOMINIO],
                coluna=coluna.nome,
                valor=original,
            )
        )
    return motivos


def _motivos_chave(plano: PlanoSilver) -> list[Column]:
    """Chave nula e chave duplicada (todo o grupo sai, sem vencedor)."""
    if not plano.chave:
        return []
    nomes = ",".join(plano.chave)
    alguma_nula = reduce(or_, [F.col(nome).isNull() for nome in plano.chave])
    valor_original = F.concat_ws(
        "|", *[F.col(_origem(nome)) for nome in plano.chave]
    )
    nula = _motivo(
        alguma_nula,
        v.CHAVE_NULA,
        plano.acoes[v.CHAVE_NULA],
        v.DIMENSAO_POR_CLASSE[v.CHAVE_NULA],
        coluna=nomes,
        valor=valor_original,
    )
    duplicada = _motivo(
        ~alguma_nula & (F.col(CONTAGEM_CHAVE) > 1),
        v.CHAVE_DUPLICADA,
        plano.acoes[v.CHAVE_DUPLICADA],
        v.DIMENSAO_POR_CLASSE[v.CHAVE_DUPLICADA],
        coluna=nomes,
        valor=valor_original,
    )
    return [nula, duplicada]


def _motivos_regras(plano: PlanoSilver) -> list[Column]:
    """Regras declaradas: só TRUE aprova; FALSE e NULL violam."""
    motivos = []
    for regra in plano.regras:
        valores = F.to_json(
            F.struct(*[F.col(nome) for nome in regra.colunas_referenciadas])
        )
        motivos.append(
            _motivo(
                ~F.coalesce(F.expr(regra.expressao), F.lit(False)),
                v.REGRA,
                regra.acao,
                regra.dimensao,
                coluna=",".join(regra.colunas_referenciadas),
                regra=regra.id,
                valor=valores,
            )
        )
    return motivos


def avaliar(bronze: DataFrame, plano: PlanoSilver) -> DataFrame:
    """Transforma, converte e anexa motivos e destino de cada linha."""
    etapa = bronze.select(
        *[F.col(c.nome).alias(_origem(c.nome)) for c in plano.colunas],
        F.col(v.COLUNA_ARQUIVO),
    )
    etapa = etapa.select(
        "*",
        *[
            transformacoes.aplicar(
                F.col(_origem(c.nome)), c.transformacoes
            ).alias(_texto(c.nome))
            for c in plano.colunas
        ],
    )
    etapa = etapa.select(
        "*",
        *[
            transformacoes.converter(_texto(c.nome), c).alias(c.nome)
            for c in plano.colunas
        ],
    )
    descricoes = [
        transformacoes.descrever(c).alias(c.coluna_descricao)
        for c in plano.colunas
        if c.coluna_descricao
    ]
    if descricoes:
        etapa = etapa.select("*", *descricoes)
    if plano.chave:
        janela = Window.partitionBy(*plano.chave)
        etapa = etapa.withColumn(
            CONTAGEM_CHAVE, F.count(F.lit(1)).over(janela)
        )
    motivos = [
        m for c in plano.colunas for m in _motivos_coluna(c, plano.acoes)
    ]
    motivos += _motivos_chave(plano) + _motivos_regras(plano)
    if motivos:
        lista = F.filter(F.array(*motivos), lambda m: m.isNotNull())
    else:
        lista = F.array().cast(TIPO_MOTIVOS)
    retirar = sorted(v.ACOES_QUE_RETIRAM)
    alertar = sorted(v.ACOES_QUE_ALERTAM)
    return (
        etapa.withColumn(MOTIVOS, lista)
        .withColumn(
            RETIRAR, F.exists(MOTIVOS, lambda m: m["acao"].isin(retirar))
        )
        .withColumn(
            ALERTAR, F.exists(MOTIVOS, lambda m: m["acao"].isin(alertar))
        )
        .withColumn(
            BLOQUEAR,
            F.exists(MOTIVOS, lambda m: m["acao"] == v.BLOQUEIA_PUBLICACAO),
        )
    )


def _resumo(avaliado: DataFrame) -> dict:
    """Totais por destino e contagem de violações por motivo."""
    totais = avaliado.agg(
        F.count(F.lit(1)).alias("total"),
        F.sum(F.col(RETIRAR).cast("int")).alias("retirar"),
        F.sum(F.col(ALERTAR).cast("int")).alias("alertar"),
        F.sum(F.col(BLOQUEAR).cast("int")).alias("bloquear"),
    ).first()
    violacoes = (
        avaliado.select(F.explode(MOTIVOS).alias("m"))
        .groupBy("m.classe", "m.regra", "m.coluna", "m.acao")
        .count()
        .collect()
    )
    return {
        "total": totais["total"],
        "retirar": totais["retirar"] or 0,
        "alertar": totais["alertar"] or 0,
        "bloquear": totais["bloquear"] or 0,
        "violacoes": sorted(
            (linha.asDict() for linha in violacoes),
            key=lambda item: (item["classe"], item["coluna"] or ""),
        ),
    }


def _exemplos_bloqueio(avaliado: DataFrame) -> list[dict]:
    """Alguns valores que dispararam BLOQUEIA_PUBLICACAO."""
    linhas = (
        avaliado.where(F.col(BLOQUEAR))
        .select(F.explode(MOTIVOS).alias("m"))
        .where(F.col("m.acao") == v.BLOQUEIA_PUBLICACAO)
        .select("m.classe", "m.regra", "m.coluna", "m.valor")
        .limit(LIMITE_EXEMPLOS)
        .collect()
    )
    return [linha.asDict() for linha in linhas]


def _saida(
    avaliado: DataFrame, plano: PlanoSilver, competencia: str, id_execucao: str
) -> DataFrame:
    """Linhas publicáveis nas colunas de negócio, com a linhagem mínima."""
    return avaliado.where(~F.col(RETIRAR)).select(
        *[F.col(nome) for nome in plano.colunas_negocio],
        F.lit(competencia).alias(v.COLUNA_COMPETENCIA),
        F.col(v.COLUNA_ARQUIVO),
        F.lit(id_execucao).alias(v.COLUNA_EXECUCAO),
    )


def _retirados(
    avaliado: DataFrame,
    plano: PlanoSilver,
    contexto: quarentena.ContextoQuarentena,
) -> DataFrame:
    """Linhas retiradas no formato da quarentena (registro em JSON)."""
    if plano.chave:
        chave = F.to_json(F.struct(*[F.col(nome) for nome in plano.chave]))
    else:
        chave = F.lit(None).cast("string")
    grupo = F.array_join(
        F.array_sort(
            F.array_distinct(
                F.transform(
                    MOTIVOS,
                    lambda m: F.concat_ws(
                        ":", m["classe"], F.coalesce(m["regra"], m["coluna"])
                    ),
                )
            )
        ),
        "|",
    )
    originais = F.struct(*[F.col(_origem(c.nome)) for c in plano.colunas])
    return quarentena.montar(
        avaliado.where(F.col(RETIRAR)),
        contexto=contexto,
        registro=F.to_json(
            F.struct(*[F.col(nome) for nome in plano.colunas_negocio])
        ),
        hash_registro=F.sha2(F.to_json(originais), 256),
        motivos=F.to_json(F.col(MOTIVOS)),
        chave=chave,
        grupo=grupo,
        arquivo=F.col(v.COLUNA_ARQUIVO),
    )


def ler_bronze(
    spark: SparkSession, plano: PlanoSilver, publicacao_bronze: dict
) -> DataFrame:
    """Lê a Bronze após confirmar que ela é exatamente a publicação
    registrada (mesma execução, competência e contagem)."""
    bronze = spark.table(plano.tabela_bronze)
    contagens = {
        (linha[v.COLUNA_EXECUCAO], linha[v.COLUNA_COMPETENCIA]): linha["count"]
        for linha in bronze.groupBy(v.COLUNA_EXECUCAO, v.COLUNA_COMPETENCIA)
        .count()
        .collect()
    }
    exigir(decisoes.verificar_snapshot_bronze(contagens, publicacao_bronze))
    return bronze


def publicar(
    spark: SparkSession,
    plano: PlanoSilver,
    destino: publicacao.Destino,
    registro: RegistroExecucao,
    contexto: quarentena.ContextoQuarentena,
    publicacao_bronze: dict,
) -> None:
    """Executa a Silver de ponta a ponta, preenchendo o registro."""
    bronze = ler_bronze(spark, plano, publicacao_bronze)
    registro.linhas_origem = publicacao_bronze["linhas_publicadas"]
    avaliado = avaliar(bronze, plano).persist(StorageLevel.MEMORY_AND_DISK)
    try:
        resumo = _resumo(avaliado)
        registro.detalhes["violacoes"] = resumo["violacoes"]
        if resumo["total"] != registro.linhas_origem:
            exigir(
                [
                    f"Silver avaliou {resumo['total']} linhas; Bronze "
                    f"publicou {registro.linhas_origem}"
                ]
            )
        if resumo["bloquear"]:
            registro.detalhes["exemplos_bloqueio"] = _exemplos_bloqueio(
                avaliado
            )
            exigir(
                [
                    f"{resumo['bloquear']} linhas com violação de ação "
                    f"{v.BLOQUEIA_PUBLICACAO}"
                ]
            )
        if resumo["alertar"]:
            registro.alertar(
                f"{resumo['alertar']} linhas com violação que gera alerta"
            )
        publicacao.publicar(
            spark,
            destino,
            registro,
            saida=_saida(
                avaliado, plano, contexto.competencia, registro.id_execucao
            ),
            publicadas_esperadas=resumo["total"] - resumo["retirar"],
            retirados=_retirados(avaliado, plano, contexto),
            quarentena_esperada=resumo["retirar"],
        )
    finally:
        avaliado.unpersist()
'''

In [ ]:
_HUB_MODULOS["hub/spark/transformacoes.py"] = r'''
"""Transformações e conversões do vocabulário fechado.

Somente funções nativas do Spark (sem UDF Python). Toda transformação
preserva nulo: valor nulo continua nulo.
"""

from __future__ import annotations

from collections.abc import Iterable, Mapping
from typing import Any

from pyspark.sql import Column
from pyspark.sql import functions as F

from hub import vocabulario as v
from hub.contrato import Transformacao
from hub.plano import ColunaSilver


def _nulo() -> Column:
    """Literal nulo do tipo texto (criado sob demanda, com sessão ativa)."""
    return F.lit(None).cast("string")


def _aparar(coluna: Column, parametros: Mapping[str, Any]) -> Column:
    """Remove espaços nas extremidades."""
    return F.trim(coluna)


def _maiusculas(coluna: Column, parametros: Mapping[str, Any]) -> Column:
    """Converte para maiúsculas."""
    return F.upper(coluna)


def _somente_digitos(coluna: Column, parametros: Mapping[str, Any]) -> Column:
    """Remove todo caractere que não seja dígito."""
    return F.regexp_replace(coluna, r"[^0-9]", "")


def _vazio_como_nulo(coluna: Column, parametros: Mapping[str, Any]) -> Column:
    """Texto vazio passa a nulo."""
    return F.when(coluna == "", _nulo()).otherwise(coluna)


def _nulo_se(coluna: Column, parametros: Mapping[str, Any]) -> Column:
    """Valores sentinela declarados passam a nulo."""
    return F.when(coluna.isin(list(parametros["valores"])), _nulo()).otherwise(
        coluna
    )


def _preencher_esquerda(
    coluna: Column, parametros: Mapping[str, Any]
) -> Column:
    """Completa à esquerda até o tamanho; nunca trunca valor maior."""
    tamanho = parametros["tamanho"]
    preenchida = F.lpad(coluna, tamanho, parametros["caractere"])
    return F.when(F.length(coluna) < tamanho, preenchida).otherwise(coluna)


IMPLEMENTACOES = {
    "aparar": _aparar,
    "maiusculas": _maiusculas,
    "somente_digitos": _somente_digitos,
    "vazio_como_nulo": _vazio_como_nulo,
    "nulo_se": _nulo_se,
    "preencher_esquerda": _preencher_esquerda,
}


def aplicar(coluna: Column, transformacoes: Iterable[Transformacao]) -> Column:
    """Aplica as transformações na ordem declarada."""
    for transformacao in transformacoes:
        implementacao = IMPLEMENTACOES[transformacao.tipo]
        coluna = implementacao(coluna, transformacao.parametros)
    return coluna


def converter(nome_texto: str, coluna: ColunaSilver) -> Column:
    """Converte a coluna textual transformada para o tipo do contrato.

    Usa try_cast e try_to_timestamp: valor inválido vira nulo, com ou
    sem modo ANSI, e é tratado depois como violação de conversão.
    """
    referencia = f"`{nome_texto}`"
    if coluna.tipo == v.TEXTO:
        return F.col(nome_texto)
    if coluna.tipo == v.DATA:
        return F.expr(
            f"to_date(try_to_timestamp({referencia}, "
            f"'{coluna.padrao_data}'))"
        )
    if coluna.tipo == v.DECIMAL and coluna.separador_decimal != ".":
        referencia = (
            f"replace({referencia}, '{coluna.separador_decimal}', '.')"
        )
    return F.expr(f"try_cast({referencia} AS {coluna.tipo_fisico.upper()})")


def descrever(coluna: ColunaSilver) -> Column:
    """Descrição do código conforme o domínio literal (nulo se ausente)."""
    expressao = None
    for codigo, descricao in sorted(coluna.valores_dominio.items()):
        condicao = F.col(coluna.nome) == codigo
        if expressao is None:
            expressao = F.when(condicao, descricao)
        else:
            expressao = expressao.when(condicao, descricao)
    return expressao
'''

In [ ]:
_HUB_MODULOS["hub/vocabulario.py"] = r'''
"""Vocabulário fechado do motor.

Fonte única para ações, classes de violação, tipos, transformações,
estados de contrato, dimensões de qualidade, status de execução e nomes
reservados. Qualquer termo fora destas listas é recusado na validação.
"""

from typing import NamedTuple

# Ações declaradas no contrato (seção 5.3 do briefing).
QUARENTENA = "QUARENTENA"
QUARENTENA_E_ALERTA = "QUARENTENA_E_ALERTA"
QUARENTENA_GRUPO = "QUARENTENA_GRUPO"
ALERTA = "ALERTA"
BLOQUEIA_PUBLICACAO = "BLOQUEIA_PUBLICACAO"

ACOES = (
    QUARENTENA,
    QUARENTENA_E_ALERTA,
    QUARENTENA_GRUPO,
    ALERTA,
    BLOQUEIA_PUBLICACAO,
)
ACOES_QUE_RETIRAM = frozenset(
    {QUARENTENA, QUARENTENA_E_ALERTA, QUARENTENA_GRUPO}
)
ACOES_QUE_ALERTAM = frozenset({ALERTA, QUARENTENA_E_ALERTA})

# Classes de violação.
LINHA_MALFORMADA = "linha_malformada"
NULIDADE = "nulidade"
CONVERSAO = "conversao"
CHAVE_NULA = "chave_nula"
CHAVE_DUPLICADA = "chave_duplicada"
DOMINIO = "dominio"
REGRA = "regra"

# Classes cuja ação é declarada em acoes_padrao (todas obrigatórias).
CLASSES_PADRAO = (
    LINHA_MALFORMADA,
    NULIDADE,
    CONVERSAO,
    CHAVE_NULA,
    CHAVE_DUPLICADA,
    DOMINIO,
)

_ACOES_DE_LINHA = frozenset(
    {QUARENTENA, QUARENTENA_E_ALERTA, ALERTA, BLOQUEIA_PUBLICACAO}
)

# Matriz de ações permitidas. ALERTA em chave duplicada publicaria
# duplicidade; QUARENTENA simples exigiria eleger um vencedor.
ACOES_PERMITIDAS = {
    LINHA_MALFORMADA: frozenset(
        {QUARENTENA, QUARENTENA_E_ALERTA, BLOQUEIA_PUBLICACAO}
    ),
    NULIDADE: _ACOES_DE_LINHA,
    CONVERSAO: _ACOES_DE_LINHA,
    CHAVE_NULA: frozenset(
        {QUARENTENA, QUARENTENA_E_ALERTA, BLOQUEIA_PUBLICACAO}
    ),
    CHAVE_DUPLICADA: frozenset({QUARENTENA_GRUPO, BLOQUEIA_PUBLICACAO}),
    DOMINIO: _ACOES_DE_LINHA,
    REGRA: _ACOES_DE_LINHA,
}

# Dimensões de qualidade (vocabulário do Microsoft Purview).
COMPLETENESS = "Completeness"
CONSISTENCY = "Consistency"
CONFORMITY = "Conformity"
ACCURACY = "Accuracy"
UNIQUENESS = "Uniqueness"

# Freshness é lacuna conhecida: não há periodicidade, tolerância e ação.
DIMENSOES = (COMPLETENESS, CONSISTENCY, CONFORMITY, ACCURACY, UNIQUENESS)

DIMENSAO_POR_CLASSE = {
    LINHA_MALFORMADA: CONFORMITY,
    NULIDADE: COMPLETENESS,
    CONVERSAO: COMPLETENESS,
    CHAVE_NULA: COMPLETENESS,
    CHAVE_DUPLICADA: UNIQUENESS,
    DOMINIO: CONSISTENCY,
}

# Tipos lógicos de coluna e seus parâmetros obrigatórios.
TEXTO = "texto"
INTEIRO = "inteiro"
INTEIRO_LONGO = "inteiro_longo"
DECIMAL = "decimal"
DATA = "data"

TIPOS = {
    TEXTO: (),
    INTEIRO: (),
    INTEIRO_LONGO: (),
    DECIMAL: ("precisao", "escala", "separador_decimal"),
    DATA: ("formato",),
}
PARAMETROS_DE_TIPO = ("formato", "precisao", "escala", "separador_decimal")
SEPARADORES_DECIMAIS = (".", ",")
PRECISAO_MAXIMA = 38

# Notação de data do layout oficial convertida para o padrão do motor.
TOKENS_DATA = {"AAAA": "yyyy", "MM": "MM", "DD": "dd"}
SEPARADORES_DATA = ("-", "/", ".")

# Transformações e a especificação de seus parâmetros.
LISTA_TEXTO = "lista_texto"
INTEIRO_POSITIVO = "inteiro_positivo"
CARACTERE = "caractere"

TRANSFORMACOES = {
    "aparar": {},
    "maiusculas": {},
    "somente_digitos": {},
    "vazio_como_nulo": {},
    "nulo_se": {"valores": LISTA_TEXTO},
    "preencher_esquerda": {
        "tamanho": INTEIRO_POSITIVO,
        "caractere": CARACTERE,
    },
}


class Codificacao(NamedTuple):
    """Nome do codec no Python e do charset no leitor do Spark."""

    python: str
    spark: str


CODIFICACOES = {
    "utf-8": Codificacao("utf-8", "UTF-8"),
    "iso-8859-1": Codificacao("iso-8859-1", "ISO-8859-1"),
    "windows-1252": Codificacao("cp1252", "windows-1252"),
}

# Ciclo de vida do contrato.
PROVISORIO_DOCUMENTAL = "PROVISORIO_DOCUMENTAL"
VALIDADO_FISICO = "VALIDADO_FISICO"
APROVADO = "APROVADO"
OBSOLETO = "OBSOLETO"

ESTADOS = (PROVISORIO_DOCUMENTAL, VALIDADO_FISICO, APROVADO, OBSOLETO)
ORDEM_ESTADOS = {PROVISORIO_DOCUMENTAL: 0, VALIDADO_FISICO: 1, APROVADO: 2}
ESTADOS_IMPEDIVEIS = (VALIDADO_FISICO, APROVADO)

# Vocabulário das expressões de regra. Funções não determinísticas
# (current_date, now, rand) ficam de fora para preservar a idempotência.
PALAVRAS_EXPRESSAO = frozenset(
    {
        "AND",
        "OR",
        "NOT",
        "IS",
        "NULL",
        "IN",
        "BETWEEN",
        "LIKE",
        "TRUE",
        "FALSE",
        "CASE",
        "WHEN",
        "THEN",
        "ELSE",
        "END",
        "DATE",
    }
)
FUNCOES_EXPRESSAO = frozenset(
    {
        "abs",
        "coalesce",
        "day",
        "length",
        "lower",
        "month",
        "regexp_like",
        "round",
        "substring",
        "trim",
        "upper",
        "year",
    }
)

# Camadas e tipos de execução.
BRONZE = "bronze"
SILVER = "silver"
CONTROLE = "controle"
CAMADAS_DE_SCHEMA = (BRONZE, SILVER, CONTROLE)

AQUISICAO = "AQUISICAO"
INSPECAO = "INSPECAO"
PUBLICACAO_BRONZE = "BRONZE"
PUBLICACAO_SILVER = "SILVER"
APROVACAO = "APROVACAO"
DIAGNOSTICO = "DIAGNOSTICO"
CATALOGO = "CATALOGO"

# Status de execução.
EM_EXECUCAO = "EM_EXECUCAO"
SUCESSO = "SUCESSO"
SUCESSO_COM_ALERTA = "SUCESSO_COM_ALERTA"
BLOQUEADA = "BLOQUEADA"
FALHA = "FALHA"
STATUS_EFETIVOS = frozenset({SUCESSO, SUCESSO_COM_ALERTA})

# Resultados registrados por tipo de execução.
ADQUIRIDA = "ADQUIRIDA"
EXISTENTE = "EXISTENTE"
COMPATIVEL = "COMPATIVEL"
INCOMPATIVEL = "INCOMPATIVEL"
APROVADA = "APROVADA"
CONFERIDA = "CONFERIDA"
COM_ORFAOS = "COM_ORFAOS"
REFERENCIA_AUSENTE = "REFERENCIA_AUSENTE"

# Colunas de linhagem mínima acrescentadas pelo motor.
COLUNA_COMPETENCIA = "competencia"
COLUNA_ARQUIVO = "_arquivo_origem"
COLUNA_EXECUCAO = "_id_execucao"
COLUNAS_LINHAGEM = (COLUNA_COMPETENCIA, COLUNA_ARQUIVO, COLUNA_EXECUCAO)

# Valor dos campos de curadoria que ainda não foram preenchidos.
A_CONFIRMAR = "A_CONFIRMAR"

# Identificador das execuções que não pertencem a uma entidade ou base.
IDENTIFICADOR_GLOBAL = "hub"
SINCRONIZADO = "SINCRONIZADO"
'''

In [ ]:
_HUB_DADOS = json.loads(
    r'''{
 "bases": {
  "rfb_cno": {
   "aquisicao": {
    "arquivos": [
     "cno.zip"
    ],
    "url_base": "https://arquivos.receitafederal.gov.br/dados/cno/"
   },
   "descricao": "Cadastro Nacional de Obras (CNO): obras de construção civil, atividades econômicas, vínculos de responsáveis, áreas e totais de controle.",
   "documentacao": {
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout dos dados abertos (Receita Federal).",
    "pendencias": [
     "Confirmar a URL vigente de publicação do arquivo cno.zip.",
     "A URL não identifica a competência; a competência é o rótulo informado na aquisição."
    ]
   },
   "formato_base": 1,
   "identidade": {
    "base": "cno",
    "fonte": "rfb"
   }
  },
  "rfb_cnpj": {
   "aquisicao": {
    "arquivos": [
     "Cnaes.zip",
     "Empresas0.zip",
     "Empresas1.zip",
     "Empresas2.zip",
     "Empresas3.zip",
     "Empresas4.zip",
     "Empresas5.zip",
     "Empresas6.zip",
     "Empresas7.zip",
     "Empresas8.zip",
     "Empresas9.zip",
     "Estabelecimentos0.zip",
     "Estabelecimentos1.zip",
     "Estabelecimentos2.zip",
     "Estabelecimentos3.zip",
     "Estabelecimentos4.zip",
     "Estabelecimentos5.zip",
     "Estabelecimentos6.zip",
     "Estabelecimentos7.zip",
     "Estabelecimentos8.zip",
     "Estabelecimentos9.zip",
     "Motivos.zip",
     "Municipios.zip",
     "Naturezas.zip",
     "Paises.zip",
     "Qualificacoes.zip",
     "Simples.zip",
     "Socios0.zip",
     "Socios1.zip",
     "Socios2.zip",
     "Socios3.zip",
     "Socios4.zip",
     "Socios5.zip",
     "Socios6.zip",
     "Socios7.zip",
     "Socios8.zip",
     "Socios9.zip"
    ],
    "url_base": "https://arquivos.receitafederal.gov.br/dados/cnpj/dados_abertos_cnpj/{competencia}/"
   },
   "descricao": "Dados abertos do CNPJ: empresas, estabelecimentos, sócios, opção pelo Simples/MEI e tabelas de domínio (CNAE, município, natureza jurídica, país, qualificação e motivo de situação cadastral).",
   "documentacao": {
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), Receita Federal.",
    "pendencias": [
     "Confirmar a URL vigente e a lista de arquivos da publicação mensal."
    ]
   },
   "formato_base": 1,
   "identidade": {
    "base": "cnpj",
    "fonte": "rfb"
   }
  }
 },
 "configuracao": {
  "ambientes": {
   "dev": {
    "montagem_local": "/lakehouse/default",
    "raiz_raw": "Files/raw",
    "schemas": {
     "bronze": "bronze",
     "controle": "controle",
     "silver": "silver"
    }
   },
   "hml": {
    "montagem_local": "/lakehouse/default",
    "raiz_raw": "Files/raw",
    "schemas": {
     "bronze": "bronze",
     "controle": "controle",
     "silver": "silver"
    }
   },
   "prod": {
    "montagem_local": "/lakehouse/default",
    "raiz_raw": "Files/raw",
    "schemas": {
     "bronze": "bronze",
     "controle": "controle",
     "silver": "silver"
    }
   }
  },
  "formato_configuracao": 1,
  "lakehouse": "lh_hub"
 },
 "contratos": {
  "rfb_cno_areas": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "colunas": [
    {
     "cabecalho": "CNO",
     "descricao": "Número de inscrição da obra no CNO.",
     "nome": "cno",
     "nulavel": false,
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo"
     ]
    },
    {
     "cabecalho": "Categoria",
     "descricao": "Categoria da área.",
     "dominio": "categoria_area",
     "nome": "categoria",
     "tipo": "texto"
    },
    {
     "cabecalho": "Destinação",
     "descricao": "Destinação da área.",
     "dominio": "destinacao_area",
     "nome": "destinacao",
     "tipo": "texto"
    },
    {
     "cabecalho": "Tipo de Obra",
     "descricao": "Tipo de construção da obra.",
     "dominio": "tipo_obra",
     "nome": "tipo_obra",
     "tipo": "texto"
    },
    {
     "cabecalho": "Tipo de Area",
     "descricao": "Tipo da área (principal ou complementar).",
     "dominio": "tipo_area",
     "nome": "tipo_area",
     "tipo": "texto"
    },
    {
     "cabecalho": "Tipo de Área Complementar",
     "descricao": "Tipo da área complementar.",
     "dominio": "tipo_area_complementar",
     "nome": "tipo_area_complementar",
     "tipo": "texto"
    },
    {
     "cabecalho": "Metragem",
     "descricao": "Metragem da área.",
     "escala": 2,
     "nome": "metragem",
     "precisao": 22,
     "separador_decimal": ".",
     "tipo": "decimal"
    }
   ],
   "descricao": "Áreas de cada obra do CNO por categoria, destinação, tipo de obra e tipo de área.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout do arquivo CNO_AREAS.CSV (Receita Federal).",
    "pendencias": [
     {
      "descricao": "Confirmar dialeto, codificação e nomes do cabeçalho no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Campos codificados têm tamanho 150 no layout; confirmar se o arquivo traz código ou descrição e revisar a ação de domínio (hoje ALERTA).",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar separador decimal e escala da metragem.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "dominios": {
    "categoria_area": {
     "coluna_descricao": "categoria_descricao",
     "tipo": "literal",
     "valores": {
      "0": "Obra Nova",
      "1": "Acréscimo",
      "2": "Reforma",
      "3": "Demolição",
      "4": "Existente"
     }
    },
    "destinacao_area": {
     "coluna_descricao": "destinacao_descricao",
     "tipo": "literal",
     "valores": {
      "0": "Residencial unifamiliar",
      "1": "Residencial multifamiliar",
      "2": "Comercial salas e lojas",
      "3": "Edifício de Garagens",
      "4": "Galpão industrial",
      "5": "Casa popular",
      "6": "Conjunto habitacional popular"
     }
    },
    "tipo_area": {
     "coluna_descricao": "tipo_area_descricao",
     "tipo": "literal",
     "valores": {
      "C": "Complementar",
      "P": "Principal"
     }
    },
    "tipo_area_complementar": {
     "coluna_descricao": "tipo_area_complementar_descricao",
     "tipo": "literal",
     "valores": {
      "0": "Quadra Esportiva e Poliesportiva",
      "1": "Estacionamento Térreo",
      "2": "Piscina",
      "3": "Área Complementar do Posto de Gasolina"
     }
    },
    "tipo_obra": {
     "coluna_descricao": "tipo_obra_descricao",
     "tipo": "literal",
     "valores": {
      "0": "Alvenaria",
      "1": "Madeira",
      "2": "Mista"
     }
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cno",
    "entidade": "areas",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": true,
    "codificacao": "utf-8",
    "escape": "\"",
    "separador": ",",
    "tolerancia_malformadas": 0.001
   },
   "reconciliacao": {
    "total_declarado": {
     "coluna": "total_areas",
     "entidade": "totais"
    }
   },
   "regras": [
    {
     "acao": "ALERTA",
     "colunas_referenciadas": [
      "metragem"
     ],
     "descricao": "A metragem não pode ser negativa.",
     "dimensao": "Accuracy",
     "expressao": "metragem IS NULL OR metragem >= 0",
     "id": "metragem_nao_negativa"
    }
   ],
   "selecao": {
    "arquivo": "cno_areas.csv",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cno_cnaes": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "chave": [
    "cno",
    "cnae"
   ],
   "colunas": [
    {
     "cabecalho": "CNO",
     "descricao": "Número de inscrição da obra no CNO.",
     "nome": "cno",
     "nulavel": false,
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo"
     ]
    },
    {
     "cabecalho": "CNAE",
     "descricao": "Código CNAE (subclasse, 7 dígitos) da atividade da obra.",
     "dominio": "cnae_rfb",
     "nome": "cnae",
     "nulavel": false,
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo",
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 7
       }
      }
     ]
    },
    {
     "cabecalho": "Data de registro",
     "descricao": "Data de registro da atividade.",
     "formato": "AAAA-MM-DD",
     "nome": "data_registro",
     "tipo": "data"
    }
   ],
   "descricao": "Atividades econômicas (CNAE) registradas para cada obra do CNO.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout do arquivo CNO_CNAES.CSV (Receita Federal).",
    "pendencias": [
     {
      "descricao": "Confirmar dialeto, codificação e nomes do cabeçalho no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar se (cno, cnae) é único; a chave foi inferida do layout.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "dominios": {
    "cnae_rfb": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "cnaes",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cno",
    "entidade": "cnaes",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": true,
    "codificacao": "utf-8",
    "escape": "\"",
    "separador": ",",
    "tolerancia_malformadas": 0.001
   },
   "reconciliacao": {
    "total_declarado": {
     "coluna": "total_cnaes",
     "entidade": "totais"
    }
   },
   "selecao": {
    "arquivo": "cno_cnaes.csv",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cno_obras": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "chave": [
    "cno"
   ],
   "colunas": [
    {
     "cabecalho": "CNO",
     "descricao": "Número de inscrição da obra no CNO.",
     "nome": "cno",
     "nulavel": false,
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo"
     ]
    },
    {
     "cabecalho": "Código do país",
     "descricao": "Código do país da obra.",
     "dominio": "pais_rfb",
     "nome": "codigo_pais",
     "tipo": "texto"
    },
    {
     "cabecalho": "Nome do País",
     "descricao": "Nome do país da obra.",
     "nome": "nome_pais",
     "tipo": "texto"
    },
    {
     "cabecalho": "Data de início",
     "descricao": "Data de início da obra.",
     "formato": "AAAA-MM-DD",
     "nome": "data_inicio",
     "tipo": "data"
    },
    {
     "cabecalho": "Data de início da responsabilidade",
     "descricao": "Data de início do período de responsabilidade da obra.",
     "formato": "AAAA-MM-DD",
     "nome": "data_inicio_responsabilidade",
     "tipo": "data"
    },
    {
     "cabecalho": "Data de registro",
     "descricao": "Data de registro da obra.",
     "formato": "AAAA-MM-DD",
     "nome": "data_registro",
     "tipo": "data"
    },
    {
     "cabecalho": "CNO vinculado",
     "descricao": "Número da inscrição vinculada da obra.",
     "nome": "cno_vinculado",
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo"
     ]
    },
    {
     "cabecalho": "CEP",
     "descricao": "CEP da obra (somente Brasil).",
     "nome": "cep",
     "tipo": "texto"
    },
    {
     "cabecalho": "NI do responsável",
     "descricao": "Número de identificação do responsável pela obra; vem em branco quando o responsável é pessoa física (CPF).",
     "nome": "ni_responsavel",
     "tipo": "texto"
    },
    {
     "cabecalho": "Qualificação do responsável",
     "descricao": "Código da qualificação do responsável pela obra.",
     "dominio": "qualificacao_responsavel",
     "nome": "qualificacao_responsavel",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 4
       }
      }
     ]
    },
    {
     "cabecalho": "Nome",
     "descricao": "Nome da obra.",
     "nome": "nome",
     "tipo": "texto"
    },
    {
     "cabecalho": "Código do município",
     "descricao": "Código do município da obra na tabela da Receita (TOM), não IBGE.",
     "dominio": "municipio_rfb",
     "nome": "codigo_municipio",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 4
       }
      }
     ]
    },
    {
     "cabecalho": "Nome do município",
     "descricao": "Nome do município da obra.",
     "nome": "nome_municipio",
     "tipo": "texto"
    },
    {
     "cabecalho": "Tipo de logradouro",
     "descricao": "Tipo de logradouro.",
     "nome": "tipo_logradouro",
     "tipo": "texto"
    },
    {
     "cabecalho": "Logradouro",
     "descricao": "Logradouro da obra.",
     "nome": "logradouro",
     "tipo": "texto"
    },
    {
     "cabecalho": "Número do logradouro",
     "descricao": "Número do logradouro.",
     "nome": "numero_logradouro",
     "tipo": "texto"
    },
    {
     "cabecalho": "Bairro",
     "descricao": "Bairro da obra.",
     "nome": "bairro",
     "tipo": "texto"
    },
    {
     "cabecalho": "Estado",
     "descricao": "Estado (unidade da federação) da obra.",
     "nome": "estado",
     "tipo": "texto"
    },
    {
     "cabecalho": "Caixa Postal",
     "descricao": "Caixa postal no exterior.",
     "nome": "caixa_postal",
     "tipo": "texto"
    },
    {
     "cabecalho": "Complemento",
     "descricao": "Complemento do endereço.",
     "nome": "complemento",
     "tipo": "texto"
    },
    {
     "cabecalho": "Unidade de medida",
     "descricao": "Unidade de medida da área da obra.",
     "nome": "unidade_medida",
     "tipo": "texto"
    },
    {
     "cabecalho": "Área total",
     "descricao": "Área total da obra, na unidade de medida informada.",
     "escala": 2,
     "nome": "area_total",
     "precisao": 18,
     "separador_decimal": ".",
     "tipo": "decimal"
    },
    {
     "cabecalho": "Situação",
     "descricao": "Código da situação da obra.",
     "dominio": "situacao_obra",
     "nome": "situacao",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 2
       }
      }
     ]
    },
    {
     "cabecalho": "Data da situação",
     "descricao": "Data da situação da obra.",
     "formato": "AAAA-MM-DD",
     "nome": "data_situacao",
     "tipo": "data"
    },
    {
     "cabecalho": "Nome empresarial",
     "descricao": "Nome empresarial do responsável pela obra; vem em branco quando o responsável é pessoa física.",
     "nome": "nome_empresarial",
     "tipo": "texto"
    },
    {
     "cabecalho": "Localização",
     "descricao": "Código da localização.",
     "nome": "localizacao",
     "tipo": "texto"
    }
   ],
   "descricao": "Obras de construção civil inscritas no Cadastro Nacional de Obras, com responsável, endereço, área e situação.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout do arquivo CNO.CSV (Receita Federal).",
    "pendencias": [
     {
      "descricao": "Confirmar separador, aspas, codificação e presença de cabeçalho no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar os nomes do cabeçalho; os declarados vêm dos atributos do layout.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "O layout informa datas com tamanho 8 e formato AAAA-MM-DD; confirmar o formato físico.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar separador decimal e escala de Área total.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar se o total de CNO_TOTAIS.CSV conta registros sem o cabeçalho.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Calibrar a tolerância de linhas malformadas com a evidência física.",
      "impede": "APROVADO"
     }
    ]
   },
   "dominios": {
    "municipio_rfb": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "municipios",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "pais_rfb": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "paises",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "qualificacao_responsavel": {
     "coluna_descricao": "qualificacao_responsavel_descricao",
     "tipo": "literal",
     "valores": {
      "0053": "Pessoa Jurídica Construtora",
      "0057": "Dono da Obra",
      "0064": "Incorporador de Construção Civil",
      "0070": "Proprietário do Imóvel",
      "0109": "Consórcio",
      "0110": "Construção em nome coletivo",
      "0111": "Sociedade Líder de Consórcio"
     }
    },
    "situacao_obra": {
     "coluna_descricao": "situacao_descricao",
     "tipo": "literal",
     "valores": {
      "01": "Nula",
      "02": "Ativa",
      "03": "Suspensa",
      "14": "Paralisada",
      "15": "Encerrada"
     }
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cno",
    "entidade": "obras",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": true,
    "codificacao": "utf-8",
    "escape": "\"",
    "separador": ",",
    "tolerancia_malformadas": 0.001
   },
   "reconciliacao": {
    "total_declarado": {
     "coluna": "total_obras",
     "entidade": "totais"
    }
   },
   "regras": [
    {
     "acao": "QUARENTENA_E_ALERTA",
     "colunas_referenciadas": [
      "cno"
     ],
     "descricao": "O número do CNO tem 12 dígitos.",
     "dimensao": "Conformity",
     "expressao": "length(cno) = 12",
     "id": "cno_com_12_digitos"
    },
    {
     "acao": "ALERTA",
     "colunas_referenciadas": [
      "area_total"
     ],
     "descricao": "A área total não pode ser negativa.",
     "dimensao": "Accuracy",
     "expressao": "area_total IS NULL OR area_total >= 0",
     "id": "area_total_nao_negativa"
    }
   ],
   "selecao": {
    "arquivo": "cno.csv",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cno_totais": {
   "acoes_padrao": {
    "chave_duplicada": "BLOQUEIA_PUBLICACAO",
    "chave_nula": "BLOQUEIA_PUBLICACAO",
    "conversao": "BLOQUEIA_PUBLICACAO",
    "dominio": "BLOQUEIA_PUBLICACAO",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "BLOQUEIA_PUBLICACAO"
   },
   "colunas": [
    {
     "cabecalho": "Total de obras",
     "descricao": "Total de registros do arquivo CNO.CSV.",
     "nome": "total_obras",
     "nulavel": false,
     "tipo": "inteiro_longo"
    },
    {
     "cabecalho": "Total de cnaes",
     "descricao": "Total de registros do arquivo CNO_CNAES.CSV.",
     "nome": "total_cnaes",
     "nulavel": false,
     "tipo": "inteiro_longo"
    },
    {
     "cabecalho": "Total de áreas",
     "descricao": "Total de registros do arquivo CNO_AREAS.CSV.",
     "nome": "total_areas",
     "nulavel": false,
     "tipo": "inteiro_longo"
    },
    {
     "cabecalho": "Total de vínculos",
     "descricao": "Total de registros do arquivo CNO_VINCULOS.CSV.",
     "nome": "total_vinculos",
     "nulavel": false,
     "tipo": "inteiro_longo"
    }
   ],
   "descricao": "Totais de registros dos arquivos do CNO, publicados pela própria fonte.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout do arquivo CNO_TOTAIS.CSV (Receita Federal).",
    "pendencias": [
     {
      "descricao": "Confirmar dialeto, codificação e nomes do cabeçalho no arquivo real.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cno",
    "entidade": "totais",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": true,
    "codificacao": "utf-8",
    "escape": "\"",
    "separador": ",",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "cno_totais.csv",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cno_vinculos": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "colunas": [
    {
     "cabecalho": "CNO",
     "descricao": "Número de inscrição da obra no CNO.",
     "nome": "cno",
     "nulavel": false,
     "tipo": "texto",
     "transformacoes": [
      "somente_digitos",
      "vazio_como_nulo"
     ]
    },
    {
     "cabecalho": "Data de início",
     "descricao": "Data de início do vínculo.",
     "formato": "AAAA-MM-DD",
     "nome": "data_inicio",
     "tipo": "data"
    },
    {
     "cabecalho": "Data de fim",
     "descricao": "Data de fim do vínculo.",
     "formato": "AAAA-MM-DD",
     "nome": "data_fim",
     "tipo": "data"
    },
    {
     "cabecalho": "Qualificação do contribuinte",
     "descricao": "Código da qualificação do contribuinte.",
     "dominio": "qualificacao_contribuinte",
     "nome": "qualificacao_contribuinte",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 4
       }
      }
     ]
    },
    {
     "cabecalho": "NI do responsável",
     "descricao": "Número de identificação do responsável; vem em branco quando o responsável é pessoa física (CPF).",
     "nome": "ni_responsavel",
     "tipo": "texto"
    }
   ],
   "descricao": "Vínculos de responsáveis com cada obra do CNO, com período e qualificação.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "CNO - Cadastro Nacional de Obras, layout do arquivo CNO_VINCULOS.CSV (Receita Federal).",
    "pendencias": [
     {
      "descricao": "Confirmar dialeto, codificação e nomes do cabeçalho no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Definir a chave do vínculo com evidência física; nenhuma foi declarada.",
      "impede": "APROVADO"
     }
    ]
   },
   "dominios": {
    "qualificacao_contribuinte": {
     "coluna_descricao": "qualificacao_contribuinte_descricao",
     "tipo": "literal",
     "valores": {
      "0053": "Pessoa Jurídica Construtora",
      "0057": "Dono da Obra",
      "0064": "Incorporador de Construção Civil",
      "0070": "Proprietário do Imóvel",
      "0109": "Consórcio",
      "0110": "Construção em nome coletivo",
      "0111": "Sociedade Líder de Consórcio"
     }
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cno",
    "entidade": "vinculos",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": true,
    "codificacao": "utf-8",
    "escape": "\"",
    "separador": ",",
    "tolerancia_malformadas": 0.001
   },
   "reconciliacao": {
    "total_declarado": {
     "coluna": "total_vinculos",
     "entidade": "totais"
    }
   },
   "regras": [
    {
     "acao": "ALERTA",
     "colunas_referenciadas": [
      "data_fim",
      "data_inicio"
     ],
     "descricao": "A data de fim do vínculo não é anterior à data de início.",
     "dimensao": "Accuracy",
     "expressao": "data_fim IS NULL OR data_inicio IS NULL OR data_fim >= data_inicio",
     "id": "fim_nao_anterior_ao_inicio"
    }
   ],
   "selecao": {
    "arquivo": "cno_vinculos.csv",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_cnaes": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código da atividade econômica.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome da atividade econômica.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de atividades econômicas (CNAE) da Receita Federal.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela de domínio CNAEs.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.CNAECSV).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "cnaes",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.CNAECSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_empresas": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "chave": [
    "cnpj_basico"
   ],
   "colunas": [
    {
     "descricao": "Número base de inscrição no CNPJ (oito primeiros dígitos).",
     "nome": "cnpj_basico",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome empresarial da pessoa jurídica.",
     "nome": "razao_social",
     "tipo": "texto"
    },
    {
     "descricao": "Código da natureza jurídica.",
     "dominio": "natureza_juridica",
     "nome": "natureza_juridica",
     "tipo": "texto"
    },
    {
     "descricao": "Qualificação da pessoa física responsável pela empresa.",
     "dominio": "qualificacao",
     "nome": "qualificacao_responsavel",
     "tipo": "texto"
    },
    {
     "descricao": "Capital social da empresa.",
     "escala": 2,
     "nome": "capital_social",
     "precisao": 18,
     "separador_decimal": ",",
     "tipo": "decimal"
    },
    {
     "descricao": "Código do porte da empresa.",
     "dominio": "porte_empresa",
     "nome": "porte_empresa",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 2
       }
      }
     ]
    },
    {
     "descricao": "Ente federativo responsável, preenchido para órgãos e entidades do grupo de natureza jurídica 1XXX.",
     "nome": "ente_federativo_responsavel",
     "tipo": "texto"
    }
   ],
   "descricao": "Dados cadastrais da pessoa jurídica (nível CNPJ básico).",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela EMPRESAS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar ordem e quantidade de colunas no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar codificação ISO-8859-1, separador ';' e aspas duplas.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Capital social com vírgula decimal não consta do layout; confirmar.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "dominios": {
    "natureza_juridica": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "naturezas",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "porte_empresa": {
     "coluna_descricao": "porte_empresa_descricao",
     "tipo": "literal",
     "valores": {
      "00": "Não informado",
      "01": "Micro empresa",
      "03": "Empresa de pequeno porte",
      "05": "Demais"
     }
    },
    "qualificacao": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "qualificacoes",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "empresas",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0001
   },
   "regras": [
    {
     "acao": "QUARENTENA_E_ALERTA",
     "colunas_referenciadas": [
      "cnpj_basico"
     ],
     "descricao": "O CNPJ básico tem oito dígitos.",
     "dimensao": "Conformity",
     "expressao": "regexp_like(cnpj_basico, '^[0-9]{8}$')",
     "id": "cnpj_basico_com_8_digitos"
    }
   ],
   "selecao": {
    "arquivo": "*.EMPRECSV",
    "quantidade_partes": 10
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_estabelecimentos": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "chave": [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv"
   ],
   "colunas": [
    {
     "descricao": "Número base de inscrição no CNPJ (oito primeiros dígitos).",
     "nome": "cnpj_basico",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Número do estabelecimento (nono ao décimo segundo dígito do CNPJ).",
     "nome": "cnpj_ordem",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Dígito verificador do CNPJ (dois últimos dígitos).",
     "nome": "cnpj_dv",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Código do identificador matriz/filial.",
     "dominio": "matriz_filial",
     "nome": "identificador_matriz_filial",
     "tipo": "texto"
    },
    {
     "descricao": "Nome fantasia.",
     "nome": "nome_fantasia",
     "tipo": "texto"
    },
    {
     "descricao": "Código da situação cadastral.",
     "dominio": "situacao_cadastral",
     "nome": "situacao_cadastral",
     "tipo": "texto",
     "transformacoes": [
      {
       "preencher_esquerda": {
        "caractere": "0",
        "tamanho": 2
       }
      }
     ]
    },
    {
     "descricao": "Data do evento da situação cadastral.",
     "formato": "AAAAMMDD",
     "nome": "data_situacao_cadastral",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Código do motivo da situação cadastral.",
     "dominio": "motivo_situacao",
     "nome": "motivo_situacao_cadastral",
     "tipo": "texto"
    },
    {
     "descricao": "Nome da cidade no exterior.",
     "nome": "nome_cidade_exterior",
     "tipo": "texto"
    },
    {
     "descricao": "Código do país.",
     "dominio": "pais",
     "nome": "pais",
     "tipo": "texto"
    },
    {
     "descricao": "Data de início da atividade.",
     "formato": "AAAAMMDD",
     "nome": "data_inicio_atividade",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Código da atividade econômica principal.",
     "dominio": "cnae",
     "nome": "cnae_fiscal_principal",
     "tipo": "texto"
    },
    {
     "descricao": "Códigos das atividades econômicas secundárias, separados por vírgula (mantidos como texto; tipos complexos não são publicados).",
     "nome": "cnae_fiscal_secundaria",
     "tipo": "texto"
    },
    {
     "descricao": "Descrição do tipo de logradouro.",
     "nome": "tipo_logradouro",
     "tipo": "texto"
    },
    {
     "descricao": "Nome do logradouro do estabelecimento.",
     "nome": "logradouro",
     "tipo": "texto"
    },
    {
     "descricao": "Número do endereço; 'S/N' quando não há número.",
     "nome": "numero",
     "tipo": "texto"
    },
    {
     "descricao": "Complemento do endereço.",
     "nome": "complemento",
     "tipo": "texto"
    },
    {
     "descricao": "Bairro do estabelecimento.",
     "nome": "bairro",
     "tipo": "texto"
    },
    {
     "descricao": "CEP do logradouro do estabelecimento.",
     "nome": "cep",
     "tipo": "texto"
    },
    {
     "descricao": "Sigla da unidade da federação.",
     "nome": "uf",
     "tipo": "texto"
    },
    {
     "descricao": "Código do município de jurisdição (tabela da Receita, não IBGE).",
     "dominio": "municipio",
     "nome": "municipio",
     "tipo": "texto"
    },
    {
     "descricao": "DDD do telefone 1.",
     "nome": "ddd_1",
     "tipo": "texto"
    },
    {
     "descricao": "Número do telefone 1.",
     "nome": "telefone_1",
     "tipo": "texto"
    },
    {
     "descricao": "DDD do telefone 2.",
     "nome": "ddd_2",
     "tipo": "texto"
    },
    {
     "descricao": "Número do telefone 2.",
     "nome": "telefone_2",
     "tipo": "texto"
    },
    {
     "descricao": "DDD do fax.",
     "nome": "ddd_fax",
     "tipo": "texto"
    },
    {
     "descricao": "Número do fax.",
     "nome": "fax",
     "tipo": "texto"
    },
    {
     "descricao": "E-mail do contribuinte.",
     "nome": "correio_eletronico",
     "tipo": "texto"
    },
    {
     "descricao": "Situação especial da empresa.",
     "nome": "situacao_especial",
     "tipo": "texto"
    },
    {
     "descricao": "Data em que a empresa entrou em situação especial.",
     "formato": "AAAAMMDD",
     "nome": "data_situacao_especial",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    }
   ],
   "descricao": "Estabelecimentos (matriz e filiais) com situação cadastral, atividade e endereço.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela ESTABELECIMENTOS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar ordem e quantidade de colunas no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar codificação ISO-8859-1, separador ';' e aspas duplas.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "O layout lista a situação cadastral como 01, 2, 3, 4 e 08; o contrato normaliza para dois dígitos. Confirmar no arquivo.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Datas nulas como '0' ou '00000000' não constam do layout; confirmar.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "dominios": {
    "cnae": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "cnaes",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "matriz_filial": {
     "coluna_descricao": "identificador_matriz_filial_descricao",
     "tipo": "literal",
     "valores": {
      "1": "Matriz",
      "2": "Filial"
     }
    },
    "motivo_situacao": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "motivos",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "municipio": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "municipios",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "pais": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "paises",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "situacao_cadastral": {
     "coluna_descricao": "situacao_cadastral_descricao",
     "tipo": "literal",
     "valores": {
      "01": "Nula",
      "02": "Ativa",
      "03": "Suspensa",
      "04": "Inapta",
      "08": "Baixada"
     }
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "estabelecimentos",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0001
   },
   "regras": [
    {
     "acao": "QUARENTENA_E_ALERTA",
     "colunas_referenciadas": [
      "cnpj_basico",
      "cnpj_ordem",
      "cnpj_dv"
     ],
     "descricao": "CNPJ básico, ordem e DV somam 14 dígitos (8 + 4 + 2).",
     "dimensao": "Conformity",
     "expressao": "regexp_like(cnpj_basico, '^[0-9]{8}$') AND regexp_like(cnpj_ordem, '^[0-9]{4}$') AND regexp_like(cnpj_dv, '^[0-9]{2}$')",
     "id": "cnpj_completo_com_14_digitos"
    }
   ],
   "selecao": {
    "arquivo": "*.ESTABELE",
    "quantidade_partes": 10
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_motivos": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código do motivo da situação cadastral.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Descrição do motivo da situação cadastral.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de motivos de situação cadastral da Receita Federal.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Publicação mensal do CNPJ (arquivo Motivos.zip); a tabela não consta do layout oficial.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.MOTICSV).",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Tabela ausente do layout oficial; a estrutura foi declarada a partir da publicação.",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "motivos",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.MOTICSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_municipios": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código do município na tabela da Receita.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome do município.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de municípios da Receita Federal (códigos TOM/SIAFI, não IBGE).",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela de domínio MUNICÍPIOS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.MUNICCSV).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "municipios",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.MUNICCSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_naturezas": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código da natureza jurídica.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome da natureza jurídica.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de naturezas jurídicas da Receita Federal.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela de domínio NATUREZAS JURÍDICAS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.NATJUCSV).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "naturezas",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.NATJUCSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_paises": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código do país.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome do país.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de países da Receita Federal.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela de domínio PAÍSES.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.PAISCSV).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "paises",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.PAISCSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_qualificacoes": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA_E_ALERTA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "BLOQUEIA_PUBLICACAO",
    "nulidade": "QUARENTENA_E_ALERTA"
   },
   "chave": [
    "codigo"
   ],
   "colunas": [
    {
     "descricao": "Código da qualificação.",
     "nome": "codigo",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Nome da qualificação.",
     "nome": "descricao",
     "nulavel": false,
     "tipo": "texto"
    }
   ],
   "descricao": "Tabela de qualificações de sócios e responsáveis da Receita Federal.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela de domínio QUALIFICAÇÕES DE SÓCIOS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar as duas colunas, a codificação e o dialeto no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *.QUALSCSV).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "qualificacoes",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0
   },
   "selecao": {
    "arquivo": "*.QUALSCSV",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_simples": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "chave": [
    "cnpj_basico"
   ],
   "colunas": [
    {
     "descricao": "Número base de inscrição no CNPJ.",
     "nome": "cnpj_basico",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Indicador da opção pelo Simples (S ou N; em branco, outros).",
     "dominio": "sim_nao",
     "nome": "opcao_simples",
     "tipo": "texto",
     "transformacoes": [
      "maiusculas"
     ]
    },
    {
     "descricao": "Data de opção pelo Simples.",
     "formato": "AAAAMMDD",
     "nome": "data_opcao_simples",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Data de exclusão do Simples.",
     "formato": "AAAAMMDD",
     "nome": "data_exclusao_simples",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Indicador da opção pelo MEI (S ou N; em branco, outros).",
     "dominio": "sim_nao",
     "nome": "opcao_mei",
     "tipo": "texto",
     "transformacoes": [
      "maiusculas"
     ]
    },
    {
     "descricao": "Data de opção pelo MEI.",
     "formato": "AAAAMMDD",
     "nome": "data_opcao_mei",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Data de exclusão do MEI.",
     "formato": "AAAAMMDD",
     "nome": "data_exclusao_mei",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    }
   ],
   "descricao": "Opção e exclusão do Simples Nacional e do MEI por CNPJ básico.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela DADOS DO SIMPLES.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar ordem e quantidade de colunas no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar o nome físico do arquivo (padrão *SIMPLES.CSV*).",
      "impede": "VALIDADO_FISICO"
     }
    ]
   },
   "dominios": {
    "sim_nao": {
     "tipo": "literal",
     "valores": {
      "N": "Não",
      "S": "Sim"
     }
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "simples",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0001
   },
   "selecao": {
    "arquivo": "*SIMPLES.CSV*",
    "quantidade_partes": 1
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  },
  "rfb_cnpj_socios": {
   "acoes_padrao": {
    "chave_duplicada": "QUARENTENA_GRUPO",
    "chave_nula": "QUARENTENA",
    "conversao": "QUARENTENA_E_ALERTA",
    "dominio": "QUARENTENA_E_ALERTA",
    "linha_malformada": "QUARENTENA_E_ALERTA",
    "nulidade": "QUARENTENA"
   },
   "colunas": [
    {
     "descricao": "Número base de inscrição no CNPJ.",
     "nome": "cnpj_basico",
     "nulavel": false,
     "tipo": "texto"
    },
    {
     "descricao": "Código do identificador de sócio.",
     "dominio": "identificador_socio",
     "nome": "identificador_socio",
     "tipo": "texto"
    },
    {
     "descricao": "Nome do sócio pessoa física ou razão social do sócio pessoa jurídica.",
     "nome": "nome_socio_razao_social",
     "tipo": "texto"
    },
    {
     "descricao": "CPF (descaracterizado pela fonte) ou CNPJ do sócio.",
     "nome": "cpf_cnpj_socio",
     "tipo": "texto"
    },
    {
     "descricao": "Código da qualificação do sócio.",
     "dominio": "qualificacao",
     "nome": "qualificacao_socio",
     "tipo": "texto"
    },
    {
     "descricao": "Data de entrada na sociedade.",
     "formato": "AAAAMMDD",
     "nome": "data_entrada_sociedade",
     "tipo": "data",
     "transformacoes": [
      {
       "nulo_se": {
        "valores": [
         "0",
         "00000000"
        ]
       }
      }
     ]
    },
    {
     "descricao": "Código do país do sócio estrangeiro.",
     "dominio": "pais",
     "nome": "pais",
     "tipo": "texto"
    },
    {
     "descricao": "CPF do representante legal (descaracterizado pela fonte).",
     "nome": "representante_legal",
     "tipo": "texto"
    },
    {
     "descricao": "Nome do representante legal.",
     "nome": "nome_representante",
     "tipo": "texto"
    },
    {
     "descricao": "Código da qualificação do representante legal.",
     "dominio": "qualificacao",
     "nome": "qualificacao_representante_legal",
     "tipo": "texto"
    },
    {
     "descricao": "Código da faixa etária do sócio.",
     "dominio": "faixa_etaria",
     "nome": "faixa_etaria",
     "tipo": "texto"
    }
   ],
   "descricao": "Quadro de sócios e administradores, com representante legal e faixa etária.",
   "documentacao": {
    "estado": "PROVISORIO_DOCUMENTAL",
    "fonte_documental": "Novo layout dos dados abertos do CNPJ (dez/2021), tabela SÓCIOS.",
    "pendencias": [
     {
      "descricao": "Arquivo sem cabeçalho; confirmar ordem e quantidade de colunas no arquivo real.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Confirmar codificação ISO-8859-1, separador ';' e aspas duplas.",
      "impede": "VALIDADO_FISICO"
     },
     {
      "descricao": "Não há chave declarada; avaliar unicidade com evidência física.",
      "impede": "APROVADO"
     }
    ]
   },
   "dominios": {
    "faixa_etaria": {
     "coluna_descricao": "faixa_etaria_descricao",
     "tipo": "literal",
     "valores": {
      "0": "Não se aplica",
      "1": "0 a 12 anos",
      "2": "13 a 20 anos",
      "3": "21 a 30 anos",
      "4": "31 a 40 anos",
      "5": "41 a 50 anos",
      "6": "51 a 60 anos",
      "7": "61 a 70 anos",
      "8": "71 a 80 anos",
      "9": "Maiores de 80 anos"
     }
    },
    "identificador_socio": {
     "coluna_descricao": "identificador_socio_descricao",
     "tipo": "literal",
     "valores": {
      "1": "Pessoa jurídica",
      "2": "Pessoa física",
      "3": "Estrangeiro"
     }
    },
    "pais": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "paises",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    },
    "qualificacao": {
     "coluna": "codigo",
     "tabela": {
      "base": "cnpj",
      "entidade": "qualificacoes",
      "fonte": "rfb"
     },
     "tipo": "referencia"
    }
   },
   "formato_contrato": 1,
   "identidade": {
    "base": "cnpj",
    "entidade": "socios",
    "fonte": "rfb"
   },
   "leitura": {
    "aspas": "\"",
    "cabecalho": false,
    "codificacao": "iso-8859-1",
    "escape": "\"",
    "separador": ";",
    "tolerancia_malformadas": 0.0001
   },
   "selecao": {
    "arquivo": "*.SOCIOCSV",
    "quantidade_partes": 10
   },
   "transformacoes_padrao": [
    "aparar",
    "vazio_como_nulo"
   ],
   "versao": 1
  }
 }
}'''
)

In [ ]:
def _hub_conteudo(modulos):
    """Fontes dos módulos sem a quebra de linha inicial da célula."""
    return {
        caminho: fonte.removeprefix("\n")
        for caminho, fonte in modulos.items()
    }


def _hub_conferir(modulos, dados, esperada):
    """Recusa biblioteca alterada fora do gerador."""
    texto = json.dumps(
        {"dados": dados, "modulos": modulos},
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    calculada = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    if calculada != esperada:
        raise RuntimeError(
            "nb_hub_biblioteca foi alterado fora do gerador "
            f"(assinatura {calculada[:12]}, esperada {esperada[:12]})"
        )


def _hub_instalar(modulos, assinatura):
    """Materializa o pacote hub em diretório temporário e o importa."""
    destino = os.path.join(tempfile.gettempdir(), "hub_" + assinatura[:16])
    for caminho, fonte in modulos.items():
        arquivo = os.path.join(destino, caminho)
        os.makedirs(os.path.dirname(arquivo), exist_ok=True)
        with open(arquivo, "w", encoding="utf-8") as saida:
            saida.write(fonte)
    carregados = [m for m in sys.modules if m.split(".")[0] == "hub"]
    for nome in carregados:
        del sys.modules[nome]
    if destino not in sys.path:
        sys.path.insert(0, destino)


_hub_modulos = _hub_conteudo(_HUB_MODULOS)
_hub_conferir(_hub_modulos, _HUB_DADOS, HUB_ASSINATURA)
_hub_instalar(_hub_modulos, HUB_ASSINATURA)

import hub  # noqa: E402
import hub.orquestracao  # noqa: E402
from hub.registro import Registro  # noqa: E402

REGISTRO = Registro.de_dicionario(
    _HUB_DADOS, versao_motor=hub.__version__ + "+" + HUB_ASSINATURA[:12]
)
print(
    f"hub {REGISTRO.versao_motor}: {len(REGISTRO.contratos)} contratos, "
    f"{len(REGISTRO.bases)} bases"
)